# Kaggriculture Farming Score V46




## 1. What the defeat logs isolate

The replays separate three different regimes.

1. In the clean mirror game, both sides followed the same farm-production actions and ended with the same farm structure. Planned total quantities were almost identical, but the opponent sold repeatedly about five turns before the base schedule. The final score gap was therefore mostly a market-timing gap.
2. In one third-position Yarn game, the opponent already had a sheep-oriented continuation when the shop unlocked. Switching merely because Yarn appeared would add supply to an increasingly crowded wool market.
3. In the other games, the original backbone remained close or competitive. Replacing it wholesale would discard more evidence than the logs justify.

Accordingly, V46 uses conditional overlays rather than a new universal route.


## 2. State-consistent route adaptation

Let $R_j(t)$ be the complete action prescribed by route $j$ at turn $t$. A route change at turn $\tau$ is safe when the two schedules share the entire preceding action prefix:

$$
R_a(t) = R_b(t) \qquad \text{for every } 0 \leq t < \tau.
$$

The third-Yarn route shares the default prefix through turn $215$, so the decision is made only after the third shop is visible. Define the opponent's public animal counts at that gate as $C_{\mathrm{opp}}$ cows and $S_{\mathrm{opp}}$ sheep. V46 supplements the original validated prefixes with

$$
S_{\mathrm{opp}} \leq 4
\quad\text{and}\quad
C_{\mathrm{opp}} \geq 2\max(1,S_{\mathrm{opp}}).
$$

This is a supply-side test: enter the wool route only when the opponent is still clearly cow-heavy. The decision is evaluated exactly once at turn $216$ and then latched for the rest of the episode, so later opponent purchases cannot switch our production suffix back and forth. The existing `PET_CAFE, BRUNCH_SPOT` veto remains in force, and all previously validated Yarn prefixes retain priority.


## 3. Inferring a market phase from public flow

For product $i$, let $I_i(t)$ be public market inventory, $u_i(t)$ our emitted net supply, $v_i(t)$ the opponent's net supply, and $d_i(t) \geq 0$ external demand. Between observations,

$$
I_i(t+1)-I_i(t) = u_i(t) + v_i(t) - d_i(t).
$$

Hence a positive residual after subtracting our known contribution is conservative evidence of opponent supply:

$$
e_i(t) = I_i(t+1)-I_i(t)-u_i(t).
$$

Demand can only reduce this residual. During a long near-clone streak, V46 activates the longer response only when such a residual also aligns with a sale scheduled in the next six turns. The replayed opponent used an effective horizon near five, so V46 responds one step earlier with

$$
h_{\mathrm{phase}} = 6.
$$

Detection is limited to turns $160$ through $260$ and requires visible farm similarity. It is therefore a learned response to a particular public pattern, not a permanent aggressive mode.


## 4. Conservation and production-stock protection

If $q$ units are moved from turn $t+h$ to turn $t$, the overlay records exactly the same future debt:

$$
q'_t = q_t + q,
\qquad
q'_{t+h} = q_{t+h} - q.
$$

Therefore total planned volume is unchanged:

$$
\sum_t q'_t = \sum_t q_t.
$$

The longer phase response is restricted to strawberry, melon, milk, wool, and fertilizer. Wheat and carrot are excluded because counterfactual replay showed that moving their scheduled sales could create a large temporary inventory deficit and interfere with feeding or production. The original short, conservative horizon remains unchanged before phase evidence appears.

The engine logs also establish the relevant event order: a worker's same-turn `PLACE` or `DROP` reaches the shed before market execution. Thus sale feasibility correctly uses projected shed inventory including a valid same-turn deposit. V46 deliberately does not carry forward the previous revision's contrary assumption.


In [ ]:
# This cell contains the complete submission source and writes it verbatim.
import base64
import hashlib
from pathlib import Path
import zlib

MAIN_BLOB = (
    "c-ox1YrC>sk|y{)f5l5GG)nEfJRo2z;t4@fK?OnQY5|dhf}kA5({CTM@^M%7OwCO9>@Qnnt+<0=t%y5fg@6D1-+KrB5+76Zu"
    "ay&A{$dh6Jp8prvH$*8XkI)$|1os$xDWN%Jp6UmL)VP|I+&K}N9I5N{`bHC?Qh5PB?#lc_IhMu==bUr1^)MQlt<r>aegs>FY"
    "X_)fBW0tKEnP|QxsJ%y5Z`!3H$olEdBSt{q^}f2ZzKn|04e>;~)RJi2h-l@xR9W`;epc9d(TUng_o5ubKbT?0<*-$L!xP#n0"
    "#eH2puu{X<o)qi3q>XTsmx`@}zZ=Hcg$Qvdz$e<6SUbJpkI-#dT*zfb)$?%!wsbKHM5B6j}$_W$ueesBJ#$NE41IqQFXZvX5"
    "0--m=|oP_@OoBw1gFVGMF{t*B6R}>_n@wL}Q4^2P*x4+bX`^#{S-r<ij{|G-f;$wUaeE9GG`1`Nn{x$qR4E8_0hJW2G{q29%"
    "lK(GO{I9I{Us~>8n1B0UFpYUL4cG|Imk${HFB|;Z|5b&5`|I!j9QOCpf4Xxx8nLSWKUyF%V?B;T_&=lnYs~-nW74mz(tr7e)"
    "b+=LGkz#d>D5Q!e{O?+{PSf+@IP1kKMv+URSwN4g8#K&{;?+-|NQb`!heJRUE+T;enO9{*Zo!EjR(CNXaXr!M?Imi8ny84g="
    "CP0BFRa8haVx|$S2c{EBd&^+;b5qV?q@vl+(2C^L)NdB^B(GMY+RAB#$UX5qXXb=u!XqZ~uY<S(U5i<zUr~Lw{qBUUo&oFLA"
    "xGyf~9n={777CA?{@h}In_?nrbD%r0c@C_<2va=oIHYBT=l64#8T&6+dq{FQsTtgQR#Jq%K%p>+iG+>O(^V`l@EP^)OWVL#X"
    ">h;^w;_hHP#6Q_~N!~w;FNfg4uWV~W3Cvd4F##KJcH>!7MI=2<o=nJ^~GIZ&p6NQoZ2dY%R*M;j5S%-1cLw@}G`<c-LQ4J(3"
    "1iJMpM*M_JFy20YpV>Fh4m1Z##~rpQ-#|!%ytb$m>g(KTU~tR42#X?iw7Auo-_&Z(QspnWonJ6UtF31Tpmy@mav&i1UR%r#u"
    "cJFgZE^0*_<>=xg1Oh(t<yhHts_5)o<!lS``)=Wb{Vhk8q*azC|rtuGVPxaPtH7uU(K{4IcU~M=7u8d?RkBy<XN(VH*Kw~YW"
    "#G3@<Bn($+{0TvTW&ixttfLrMJsqyNLSAa90S`qtx+Zp7AYpqNe-^*Oxc6w>`w;f==P|c@9?D?{JhQN~36bIyq&MCXS-NZk8"
    "UI^m2Egs}22ex4;)#u--y#rzx*T0#>dyCorJxxO|$+HQs~o^{7u+JK7|Z$#GfFeZna2yPVbQg2K890+r<-sKuk(abgwiqQ~T"
    "zyp#aKss)Xgg0C(E*b9~|<)poygF*-Bl&R#RMHH)<By?zkgWknza;w{vg=)?XTA)B#n{0GN<DEeQEepW?q0UbWNIy{<^}IW^"
    "xihXD)!JJqpKRKhs$$X}{DInvh{Bk!+p3VyFu7iA`7BDt$$&0lUu|S#w?EDns(iD_=N!eRWu9+6wIvVj&nyb$6xJu8Huv6u#"
    "mAj%b<HC+mM|iA^->$FspI2gyGsUZt8!#5O~~9{xKN?a{AB7pQv8A_mitkwW1a{!x5f>vIo7A@X{SE3m39p}1IQf)SKuLE`j"
    "?R0;Ww@~-BP1@0n)_hx)L>&WnNM$59oZzuJc-XLCVmf!WLRz`F^cI(;x#?+Qe;r!V8SNEU63dM03jYP&n0wwJkE6k=Z+&KA~"
    "(2Wzb*|_y}2{bw3+>kkou2>q&LL+qBkOrxuObdUZhcj>8D_%I#fUn;*}ApxDC9zKrn}u^Fztr#!U5opa%*lo|O$YHP9YXPV`"
    "*4lL83=*~DeslCq9)1>+`Px6OSzyoDM>aqh<-UT-zyAP84<+CEO)Qc+gOUA-7i2i}XXxWy6S&@)uW$!(|`S^3|e;;|eYlF6d"
    "*J1EBqR{TEFkkw!`gAHA_bA$<y8_oKPFIW7dDp?QCRHx5%4%4;Km<~92U9BCfNIo#ak4m;XIZ1=oR(Za;W>zMyOU8n#VhCI{"
    "0|hz3nvmh9$#pFtZm=j{v!fPezH^}3px5$r=q4<{niGMD4xpRg?U2KYU3yBua~UZ?Xh8vE6B}Wj%xcJr<)6{j<~?k0?C~;?z"
    "C#GQ$@1yb|k(0D1wd3F+Oig$zhcof@q1eEMO7vX!;|Q^=_eA->SAP=JUqvDGX-oJ@YgE?k%}G94<8nQ<}<^hWD4!P2uK?1=;"
    "Fq)pX*)vjGU+N8JT!y=Tj53oyf(Au(f&Jr}z=#-rig98)xXTt3;bo{;C!kk@<tY&FiGOP@Q9uXONz0n@U4ZPyu@mI_pDRqlb"
    "4w`q4nZQGxf>StTGz;ZtZvh047r0S)qEdxWWJN2azNu5^>RA~M2I407i<hHwJMT_Y2Rht%1ujjIhSd)nqO}_rs8HyVVq(?dZ"
    "!$#fxxK)PIyXtf|_A~=ID;?vVuqh!J$!#v}3k`vJg)%0};YWD&t^=ibh_YKDtL3Zw`$`~Z12`2SYN^$}DawP+I9>GA=oC-)p"
    "o4UZr63fz!loywFkMUK&c>N_e^AFN!TZK&%-&8@7aM+s5#bnXK<)YV(5@7=gXEFZyDJ7IPB0IkS!vy9hg2rb6B5s<2>8h-!R"
    "`5Omk;WQE~|bZ|7fJWj@TbnmD||;xv&fzMVpq=S`h`*KK!ou_Y&K#p?!VbyE*1&P@Y03<7C-Dbw02}4o2cTNK!4jJcL=TRs%"
    "a`9OFM*Qde(jw69k5ajS{8ztV6Y=}OxKAKdH>etixagQMm!?_|1~uFTToUiJ8y0{DxqIFWl~B95E8=kBP)ReDf4&ZwIIdYwg"
    "?hTm93?@HM*Edm5DfrDExKMg63ZUg<V7e>h;Ccn0;MtE(ivuk0tT|0anG_?+jEY<9}YCa0r)lzT+*z5t^>gr`#r)tAf|A-EY"
    "GqwlSwb@}>=X>}BPDEdn1YnU$Yw?a(8~s)_zV`}$pi(c7tpG6VsbO}UNwHf82lsx*X;Ns9H75p&CSL=1wN48WdlZQFWI)s_T"
    "^q>vg*bQKhguQ4wdG*bZ%OdhB#XBm!~s`oHuvkB!xDFZJqiSd4hrY;Snjs^Dk0@7Y(6|cRQnUIYR^tdWa@Sy{sZ+ee%ii!Jc"
    "vny>~ueV9(+*lSC5jmr7Ik<YYgUt^>Q&zcFoUHnWe>rz@Y|WJCJgDHJ@<5%U_}XIy}|(`*+GI843y-1+0zoWxuSjrAE*)dNX"
    "cPJH8J*byM*+*50y{FN~F*)NTb7f~HBBW^c{iPHTI;mx@CT^tT5gj6|KGKl?=;Afv$#12NC*Vcn@&wKLM&9_3$WrdKY7Np!n"
    "GWjuC{tH&r}e5Ox9)w9FW#iETw&m1X)&SEtZsxOZ(3dp09Y_30Bg~uDEaoDlmOQ=U1<=r$U<5U7YSjp%3w4<ODx7DY>;Sbb|"
    "l6Sq~u<)2T^Zm^-$)((DJ4<{d2%A#u6{Kr(*%9+0Ff1T)w+p;aFJE7*uA8OFPqT4v`0=D&b>{z^uO%!d^1`wqkS9Jo4*T_1Z"
    "3ka^5%dSjc%h<*>tn64M%|sR)fSZ$Sh#f1d6k}}@OH+RF+I-)!Kq`JOR3H}2@AIMBi$cvnm_!5Z}+oO-<Um*l(^ZtjpO`fZo"
    "VeVfXF?yq%DpJa5zVgb<DAI&+$$#B50;WY%oZ%yY2@@v1sPReuRb6;1d=m3LQ7*ZF|I_$o0H1(KhmxSg7nu^#N0WYfk%JuUr"
    "o_Sfzc(4*bFcJ&P-g&|Y1{J75dCe4+KyWJ#yrx${SL_sm(dpoRrciKo-K^TxVi!r2FWT*n^qu{aLF$2;7wR{bQ)`HA_SUK;H"
    "|cB@>K<HT8&rou|L*k%lQR4d)6O91FvtI^5daKOFFDm&#5RH9wc)!u=ZBljn!;w4lC$$gP`Gw-p?5Bs&J--0(Xq9xk6uLCB&"
    "%PG9~>|&$D6mOTcUqI|eue77b{Q=@E-pB!Mj}JKj@k@$K&sq0Bym_OkPu9gWZsoxe2nZ$Y6D!euB}42`_cak}uald{QhHhkC"
    "AoerEk=~V=IGm5AGQ54KL%*c!i&B2+;XPX<Rs>Ph5Jh1`u#$7<Co2Cwez03J_(yMC<8_%F{^MEuiQt*q2S3?yl{?N`i^)#U^"
    "7Vv<u;}r@odu6tX))HV5zkmzNcF8zLc>^Yqn_@BFd6R1D*$$l|N9;bkC55zFYFZ?QU5)PMaK4J$ln0MmavKvj9-Fsv{1<uWE"
    "6FC_mWFdep;crFri+0>o#kDq%OOk{p(wM<5Gp_s8ni<{M+kGF$JeUu~K;JhIFxRob}8-eY*WzOre>_c~k%0rqP>IVFt5!Aix"
    "5A=dl@#q{HP)ff@#Nsi-EC92-X$<JVT*&LN01-Gs$1m2;;gym|**D~#ywP#I&JY=yB*(uL;(RDfGexm-qdLB+|*ZhZs`0rCx"
    "=gmYj^2CylkAS^)>XBLUiJ5ky>C5hNYz)kcn7oCDc6vy`-hcH208|@Yr{AyVcq1N^p^Oi<o$Ebp6V&yoTxe14j0B%ndwdnP!"
    "8Q=6f1vJ3ej2agYJCEYl9hfQiD6PN-SXrqZ@nV6=lOs)?d^8JLaPtSie$fBC@oeola#fw(OlnqN&Nci{d~N$(GkY$E%*GM!7"
    "GXEuCeOoHX-N4N*<s3o4KX1i<`>mt1&rPi16mXa65VNV=h!2R$Y@^KbKpkusspVt#djhw~y-tK&v#`$6t^-<r%``<BPeZ;vB"
    "N(>$;n_P8{2r&%5Md-iF?VqIMOkw%CH1!3$S=;bym<&M2y=@lh~ZJbNxUBFc8`NQvJgk!k5%l=2T}`~zS%;}|~9m$GxO0KfE"
    "LWNE7~_gGb5)(`hu*0HWPY1)NW=cf5;Nz_Tf`?K)9%s{*Ys_W5u{~9>8_PJ-z;Cf%01Poyn$MZ6-X=zF5w~qZpl;kBB*XD~R"
    "0e#X|@9_tUHa3Ozm0@dYui-x$(L{&M!}UX>%9Sd&zkQZNbs}~3OKiehsk)i2O1Mp0YH|F@T7|OT9<SY)n&O+Y5^w!M@xi_5U"
    "JP8FPlz4|gv+i0VDk;cKd5(h8rS)IeeOq6RSB2qu8iC+VLUTO4t#$JX~^J@((v?*n}(kh2Su3&nhx5S;7f%)1U3BiyZao?+2"
    "`fLVOB`kY$f~nLD*i@tY2E+2-&^nuK37$A5Tx+rS7OzZ+e32$WOJF<5`yyN7Kj2*Y=~5y;0da+L=|^gzm}L%yA1tZoBekAdD"
    "J+ps1fx=dJC5$mBN-G>S=mutqX{t&|@lNf_30cF0=N?D_N6jjz&Aci0myIsO3FE7&YTt*o_T=#zA0s|71B3!4_Z=`UJp*CC~"
    "HyUdqXoYxyK!1mSD=8wF;ROP|_zTavfb{Y<iqqNmmj{s*0T0A_^q+9=nH%?n^P3NOl%9LwG6|}%^wu~CYX{HrfTt1mSR-cOU"
    "c$*FE-;&f|Xw0NnWCg{1=!Y|FuwPH4cNOh(@SwSABCVWEY~Vr&;A&Yacx|X!+Ep8&d+4W5y_%F?g7~O+1@)#J&g;ddW8%`iC"
    "O@}d`FWHzg;@|j$Lf)M){%OqA@A-82;gmPa0r(@7}>qlL5zjx6_bK7B6~)-NNzG`!IN_P@U{ayb)U!#<JV1f)qCB=+Z1RqTS"
    "1s?kmxqQ?Y7??w0b?q8;&HRd1%tBiFU?BR&?)0ZoNS2g)v+o-sjHXpJAnNXZlNB44PH~_Ig&m#Lpq=81LM){$a7zE_g1t6eW"
    "ZD1+z(=04ZM^@u<>CpI;W2&&=NCcviQrd3Wap{0?XULu0$NQEpuy_$nKLeI9kjqg)?}y-s>h_BF_w@zKiIH2m0qSdT-YL2oA"
    "vB+<uYrkz1ly<N0M*;~s^o7N7+KTx6tT>IFXYKR@jmZ9Y(9<+y>eSFGyAh}*0l@lMGUGCyHjB?$&VTn^f&7*4;kK0}U*Lz8)"
    "yT^B(k=Gjc4<Yp70&^bfQQH1$bfFx2Odr+a;8m?6YFmlc<?)UFcnxh%vU>k<eM_Aqvs{mxNzVl7tc8JISAMtp8@*XBnQdUH-"
    "+!xp)m9_v81<HNrCR;q&YH0HSk~u7sn5&w_>J{(aqI7^ObG>m+G_008nyYc#CwTS)w`41gJH)1Uzgvl^}jRc|L5g5=>G4Cc#"
    "=!4d1z{fQWY$-emy_mEEp7uyCCY;s(bx$S}^x0R8g|IRVt~TUdT)SWx&!>6R_o;?v(k$6qcWD=BR={r2boR<aN?Sri{I5i~P"
    "CTV=-yb>`m&6yD1yyDJ(cou%h%fv20Dk;kdIFOVtxlW3R1laXr$Znq;u$Rw(us2Mg*%3$$DvG*e6JmNxzH_~xG^4SzjK#+pM"
    "4IJA>hj&JenTV6la55d@Pu6%dR%1(citZPrAknWtd&4XVRH$tKAMV)pyn2pNm{h$%>Y&c$pCej&<Pg@%U+ZQ%{Y({rE0*9lq"
    "T6MR5cuhRzDOu0Wa%A}YI%wQv{F0eMwu)=NL<`7mFm2t=g4waRW!R&l$C_beX3tjZLg;=75dI+#T+45`hx!5vweQS;3WG*BT"
    "wf+dNtYb(MZWj`;JqF_8{BjYU3$x7j*l!WP`f2&t^}w3#%ixWIbugH;q1QPR$i+r|1g^Xc<X_q-gQxx7^+minHWvI!TxUzke"
    "jW3a<UzgFwXzL#;_h8!gs^_4OW|Ozu2jv7V@^m;(^&*{ME3!7TNPCZw#M>&8@6%tu3~KeL;yy@MiXOcp-Q|(ArIcM>t&qJM$"
    "!EyB$gj7nX(tY0BNx`w2k%09-Mx4)RTR$HxLO=<)F^<C>r=hXo0+CX1gJMG3ctl6J;WoyBgQ*ARG|bf<a7+QAlr<>CUl-S$T"
    "ZXtyr#vmzH%R)v1^&cjepwn(V@S?*>eS#<p+cRDl<uHMU#VatV)Vx1zn=z0qEYqMNV7C-K#>XuWauA?3Y!2+=W7fJ7oQy05U"
    "G)kOuw|R^z<;B7RYlgw~=Sy`}gD=1v(m!qVWkWeLti4Y3kzixtFFk6KrPFgpUV2Xzxh-3=IRGvja7Ww1h|^k#mU^WcDHK*dO"
    "L{VxIAtWbAKkoo4kojSQ}pg9T1Mubd9(V)=L=haR2q3M7X2$F=Nh(HLhtLh`dHhid&61)EqC*q!x5)Me=*A~yC{N{%2v|JG3"
    "oL~{UT(A29~dp#f7<Ky~I#}&G`m2qDu!ZrnU5d_*glO=l83Ah*f=475!kBwT6e=b9`)Qq9P7$J{|#gdYt`%(g}1@dQ}(~s*&"
    "5&k*Z?1k7KJY+8n*1QQP8RDQ|RWZ7DJc75y-%U!bc^>5bVV!qCRrZ`mK<#w1GIC!)eTw|a3QV)VPbu&x>^6FyS<M8y2jpFtO"
    "HHLL(9@$fUvwSt>5rGndmM%DLbWkA?e&r?xbZXP7x>hZ5_nJZ6AEn@Gr+=hbCf1q9$Z@QC?k+>bp*M122X@XsAE@Nv&?#Pvl"
    "+p^2Qlkx^^d3C3<U6gWB8xJl@AiwF=y{i-^+F0*w?hm}ITN75g7iT?zGW(m??Z^~g-F}wT{y?2JgOY|+>|?g|G&D<c;Cw-&T"
    "w2Xv+4I@7;9|4)Na^-sdfVqwr@vdH<RfWKO4I3#&=-DTIEdG)Rd1zgMH0B(rbntVXE(#n(FU6?Af3zi_a7+XTt>4uaAeX)wD"
    "agd<mxN-URAFK(7Wuf+I)Alx;-MhfYS*Vb&{>NGWz$6s&lYz9Dy~XyMicf#o#G5tv-XFH;eo10oCEmtuhe%>*#9{hRB6+rzo"
    "}R&+6jTx=?V&iv0c9%$%X$ZQBSxaU-r@=R;2#MbB!`H$m&Rf%3Jsiz)~BQi*~|Ja%l_Y`id_*F9d~YfHI5z7V$tFj{0q4n*7"
    "XwyO>uYKfj}_qH|RpJ&Vaknv<uCFhC=FI%5P(rz?ezm@evq}sKcjR25F^0-k~SK>67XZS*WLMYL_R`6=_d#+PQcAu1lHmGNP"
    "AFeXB)3ObD-F6;01aSy==gsb=D|BnE64;C`t7v^0)l1&<zE!G++Halsnm*IzL33xLW)WEWO5fNv_S5OjYF(mMaRhCotTRZA!"
    "CZPBUY)E-+ovApE~ye+^{;m`D*@=$UeOtZNZT;<y~#>V<C2XlYU5GqJp{Pi!@o9BQ2o{OFYz5VJGD}PP@VHlchRpZ9iwHQw%"
    ")GU24WBG89q)7Av_Id?wPbt<aI6<)z{(bQU|6ajqH=J4BE~d>Q2Y>_M8oC4{3ka(e)y2-}!+fequ2SiM_OK!{y)FUPqsrcBx"
    "C4@9$T-V65_o4~d9yKNN&^@D{Rut=}}U$6AY4XRLv^$le<t`cre{Rp|!3E^h1F@q3)sgH^f8GLvzh*@wgJN<3=gaIcNCktK#"
    "euTs{7VnbZiy{`e9@UnO-2EA)3sdu1ZrEPU#{NtYrLu$l^aC<wkn!}x=KV+p}^i}eD-0isM2vUZ?QyuhDbC#>y-rF&<)`?tI"
    "oO8L3#x-Z86n5dgJa@MB<QtU7+5Xu>x&=OEYjaqPf}O-A5<NjWir*>j%PC%1j9waFE}W3gWD``1Y-sYm@s5m!$OUTl$8bUp*"
    "~M<USF(QGU*7HM_7qv|obuxRvSsx3KTv(;h-7Pkd9EIVgPpts_u2<`g{r_W(Nx4i)z-~yK`ii-hZ5nNS7d3#49-5>%2jS3B0"
    "1|$NNZS1FuCMimSlyyp9{w|zaZ?S7Vj_C{_-WJL*Vi>CAC`_cHJ$v8fP3|1^l*Ow_m=jOI+5hLErbv?59h&QG^vG!m3k1M6^"
    "uAAF_HY7A(0EvX2Hl9V|NH;*O2@r_8|87GIgJ=8zV$Gtb?7j?=o<4~yOTg4zV$785|kuWe@cDcLUl9y?b%r8Oi|tqN1{`c4V"
    "-s(TJJ59318bB|CTPnGM=gw7tXCq|y7D<~8SM|75QadlksTJQXhjXgNA*-nj^y<OF9?B^YU1L7eutp{=fSl<jeV}ir}L47vI"
    "amIixDaUX8{KN>2L|Huic3k~!S|A$32RU}Z`}Dn>ofnbfap<|F1(zhS*f$ua&vi^IbjzeG^w36q=5%`%8m|S~hVf=HJv%;?F"
    "Kd?YY&6JJ-NnWGf|P|?jW}|O!7@H+KbSvI$6kGuJ>Kw$@`!DtW8i|8tl9zDnW@61{z8Uz#gC8bm}o0*uGNJ_Px0G}j6ddX=K"
    ";I(>vD8GH}K#&)4FZB(3ywJ`Re7TIBqs`#^mqD3ityR=_GfYDJ*IV)Fi%lGv$mG(51QF&5ZKpin~wpKrts%$-0{j-)I;`r0`"
    "2;BxBOJcvh;#ulKR5MB7_VF^+tBxfE8vx<lwOn2a_z`m(*_uad^0)DDoh>0p%^W%e)zAKGDIOr3itRLwMKU~Dt~<boR=Sd_0"
    "NmgSE->9_p+6zwT@qoxn0e(SWkA#oF@#*dYm>uhu9nrFm_%^`13g2la3@1fA)IRdBSYT_1aJ*qe?t=d-&>J@HFMJsT-u=`2n"
    "GkN#nCuKscBQ&#Sz7A#5=<`|w{!xk%;y}Bf&ID`iK^}+(<&CS}+uN?=IaPeh%iTRubV8xp8)vizE&7+DDs_;BoDLd8D_!3!d"
    "~QXq;MU}OX8)!eA;KRSxLpM|0<_O|l`i!DiI?6f@k@SQLr;CuzE8$Azuw!_&FW(;b$5pSojj``nChr~L&}f6V}qr4{e^v}>n"
    "mRCg2gTQoED$6!&DqC&mx=^2Kkfh@ofK$6v`vTtOqV_6TclLha#8Urno;VP$<xx+Fj;QI%Rqu6)QqXFy;*_=_t?{q*z8c45%"
    "B^sP>{|-?%NFN)cy-iml3%WhUnKr42+K%zwrWN{c^fd4CF@dprCKRHhSWP1o4ol>*CiH9M=JNU}uL+MzWYpRkp5Pn@^ZqqFI"
    "kr#Y>1##3YTD4;JDWi4S@Iz3%(P~heHb$@AfXDM*|if&+1TW^;cUJXipomB^_FRiHQzD8B|jR}53sx3mTG^<cFCJj|NMS)yz"
    "H{=_(zI3h9VM|Zt-d-@);~#JOdr+e1{A;o4XqC{-DzCttmINf&&MIf*I$;3{xgRF(kw~1~X3CdiIjwfry*|I?JC{nk)E*u=a"
    "8a#Zv6~I@D?B2g^lI&gh3@JrzsI~GTV}6Uh3X~a5}dSZxxNm--tQ^n4973~R09S~TR)yEA+R0BTWEB4NsVr9JD-fsA}fY40@"
    "C$hIP6&AweG(YwXHTx1Lq$S(m)EwtCm6SmMN_-ot1bFsyB<N1D!U|bIzjlJ^SCu3Uh5%RyAgHJ|l<_#<!PnNGQah^yz*Cw9m"
    "@&Nn8O%;?)zk-yMfX^i@IC#$?|gAlcI<?ZAoEXx&bz{ty`%XK+wnOiHW5bm25M_rdm9{sVO`RIq_$j3^C+3;8}B;I}7|MUUf"
    "$>_jRQZr;^oZ%z-VG{iQ-w2?R5>;|s6x|-O__39-FXQw)$HX(yHv7zz2RKYP)eizi2t*hrjc}HMhg1xHv!XwF`b3J>1yk@;Z"
    "mWId=GiQ=(7-`t_S*WMi34oEmiSS)?n@To-@%c&bJb-p_&#9OQ&V~hr25kqfFtEMWD*6SG3>t81Tljt>4(s~D3!=hF*vRKFi"
    "4TZb`;P<uywNruc{88w0{qr>(7QU%ocfvQDOc%d)SiIjzW#GR(A|d8E@Z1`vtu~b7lEXg78Z16_&l4yZ@JmKwQK9&Ln|>WV~"
    "J5q!1&Z9CY@xou-<fG*4$9tjQ*`_tu+45PN)6zaKAv8)v$lOO*&ySSivY6HciN<WuBs$<;1Ra+7vcwsy&z;xis2+&>~eOa$K"
    "uUNuGJu@CDnh2|%*=RUz78?-|xA2wY?x{Pw=_a}MA4n+ws#6RpG^e07;#t+HM96<;UAkF&i7KH4S*`R=l11rr-h*<~9A8jsR"
    "bD%0zJZh`A!Gs~Gr`M3M!Fh4r0mY&!|d$sy$4<ElxX|J?{m1=LmBgbzH-5ALyT04Ae^k91p294)#eubQti9hl5MRfNIl_etZ"
    "a+^(;<*h`4*(Z}-`Hijpq1Py?x9NfQAupn{>%_&=#5tTZf8PA&PO?1N6ttze8JvZ|%qy2rqBo#UNPyE)Q^<THSZ01n^LJdjs"
    "_b*-!);o9eRj&v<iW<Yiqac3Qc*P|xo`GXll0U=r<pzNl;_s-;A}d>_U>DFj%V`=8Cv63Cyq^F9=>F_!i)F&w5<bR+w|)g`M"
    "|F*w(E}>WLX^;1TIOlTEp<}(!Pn(a%P_|Gr}KXyec&Fc#M*fJs_$=aBZF^IFxO^dt*Jx1@R$XOX7x^A6ksJ##l%#@Iy2-?et"
    "_?ha)-|-wCU{_is$C$aTshq1{IHG1MPASDh>+RrSzxG%KGmC%$t(U4wLq6P`KRH!b(No0#%9DNL3;Wgy&*+$i^-&e>=fk4@I"
    "GWEmo>i?mc%I+b`f&b1>UpoDAv+?k)v*rm(OWv|#ELqlD5%P>!D?ZwlDhRJ|>E_2%AgPN>l9~!gb;_n%qB<(YJS6Z~o5x(U7"
    "c7A9p@v_j-XIBRPZGgn4<9U5-aomH`iE-F-Y`j-|j0g3r%#4=8Slz&f?$Yu;`9(t&&5rh=_@JE^2b=JY@JQrB;6Qn;U;TgfQ"
    "Jh-xW-vQYl~v--7T4(&EU^}a{ef!FH$VM8SC&;TdbSX<N=6fIykXimu5sFfOb^(7@?UK2P4`C8(+-;=+g{+9vlzxxH7}hg$b"
    "hj9eNlMV+0|Q_!qn+1+MU2l{RhOY`1pJ1cB-w2>^6CjP?2SBN<eK+*?|x)9rV_HgL{8njoO~5r;qut-_-n)O6WMe`)nE4&u^"
    "!YJ3i+h59Do|xI67-(yafp^nhU^fk>0ZhQ&4De|vIf&jY@KgVPYX`HS^zGT~^gn7-z93FtM(&^2aJ1CZXEE_0jP{G<C@&$JR"
    "})+`-*0~BROiCnlftiZXqsp56~#EU@rj*z=;^bS$4lV5lKP+eCh{)+38zezy=zjoov9%)-IQZ9klX56S&$2cZ&U#>K4iqH5o"
    "%5R9K4Twrp#lp#OSlU-gv3!~cTWC^iP2;YlFKZY?P%_9W^^zEQ(v{AB6WOr9Y$nxsg0aVN$7eb1y!@?~+)iX{ct!>6E;o0z4"
    "AWDta<}h*?3hK3soCh;O$IbEOg`k2ZJwt+xUdPDF*zm5yEea4jd!J#ZCBw>FFAe7bCt;tGax5rfifzy_(P=b*d>$eI9%CP#n"
    "JutmS$A%#~mdtdL#I!`)OUp$_y+h&-<y66pHrrtMub#Z^bJ!Gli)xa#@)6bz1<r&bEG!V87nKyFU@QOczWjUz28H*h;Z8dA2"
    "9>W_=ejgRhuf`E|6mM-sXZ=Vu&Gbug|A(rCCZt&;Ux8uGP!Ke~?3={Zmj0NflEHN?fIaB|%7<)OL^9&en<N5cE3w)$bY_PN#"
    "xpYFqIjMeKGL#SNte0>@}o$GT3YD2^q!hV04$yUFLfE!KLCWpG6-<zAB-q4@61-Py217t>0<+0HgV)0@tRlZ5>G^;!viZzG@"
    "E~W>ym?~Q)!@;3xso%Ut9#e0(NGzkvRen%5KDzie_=G@zz1~6>=lC`agR1=uDSDPv7-f^RXY3Bedzz6Oo0roo5+%(34Mr#W%"
    "FV9LNugh=ZCe?c-d^JAoScPlhcs<`*;ArXZQTaY=XSK71xLaB5oW{6pz|J7J47eV*AwU6!Zko--{*(<&`<*m3{6>b8*6qi5%"
    "_x^{((Y(W?H@I1I`iUqgNWWmY;NInO=-wMs**<{_UXFi(uYr+^NaERSGJT$5b~^n4k(I``(f_`&4>ZZ8v|hWU<D#XGH{^GSX"
    "-<J_JEe-g@Y7gHZ}Rv*ik#thu9_i@z&DZVrd$qtm(caP}DusjP*1_u&YcP`FXp9yai7ab~G5LRz2Q1(ar=*Z|+H`FOfVx-T_"
    "z697bT%&+jIE;}qYi(j^5(G}6&EgWO4u8U>t8}tCl!hELG<YJGtnur-#Qassr@l*PIWYf0Atcgcv|MnbUhk}CP*^J~o>)jsP"
    "Nwzo^b+4EZBb1`6i<?)vA?xBRopoC8PYBMI<!QSzI?Tqw)n&ce7@m|$jt3Q;3VzTw*fbOwjiP^9(-p$3pnRBf14CSqF5Eu2g"
    "#lmKHbyv?9e#`ptN=gT^ZKH1Mn*pqM1!ff6@Uek_HTfxEjy!2soz|qKmz4O06brD5IS6U*_Q76#bDkkA&X5PE@_aG>~>Lp`I"
    "%^QHWi=wBDcqba_`XEC^yEkF8Z`x+8hQ-v%1}Gh-1NZII=wc-T2ReOpgg+Lc95ooHfIq_(qjm*ep)2_KWB@%GRvVrR}A3$AC"
    "lkHjMGXLW{idkunNOHGrGwZ!=+I&{~el)O2cAZJ8Digh|(>=+-<BBKH~}qs96|uhrk<S(JpMbx_3z877R_$RZA-otL#}JY7)"
    "iN&t;n_pWpy!8hp<*bZqctYXD0KN?1ylvc@S9(m^bK+lQPfDt^xtdV0Yi)Xcs7QE7~3Fq15Tc2sUr!)M_RZ97B#NBz9P{ftY"
    "&P>VZHu#4vgxup%>})s5aiNT#vp2OE(HHEs>A<F!x3;^+xRvtB^e!0wJ`cmRK)OF`!Yt|<@XKe(d3SV~?A)=G(CVR3?$OaB8"
    ";>7b-r_0Cf3nIh`>_Y2HK)5MI1nTH$#B&gavcCs-L_h}@spwv*A{+TlS<EexAEFmHhpFK=)nGwdCnc!oTWW>JR56?*A8U<&9"
    "1CH=^i%gr92jPWgu@9G<)*ex4Gn`CY^}FcJrjP&4xEZD_Y&RiF4F@+~r4*33ARi__JMd@q3<)sS1I^8y>E1gSFQhW99Tpr=8"
    "S7hRp$xLsQBu^cT_z9#H@pxo>Qb6}WooEII<S;#3%$GMD<w_6uCpvK3Lt*LUJ7K~|I2%#>~q>Ra~PzA$LKXwzN@i-kulj7JE"
    "YxZ?8`YsCg9goj(%KEk(0QQj=C7(BWRJNTSfj2E@qs>{Yw3QVd8x;APW!{VFRZ*unx=XMCHMNVptdlxgOd7}S~8y)!cdsusw"
    "Uxl=)S)p>#uZM(ux*M<RfU+31XtFQq%ABpkP{Ezob|i)#9EIph=@qGBRJ+S=!To0=X_Xs$Yk4lTrs<ft%Z%8zp4@U)P0%A8y"
    "uRt`_NW({Cj~V}$12_`!RKS`D6NpEE_9xcE;px~Z6g^eLor6T>GKqIH%*m5gJ;SSGBTTuty@|?8>2Mm8*yhPwTRe%BYhbwrW"
    "zx)(eCglRKM)sjlkh!icHUwy92D9>UQvR%o~$fo(AS@kw5M8Pj<R!(cywF2Rmt6?$<qqtNNrlo!_6EXBX1uijCW;A=p_bEC^"
    "k#?|=nvvg!`P&CCFeuQ6W+wKUi2CdDHntEIA3Vf@{U9WVH8`QUo};k$@dOHFgz87AxT$j~t1yjVNK#ls~R?hT^HJ@iAh70H*"
    "3@>6m!WV>_f3B-4KLe2mkF9F{Z(KSD@^!r^grlnRbx^E3_p53>04|_vguLd5<uojY7WvlF+BRLV=im)V>c8|nFm<TOJ-uVf%"
    "m{8Nv=sp5nrYT;j`ANC=W;bO$EDKrFXivV`I|o*sx6b_-D|F!tdSsD>;FC{HDO4SuPVM>VUMTkT>rS<{<odF5?Rj03m}TE^&"
    "U9gy5X1fLxRQ4vNI{kLYkVBnm~JDjvqi6A$$HW3huU}HyZr$a59GHYjq>?HyQlMU1bg)DTGMM#h9$}Ds-s*ifp7HJWoAa@h3"
    "L~@yg>n0g&Sg=2fG=jy@1=rMRAaNdZYR~sS;!SB2lA{lFR2T>A`gaEsrhI4cS4*_X3PHN?Cd5(6{4{aCPqFCr>#%-z#rEpMH"
    "NyHT0GS+LD?sFJBLB7wwgAu>Qc*Y`eHB`;J(hf<pZKJQMje=qGpjtyfUFP)?&ubq9ve%cil8M9-U(1!xg5GYGZtTcu#OctAV"
    "tr*L0V+DT9iufslDDc=(u6O@_u&mLsQz{RnWyizG~lFR%zX4uMMlT;9uB@-WOr!1L<v5c(6Myq2~uRWq9wJyKKn6Si0t+~sX"
    "d>>Nu`M$RjY`AbSLJOFYEhHGs3ngTiFkJmz>2|vFQ@hhW_2(u&SvK8Q$o_cDlr4s#@+)y)&4*!8(-3cTK{TQ@=tWewd1K!jl"
    "oWlxr6s!7N{ubDs!j*@`EQMsIB<dxoM&N=2u91@NzsJheYeYbjHd^NRAftjJ;uFQDZcg>QS=IN&$_Yp8&}nVJQ%xd<+`ik<x"
    "OKlejG-5vy7S$(Z6|>*0F}Us~c2&T)rn+VQL#EebX4;TIwa?B;gof%iXl`jF(`XPni9-5pH&U-?0`ikyYQ%{n)(T@W*11nbF"
    "T4EKsxRK$I5M;q;d!lomBlZ0c35yRGe4%}c*N*2zV&g*OsAACMlpN@t;WG-_KiKO^WT7I;+S3J$&U0PyYlZj-*RC_SCsf(P$"
    "gS94Qo#m^=~c@s2e$4Z6xQ2MW%Mj+w&p@?bS>ts^u^G^RhI!@c4_$)5QXZ;9u?rp{_Z@R6+s5%_hSPrdR$y%Ab4|Q--Y0Vk8"
    "e}={zxA~aA%Kr2KD`}H+Z}t>O!X&&4zlVO@6ZcIpvxZ!Z%|K_}BJW^}HzuuS^C!~s<{~LS-J%Ar8mAfYoDJLPPV9=SQ}aI1P"
    "rw+LZF^cDbyKxfIcwQv8bSRuF}fFO+L?{W-!kp$S-6nLt~er(9ZCy-(yH}%4I&F2k3P}*=HB8c!|2tZGsogm4<|9>w@gQGBd"
    "gdCgy3Nm&-ol!(d~YG;ebnVD%=jeBABQ*=o6t#Rt~&Dif#(U$HwXe;&4)!x4hjo3$68XyUIEUeN(bw=T!wqF&Pj;AF$nPtq="
    "*{^NK4qylT7U8WuZ5s{hQ>tdR87+nlsL?{zU$relk9Vw4HTwzcYS-Fg`Y@0HV~)PC!`x?e);cbK{B^|KK|1`o8b^BpIL!)Wj"
    "lid-oM>W6P*nY~vYeV=$g!&%EZU1rKNR!5=CO`gD^{*nc(WR}_OOV~}<=H;+Z>-p0Y3K}`$W<Su{&xfCI__fdy^`jsxmtSKx"
    "fF5-T>o&>0d#v5IpOh%nejSHgYJQd{w@hJZTzf}zIgx_<rZGnV(RNk{iWvHGuhAypb%ctr-n_KNrsP=OheYa_m5*+G=nN-JS"
    "o4<a!Owiu==s$>HOMX3Fwcuyb!JxE-zXd|SHR@SZLrtil$Potf+n?ScR+&i{8n2#ol)~z3lFN8Fq7p#$xb=YwVV07knf5~s}"
    "X?u$0c?uN?FTG<Zw#|T(-~d(7jx*bUL>D5ce7Kb%*b*A{s&WDclAfWxVk!D}60i#QL48pLrF;#_0TgrTs#=t`jpM5`K!Z`{S"
    "ud-s;pc?MGR|yGxofeQ&X|Hq4=*O~%j4<KTCGd(Yy+t{A=f?23+9ExndE=LO5m_^Ft`0=-q+nV4QHk8z@ZLZ68$DDcazXTp4"
    "umo>`Eg+>@1qO-o%+(PNLzU<HzWjUv|ZUnU!i{p)j4}O1f(;iksEB=pux=I*+xL{YnD)3I#PS7ld`r$JCK!&?>+b$nU!xf8E"
    "Q(|z2RqH$}bm-S$CiWnHCbk%?1f1jIv+6t5lst|4vDIF;_QyJ-1&U-I8Sj3%ezRraet)w5XSJS>1*dA{Q}Vk1!DVet9Fefjn"
    "%jI@kfbh54*3#{5Z=!N9Or}($(nI?3q)<G<EzD`zj@WvLGi5k`NS_qzo#&$vV<?Q#c?~_-py7E+^H^b7sRzc5o_wHP#+QR+1"
    "tp6B}2-xxv^*161P0xgL0yJw_8dJ&JH`C*F)Q;Bz>2hS#o&O{oBs6dZireVx7CW|F+F~ZuCxT+n>f_vR_S*)|y^MxIk#7q2Q"
    ">|NFdEQwtDP%ZRFN#ty%L(^movpd^dnA)Lc<>C9YmwMwwE{qgFq0@1agF+8nvle*El*%_kdp_$_9nxIL+qGN)G9(siN#!-EO"
    "XsCX-BU72nNfm0{x(nWk<a6@oJRjve(b)Tqaty!%RTlTjo(wa29knL*s&@opRHv#9Wd}Q=A5dG)l`fS?+uIAa>0VkVlpjQUY"
    "j;2)+0(p~-YcPi{<bJn4mp045w9ky1A?Tv#6$U5ZkTjp-9>a|F<6BBq=7reNYTMr0-QsS`J?s0E)<gIih5P8uoQ{CU)*tipz"
    "CK|0tp^>RvMr!&Sy&q+?@GDZr$?2cukr#wngy*{MJ46-JlIoG$GPHfd4Y+9!MTE8%T0>I=I-3>EzJ48&6Ydea?yG|lj?GQ(t"
    "!6hyez_*u~~OmdyBBA$euLA;=-NMMTwkP@OeN@<Na^XMBhI{ZnK`RdfWO$VpbDEoAUxMnhk>KDWzw@w7vHsZs+%gIG0{(XjI"
    "*8D`gJrMXglr#KXybah)<Xhoqp=i&2Ao)qG&|e(a!WuN*zaTGso%TIPi0`%xm@=UebLK#K7MJdn_pEF7ER^27<OV#7#$20iK"
    "!$&>e`(1RDtbG^4&pg;EeH+rSjmK6AlBk%w(e$75?>__GF;iyDehh6#*m6!Uq|AH=fpYMqWo~vUc;;{#wqo;W)mtA9jy`8ny"
    "pw}qh&T-pH#=l`PwpO=MUmn=U#zdEo!@;_kd3*$t>#Tt3R*@TsG6C$d6}Iy;GFAOtCok7cKyg$`lva>gE*_tgP3l#E;l`5&I"
    "yw`M*7)pldHV*{sAc!NquxK=4{-#k#=WiJ#%@$QHKGts4r3qETrpg>v+D?ldXr;Nje=t<s_gWACGK@Nwi*34@Gf3^;*8%Q;@"
    "mzy7MI9;UHka<mJMUJIMpV!zwwOXoZ#`<48`uwMcv+eOC&4QisZhGuCi%=@@%JANi63l-jU1ll1^_w@wT7wgPT#~qH?44TOs"
    "dVI@F_A$qVb&VOr5D-1d|!h4tgA+E#5@Bx+oDGn-~dqA;F`ZnKX)-qgPG+kp3w<ym)#w1$CMW!po7>VjoCosN1FTHW{1e!cP"
    "Bl%+NyH2RnF!jfaVtM`A@Yl~R}vt|Bfo{c5@4#nqKYMlm<Il`$&@=03~vh&GAo_%Jd{MJK}uK8OuYwwGxUcoGsJdS`vlN!a>"
    "!8E>akIS?@H{`ZVuvXYEK59Ae8g$NW^H_kNG`(_=R@q4Ahc+_?yASZ$-iK8H?VPtAYxP(huMNA@A`Cza^>4ZGMAWJ{P#zm$7"
    "H}nYNgw3EW(){_@+0B>c7X#tr=rz@SZ}E3v7vQYjMNP=Yc3c4(UKG|nGtEp2~n`P0vuL__h<LyU1BW{dp-zMidO^t{&xIa`u"
    "J@DmPGwhy07HUbfCTxuTM|r;rh2;SgBX8eJG2&%iii%ls6o*1(U|KqH}&Uuw~ZiGm{U)Tb!U_)vmRZ@~&5VNmXu6>|7?6iXF"
    "Nk2#vh^B9sN&J0t%mXK%W#ESIH;{*SA8P1I1#9YX{ZOF=|I!45$|tx+^03eqE}U~}!|7m?dKRhg&8ILANR4Xnn+Tys6Mhm30"
    "xs0@Gx5-1z4Rtg~Y^qZ=ILk#+`gJdE4szN5Mxw2ODiv<9O_=3OmM#Nh_KkU*i38^cS!al}lugu=Fj~whk3uej2cK~p`O$?L!"
    ")v?1ugEegOi?-IdEx6g<tpYe+f~MQN>X+WZUk)?Bj>|5)R|~yrOUdgrftoL1LMjL4K<|kv%pm0vot|L;I+X`m4j=^YL(Ek@*"
    "Hz$rUq08`hRK^%xoB35-5u8N-MX=Og>&D<;g6sv&5;|vOUA$AYi@(8R;mf&#Rj#wdBM#Wz8vHbw5{+2tj^j-CKlL9iWKygA)"
    "r<N*pZg)%YEaOz0m95$;>;|hBh=iwa_bovPzY%Tvx-vW1B4K9xrQOZgdl$Dczm}1NSgCPRWZuY|Wg?{6d06EY1HMXwPaa)<t"
    "WD@oA^nTDb5M#1t=R_v(`=MKuJCh&0xCjB362<1u@9jUpx5<vV&t?dyYnIodBoDnh-3cr1-bFU2|LCbNBNpvF78T#(P^CcJu"
    "Fud?xKJ!qFn`%$|5qBteKkCWp)sqPIP+mb$T&Oe}P&%dsqz0Rh+@V2$19@X8w2)*sLy<PrTwsTz5CAZS<E7IdBmv~UOmi^nn"
    "s_x3gW_`Bf&kJF>3yH-l$)>+56m4(3hCdVScm=2>pL6m^S%o{XUy?`ehCyjLW!6@wLfq%4P{zTHM(kG;)cl;PTTpqef1q!Y4"
    "fEU}ut#gtY+?LxvZ-HDvfeIm4=fvuFn0v6HRW9>KS}a}wB}7ruIM9|eM0H@;teb5#=h2L9&#n%)PfohQ5PE2!Ss6DQHE0>f;"
    "P6Fljv8Qb?>3m*`Wix7B`bR`>LgN!E)QL4N$a|3fk;zfHYXD^panD*a_d^26|TD{&F3>z!BUq<5T}VZk6xvWG!8Na_H(gYIR"
    "={+mUW#mw5;V008S=vmJY28>FwI5h~NW+T{RR$}9Mhq)(c1UXZ_PulB=vt;=Aee8$C)5$F@R$Sw!DSWtY4uAx#59^B^){}j("
    "$f`;&@-<+p|n#__TF1iw)O`s=I8$NSvQ6gP<B8j51JgyGQn!!y9^m%)n|B4Z1g}|R|-aj7CiP$H3#k8)=UH{P;JoZ?dJifwg"
    "5%AGyUbLK7iS+2MEOeD<gKTkx7%Y$vPzOf9otR)P=&ghODUKAMm<CUF$0$8yLc1f{zwt|i)*4S6Lz%#Dk3;dg#nTA{;U`)cr"
    "f3N@u9v7n)?={@vc>7u$}_y}9ch{9hNkwl01mYeX_8K*ix+%`-pKv)<T!4J`3_6?#w37y6_%5!)hP6q>mxt>;6`jD0VIo$(2"
    ";go!Z}rz(M?sXJ<r578f-CDB*Q^1Y>lb*sbQ<<+T&M09iVo%wQlvcmjY6NCO7VY^pIJ34kAr!XN(Iqa8qw~tSSpr2F5vI>KH"
    ">-4e_6C(>K}`n8_Ww(0+Hjl^VJ1`^_mqm6~6vH1r8_U9Ryf%m%S_KvOYO+II?L<{f%KZ&EArg?i0nit1oz+^uSHmxgY?5K_3"
    "-iEnVSsZAfV0&wULTZBC6<lRe*4$+OQjSqPeZEFQ)cl%}iGn9Ic?4LTa*^Dxk0nM`{a>6VFhd{r+P=Rq(|7ngmloGg-pDx$-"
    "<hezX^G(R%Qrb79RcDqLd(+GDA@2LZ6oyh?!MZg?jLkuC{z4t$(l@PhMwOKo8;+Xw^y%jN?QldEeSjdd>#+j>%bzLv4&F8J<"
    ";i{zp4tGP6O;4n01L+&`P#>kP>P19a<nA|&D}-*8t=`0u`*ipr!L&RfzSy$1odxzlJz0bl#;$Vw3TzBxU^l0@7?D6@;dJ*W*"
    "_eh`nc9}8e(Dx`&O9`?3L*gyX_+>mk<XSk<n-{jV4KX=pTF6!fk_{zr)tZ-E*;TDD{u%P`s|M%6R*N8~z6S#*n)TV9wOWeE#"
    "h4L$$@m2QOSdk-Kwj-m3Ji6Qr+#U9L1H{o8FqVM&Mxr}AHRMLG?4#l<WtO-Ax?YMmn7IU?o}fw0`2_fkkZ(Mr5sJx;&AM3ee"
    ";lQJFHGp^0qxcc6EQM*f3pt1WL%E|h%BBf~)yR%B`4j6i9Pga25)St+csWhhh`$F&W!t^{33+6RY_g)Ix-Rv&R^W`eLudgFg"
    "RVKgw!~GIJmeAycit?tqh0blR;3J!$6U{=-Q8#wijW$FX@9o7tYMycXHGzh&eIazZOk-YEUHnXoTds!`1$=WLj-%and-gT_v"
    "M&fNa(_+OhW?uYqg9TrF$a^A<;v(kR;~o8kTj9J9n=pN^t2#dX8o?C%(uHYFt<A*8Hue~VbUv3>Wq4i;*->$jote-^4R{?NZ"
    "An@ukUCyEnkoFzCP?+_rz~n2<kZQU-rm@F4y4V8?Z#Hijsp0Q$h2(H`XMpeH}osmzCY=au+*xG>2vN{A^r@X><a|GLrD3;rX"
    "NMB;YREbdS5x{9=^Vm}p<vDz`q8=J{84z$&dD(WMta=hWmj-U2@#dfA2I6!*eB9?A}RO-J>(4i|oE=!B~N7`2L)tg8~(+Rf#"
    "45zyQyy@4GX3IVY4OKk7SW!n!+wJL^e(E?Tg8ZvwBc9+NwxYjozS?BM>OU%4jK0NDwkI$MTVF^yol~N8DyAr7Cbogk=U0v16"
    "!&X57rb;ON%%zxmIaE}s^nmpB>;5=um!k7)Gklz8hcDAg$e<W%lU3f!%JSl!Fu~!}Dz)NAyGK;F#HG>ghWfbVpN5y?bU>-_a"
    "eOPu&luvCAbYLM{RVicjEv^3*El}7d2|1fk!@@g2vX<aLF3XXy!?g%OX(3`Dl2earL|JJrAuRWt!9Nw<1O1Y!F$Bn3PMjqh`"
    "X&q?ZVx+S83Vn>B!i~7yzSAkCWXV?+3HhWlNuF3>qP|#yKA@8M+s1Z%iJ6%6F4PjehEix%>V|hWy!JeU!Lfek|>Bj@U#3Ptx"
    "#f>TMRN5<X9nJgC4bHfhQahq4Q``Ppci<&`6sU!xuVkgD{vkUlpBJ9;X9rE$zLPA>JiB%h%XMQ+6Q)gKRDcUgy@McIsIJ|6A"
    "bMONuT%=1@**j|?<R|HG!WM7)1Y2aP#jCloZEjfDf+nrt23A-KwNpoW-dkDr%PnO}rM@G$xAw7(FRRw09P7`eNllJSgIi$G8"
    "8+)VUsQc}XpjEv%_P>#bLT}Y>tuOirO#4)6Gcq68HT2Jtf4ViRwfd7aW&!yaVt4+!yDqwi(dv+`%x6p^Kru*Sc?>ftSl^EYk"
    "K%dxCbt$*byMGtm~=(Mk3PiyD%{*1iw?T#d$+C*>!-npaI<Gv=r~9RJMU$AaXfVK56gfLo1gQtA3ddETkl*#w~pqo1aj>yG8"
    "##{UiW6JjkP2#5QDw@*x?26^Z*!^8;@X;|HgxM`e{-T&Pde2@DaGC-k=p#5YJ|h?{GHgk)2Zwt*WcWPO2@>Mc)9vX$%~D<6S"
    "H5qt=+xK#e=xoNf7a_m8(+c{=7MAB~Ncy{?w76}UFK=w=v}^V@{AsA&3X;cP3FW+n11J>Cz7+RoFPO^ml?o+!b+1R1n6O!(s"
    "S&WH12@R1)+O8Riv-b`Eg^{!Ui)*x<WC+nwt`4N9w;Lk!v7QupX?ZXJ~EuxH*_VO*rT5#_+u5IxPd>X5JbxZ#YfU#yrhskle"
    "D&J}CL4DKJh}d+m`+j{ZN3{d~^I?&Ysqd!a&hV#I!+w}famHwMBR)Uot1h@<6{UV|qOxjul4!rhIkFS={ZtP2HMuRPMyEe*s"
    "q69j>!dGf4U?WyQ?dB)NF9#ZWno}I^=0Zil#A55%l2zJU2&?ttTlkF(!LoD34-?Qje6{iVtCthThrL=>k)e@VZ~kFC5ld#%h"
    "_m!Fpu3bsScB0MY~sEyL$Kd^<+7>TsXO7hYzd?4kQXMiQ}y&_m8^hY?@r#XfoZB3Kn}Wa`;MIcLgt&*InOKZlHJMI+ehH%2c"
    "zR?*n*UEk;=gZCpT@c`s07ds(=NIp`NmG3BY&5+Bv0Zlw$4TuhFKu}qWmbE8<O5V!ux<7v*ub6Q~e?t9Z|)ainw9v@@^V2_`"
    "m*9}nQ!p3U2@`LW~GOtbo`btqhORw7tJg3_?o!+lQY6o|WzQxTRt58HTFI!wrOUN+d=uV~}SzKLavA!@Z2^Pf0vl5&Y=k|Gb"
    "NhD$?LaZm>5I(otzqn2sEVa2kfES#n<aeoe+F|?CwrSM0mr^@W;ZMG)ksl1iwXH?kQ>xobdOJ~wIHoDI81or;9kS=`B`*tEZ"
    "C*U@A&ft0O#$o-4nFrAgWZZ};EX=ZPB@&+nhPvA>g|$&vAR$*I@?G205j3<CiNJ45BC;vNqfO*^q!myT5VyI=*J$)?zzU#z9"
    "M!$o|;UNxmE`+tw4(wY5TC$P{QJI!wM!S?@r(#nH?_=dNoYSR<GOo0GS^oi{=%z){KF!Ee71I4cY5Tt({(Ph0RKmf5x=buMG"
    "QeL>38y_=<u@f;$9>+S4#aSMT)_HL{A7aj;JH>q7*`0|c3kFWdHg9{Ozq1#SA=DTc4f`d&BZ@umey<2zwK68k0IyC=Y~rB+i"
    "i>Y%vjnRcx&G#bB-4cBNtY{6~c79Q`;rP_KByDIR(mh{F9{j4_w&ltQj>@`s`zA>Q{rL}IHy0$W}I1?uHx*HG3yLA;Bj-}iC"
    "xjS>I_^{n&)=jguK}=N%da2t!mhH4J3*ef4He92PCKX*5%@Bf!$(EXxdxOihS+<{w*O^Ea8|h-&bzO`+FMzL|+2O6OR@<d!-"
    "@}BsH*QQC#8nS6>hT3*x8?+w53BC|8`LERUU7R{wc8yFX|}GDxi&wOn~4wbGfLPgv*yf}tB;y`T0mRr#g#<1)XKnBHJuc`Iz"
    "gm6+TdneYP51+JXOK*X!_Xic6l;(EBi5j6{mSCY}1vk-I@91U^zpadv+_$OREFZJW^1R4f{DBiSy}kv>vWrqsbz!dDEtJ?oO"
    "<vz!uBjSUJrqN&N7goMvBxp|;PVNpgI8@<dDwTR`VdQD34ENwj(!;6SS2fyS$W9xRkiEp9d$3A-$Z(UH#f#=SJ%?`vhNJnKV"
    "TB2e78CZb8}xlZdF3YAD^yf2&17yWKDVf{>>A<bPAfkd{0S5m4Lc<lolm9caI-=Ce<fX7ewe3~{EI9aPLTztCGaiOO|F0yIB"
    "+u2aN2H<oC^j<Z-Nl_m<a~8$cVSI7cmJa{sAlj-WpZdWoTw=;x2N}(Kkmhl!*jW|fHq^T%nlQd>%t5ye&ZqrH`4$Xlw68ZXp"
    "w(5bK74vGMzmssdf`^EL8VY$TIs61`r5h^eJ=R9pLUolbliWSf16sPE;;CCw46&bYBbqn&m|~d2ff*BL6_9!Bx1&`4-`*{%B"
    "e|SD)^zCutH;+t!m!&>yfq`S8~jm6eS7|3+D$d(51q<Q3+RV;ytNxz;ZP$TruPL2FjnEe(=swKmhC^NTaa0+;_)h>*oyiQ$g"
    "uh(l339KU<>Oabzbd^zP-oU8xl`k$bgTESWdYrfzV}8)Mw(rF!bOi>D)s(Vv68NEMBr7y4GWw-k-N9eb|wKBwKHmvSVR`QBK"
    "dtH<R<GCw~U{p)Fvm4yZ=Kg~LS*qKmad@g5;wGeiW_XACxndj`M8Gf@Tm8aAzsf!UAo^Ie;ZyZPY{qAhJCT8OmWhN1B>#UbX"
    "V;J_siK*5%FwegUYumf&w_qD^KNoRz&qCkXh>R#S^U}P#P1l(XKZ7Z&L4tbzJzOADIgU4ba(ui33Z))qvw0YhEvR~cZ%qZ?i"
    "O&j|#-+OQ5W8nvSJQ?vjVrX#k(j*Hsm!<{9Ej{x0mfr*7E+U!y|9oI&~B7PQKpXv<X-uW4^B4-dF)I_6SQ1Av%tJinJz4^k`"
    "JmIky`S`DI7Q2XDm*K(nBv!zPKDGY&EfqVB6N9>Y~hPv{7hHO0(xqGZd*-UY2~hauF*Fs7JxPjqv#tff~WfG*Sd!ww~p{ODm"
    "=2`@A~qc*3G<rCn|sJ)=4i@``!zsO$4L2)?)}RZsEJMp6;0AC?SxC6#R~0RHObsh2iDw+OxL0@K&hj#^!^m+vZ*bk*zL+hw}"
    "27E$RVe<8Q`j(xm0=(JIvtV&)d+!Q+9*{zLFDp|})zlPeYY>g@mkpMAFBLoG>!m-cx3cHy+!CK`nnsQPJ1Gr*g*y${t^gf>K"
    "tj3dzb=8Y*T~Bs}`F_<&Z@4I~@6xHk_j|>|dK(9a&9rbgQO7rC`_Tqi<9G^ulVRYZ_1NB}aHYi(<<_F~yj}oJ{JePbP`aHUL"
    "Qc&!lk&V~RzymJ<F-Y%FVHr=-3O~qojc>kb^D$aYvbT3xO6NN$+T+E3iHUDmBfDItCKnOFX8$Dq~UO-zn2-;>ygslMq1U_)j"
    "ZW;<jQF=>-qd@t1p&nsmKm(W;$IAF2$WN<hf@|5pm+~;KEC|ZF7<)){fnaJbn2!9+zhp2(j!thve#tb@X-9&`Y?_m0sXY!1#"
    "(cSh1~@(d^-u20Ln611=i2M$@ikcz7FvYP0TTqI6z<%~FcJiy3-0JT1(dK6;J5xnE3Ty4ZnY)9#Vp<HY^!ktT{^9(vl|fBMA"
    "0r00bi)&Ib=$exjV84PggL5KR0j6Q!Z+##!mz3Mt?F!<NCVOWRl@C-Jz1;e%RO^b_~aCKjK$GD&#(l(sU>Gg)YjH@`X*v#lw"
    "U;q)j+fECm^sXq_xVziTAJFz*m}bdxys$Pgh1f`Mq(+l7jcnh8od5Mc9RXf`wWZl+JbCv|t#s5Idt9H*wnB!jA+}Pm8-L}o^"
    "*oI%+`t|?+$2v+lb-WNaJcOa2b%f&xYgNM&M<N+J!3dta_Vb2+Iav_-wxy>J^wvhtUCAOy@<u>t~<_?=fV3ZKXi`;rg?CcyZ"
    "bDoj2bvgpeWLMHY)=xgJUIt^qcma?ykLa@5gBV)RA(vJkCu~1;25$o019eA5f0(z2J_o&90g~td170dserSOOAH+7D3LY=H1"
    "*gE1bbyRY9A}0sVESFSIBlztx$j1B~>Mdl`e1TOKTpxYGYC(^syl<K#FoL1ddZs25yfowy%$nBvqU&x6Otm&B}986A_^VtNk"
    "=b$5`R6?v)?GTy@hI~U72`3vgC?ai!ZbfGhf@$X9xZ5@(pEDFM|0*!OnLX0$*>+o^@9`K#Mq&QTh_(68DbEH<EF^yKi?^G61"
    "VLZ7NYU3-Me9+j#ncd!Bd)++doi0N)tlQ~%Q%Yu=LRO8)N>y!}?RtoG;TG+Myy_IKQ*bS^RBg@B=-|!>;cgqi^AcF=F=uUQi"
    "L^MeXk2a=s%6S=EbAI3?#4-{2AcQZWt+Y$o$CGWub44+>J0|MmxQ4jadloze#P@`lxwdu1!`2roAE*hhfyngrIK_m##(Je*0"
    "C86BvtRtKO~~*i>)xQFG}25#_^rJoHo!nxGgW{gjyHcgFNVH`$Itv*1P+!#x5IA7whTN7(r5in9tEYAhL9aBfHu%+`}|+JA|"
    "XseA1s)g#}iwFZ<P{ZbGA0jy9zdqvfnZ8cGDkFlF!1+kOFL2L|+F`KD61+}FdJ75pC4uMPS*vRbjXzy>=KIUQRYGHFc;yaOy"
    "6iaKrTeZtG^t1oor-D~R4e~tpX&RsFhvHfjcePAlxP1~A3O)Ys#c%GM0i|F+^HY+>lK^|TY=GPW_+Y^&wATteNmOX2A!CLU|"
    "hCfHE1l}0;nWr}e?n%RYtnS+^w+`FM{^FX=C!nlh;ZeCxr_}I*^~`H~z5qV}@i5JXD||H^3~G_#6h$!ctl!){(dlbz79Iuf>"
    "}ylCC(Ve}hEZocN+~Ecl(;#AlW-zQ(n<y?y@2!@55P*!H~ZBe>g<)xuetz8du$uQn-}y@+E1hiE7WCtmZ+?QV%yJ7sLl1q?`"
    "et==IK&be4g)S*Yad~UbcTc63Yw4CB!mci(2Uw>)RO@MpoIjtTOiiY~EN{f@@`<GB;XwhRjT&)p+%#eMv92M(^t$e3>4%8p!"
    "z(gAr{uqqc&zYsQcU)wAK&787(5^c%J^*+k+<WqxBji$PMD8cwNH$*MKGbQzod7EXzzmK+}@*O0gMRl?1d`Z-(KcB?GwUU@!"
    "PeB^+*d|3Kr-E_EMA5Fe4Ij0wbBk#8)`5PD(?!z^)Fcx8tv^gFf)`H}iqQTyiTae4LX*?fs7KO^IT5m(ZCib?TP}dBsc6cAC"
    ">lL^v0_du}O+os^H3qrhf300*D%7TTBtt>!h=DdWTB5lNmv>V8ZV!N^ySrQ)jfj~CyTUY55Tm$q(ZRA`$pyoxE^K_ypI3r{T"
    "=m}De&p<>@l`u9&1Xa56JS$fy0_PW<I9XObq%-x7$Z@WDdv^N1TVGakeE2kb0k~ausojXO`ruFaWZuXry*64(K@RS5)>^N)#"
    "B}RbN0m(T)ew)^$3)IBk6uzLzp7#rnyPo#tJj9x0enz>=iKy)6GZ7PEVjTXmuV;O?enxKHjb4>FrT0`_F#YobPP*kelNHSVX"
    "@D9}_lTj@an*x4L4wKslL(*@bPpaveXUU^c_<i+hwVBim>erTtH-janU1&Xvxzx!?nOFn$yiSK7224`%E!n!no*Y+2rC*VmU"
    "euX2W~?t<&IK>dL-m&Gn=ANNVQ?~n6xHb;?%M=8(y5`R%0<)lH(ra86CH%~Vnkx~oq(YSLN${4}ZG9&*W7=3$xJ*tVPNZ-#l"
    "b$X&^SUd!7*L}Amc1C^eiYYwBmUT<MT*nfe2%n>?H8TcnDKA*AaGRv}`c57e+7BzX3TTE)Za2e;zZ8Pi$8o;tEH*HwA>8Wty"
    "XjxoF%&CxeR4Yv_QbH{kzLJ;aq?Ve)<4B9v-#Or`@x9k#K&#DPfFCIdj*>E`@S3Att&GX<x>rWO~_i+LT_0BmJ^m|k1b_j71"
    "2gF!1w3hp<dX9<erQRR!J`^<B-+3dOYu;qjqNmi*KmB9~}Xg7!NwxRugBtf!pnjPezhk;zU6Y&GmdfQLYE>OzyD{PbA9o+6A"
    "xIMvJ)tsuFH&s_?IUT;#vUJG*+WA@$Wd*5wmCAJ<ZvRC4IOZTCq*B%X-3Xs8V$%}-=Q_4UPxvDq+iCah*7fHuBgW8>9Mfe-x"
    "@#!dFE?T`k*YK0}GIYGMX3hSbL{w+MwdN(FYyeN(EBG~13PEVMe>w~IrV5hf25n61g=PP}u<<?gdq9ox{C9Ud!@Dgx{l9bCa"
    "r^D#L`%2-1WrW7tDQsgM87b=0m(Kepe>b_bvoP${&c<ea1H883n=zpp>%(Pt4Ih2Fm3CS<{<`QWaOY=!_H}yEy?~4DG@c+2t"
    "(jE`ecf4YQmyiKE~ERcRXm|l2OLzvE(UWK2ZQFV#=1K4oBS;(24*{~oTMh)9DwiERMc<A5%OGO4jB3dUp<U3Lo%^Zb1t4IlL"
    "I;ARy9cPbXK>B@3E)oa0WZ&_Oo8}ko}DstSZ}3xttwIeYP9>X;>=3R<rEbojEv1A$YjRj*F{(YIt5{Cv6b#5|HCkP;0M$<I="
    "VZ5+#{tBz!>Q<~8o7PF@|?x~Gxm4dbxMxGhot8QvXi-Nhuo1R%0)Lx9^VHQ>C5X4^^{IB*{w1(e}mQ8W@{yM~q5u)K1_<FM|"
    "Iiir5DW8}rd?tm}C;Z@7d@}M&fXg_@-U1u$+$@XW~ajCUYeVO&<(D0Twag@mBtnrY*+hFjKMoV?BOADh?Pa&i6Zgbn((}#C("
    "ov$Rjspbu}bfcZpDT<a1!@Lu~?leBX;E;Rp_WIaILWhY@AUJQb`{oN8puJ=1>*-zsgoT%C1A734PTt?0w-35iC^s(#L{|&)`"
    "$#KT7mOw)G;nbCh20hupz~<is=TFfuUrtnewmZ%)iCHWlfq11mDpL67^dyAoLTKjKf6)Gbr`J%Gx3A^Z&k<eYuND7xZPbRYK"
    "Kd$m~=JdRk|agM(dt693777X0q3&JSvwCT<n_gS_FRGJ?gA5dD}lGSVqA101h0;9tu^0x&QEo*;Z~%y1T(`%8+aOIl(l$`%V"
    "4E?yMWjJ=~o%uABw38LX7JF7@1on0fvC{d~c$kTpN@OR$2UyO0Q^g9X0uN!cwgiv+(QhL7&-G&U@o7+#Cda6ychI!)B{Mr+1"
    "S81L2X{EA4$DGb#+<#}+@ALXKDi_LC?RVxEVz?Onv#1GA*%4hE(4D^WfxasU__|B7?0hx3Kn%yaAD`B4kU(CAP6lgxbO6%FB"
    "8I^RzxIV*!P&|^`?qJ@)%U^p1L<EK<Ebpu7KtfmK5eY%~U<pI6M{~4AIQXtv+}`Ho5~uomNTg0YMdnUdkFznP16uhQE#|M90"
    "$#%#fbzp{02aCL*UQ;pSFbJs_<5;j9|BNzHKb8#S35*z{LEH^TfAk-TXHPz`Fp!zYo_^VJD|T?j2b@9`Jn&W(!EZrQsUmLc)"
    "kk}p{K<+Yr!^IR&5V|MLuEIl-C1l*uBpOH@oW*Zu=)&x^8H%x;Q#td6)$uRB*_gIO24mc3+-+TRLuSP7HOal-BpiX$S7Y4{9"
    "65w)Lvj9KG?4JvfsaR@^r@*U4CT+U*u5c@b34PALbGl)*3?EUj*cR@yRg7(!Sx7*!o4Jaq>vs<$c@k3+QNawCJm8pe<re;u^"
    "ntfR&LvUVKagmFIGbnwl6XO;7H#T|_M!HiLfQ%DM5HDSJO@tJSdU-`UHU++rjZoN~4G+~NBxfpwe>beOR)b@5gEiI(##p<7j"
    "^qf76=6SbR_w2XLG*o5jG=bnn|MX6pAzo@`EF?d7-qo0eQ0O|p6=X!%?s#jeVXmuMDHUFxLKF>GfkyDq_f#W4EV^dz%ZC>d;"
    "W*9mZe_pC+r^2t%Ah;2kBV5TdcL$jiiQyx<-!8ZabF@^t#-EHia<8s?!bN_zg`QV`kZq^xGUW*!4h3}+1uA07un2R`LsJU-3"
    "PKqT$~A^t@JbQADdO?YVA-O>t8NBR#+Y6%@nOVWXmdA_tAAD0o~-)E;`MRWLSR@YL48Smd$zFH0$k3P_>wLWv*%73Fem@g;~"
    "FmRU3>&KFyP0X~iqa;E7SQyQlm4=NVWMAoeoim3Z;uGF)#vduw&XYm-X5**Dkf&njNw&CCAs^^h>pq*iQp)cM4YpG!bynATM"
    "F0j{kJjus8a@i!0rYk>-0)dMexFmRZDP3=6r)rLnEsCSxoqBjm=VXb3vz594KM-*w{pb0w*XSsbRUW-Y<OU?_c;nU65LOU>m"
    "l{Nz0skWq|o9Y_dn@R2AZ#^-0z*pwf*_D-mM8a(Hmecn1@!;<@{lik|bv8NNAzbZqyP9-{?oged*-xlBAoxS7ax>E{g~ts$z"
    "NOdmenTR-UQSuNCT#`v?Tm#RPG``sXt}#dq~5L^=n`PG2gHEs$ohlr^QszE>%uolJ&?MPP1PcK+bJE|PK{#D{ZftY-6Ww9u4"
    ")@T!LQs&TIeE8%A@C0@3g83qsX3Ilh#9(ZU-WpH#ZJ{gwxwW7@7I;VS9dcv>D`{TPLg6>91G%=o!_U@kuw?Q)xsv>hn1h+Wj"
    "JL?ZL*z?e2kL`T?iVPH5XDrJRDhR?nD`jZRJ5;c>w}e2!zVBNjH+!MIGv4tYNPuk3RKDuqYPbvf>}>4qfNs?~!Fw*P<_58ms"
    "gT%WUxVw9~lNiA&~Yu6i=P6U*9D>bl0d^1$@#DOzzp4c)vV+Z^I_}j)*D~RQSqgvJ_mHxJ>`Nl}dfY4jC-6hkzi=fEPT<AD!"
    "54i;py>A%Wv99*gACPoAFO8*)4c`=iEb8&vzV1%=Z=BP97Vay~f=QyB><vC>roAzL;)`9K>W28xgu}J1($h{jez8l)&p1j{t"
    "dvb{Jgi~-r_1syC_hr~L{3J$ar(JV(>=Ar>)F~*{2BY$+@xPatOgaoC8svSxw9;^MRmz36``&_r~B#5u3!TtKI}WXqgVH-QG"
    "FKA)B@cL60;v<<8uArcwrga`!Ky8PkD9-dZpJy@tw%8#H)>kKIF!31YK7RAdA#rz0tNUzFd1e#T@)yA`jPlW$O~}NwyJ*Yg<"
    "vuYsiobC(W8Kf7rM@h5Ow;v{WH%?Yi*s1Wa<n=EuAv^cUSQHIZZVoW#@n4Wf@geKqjW_oDh6ru+!s>j!nH39|yh3^kE(lh1)"
    "tE=zHEXdN|uRz9i1=000Z5+M-kEKcz4EANdv`CCO^$Mc3i=FbAOBi+-()mBs9DSQE&aXrh#{U96p&cxd7jDubgv(OurzMZAe"
    "?H*rRYiPJcoiPVtDVy>7Uo|x$g<Xb5S?jo6wMo`erFw@&EWJ24e=yp2FiofOd3Jl+YBxJfeyz^C(PtTy{7FE?idTRBOseE|P"
    "43o3g`60*)Kcfmz6v_g19gxpQW~Evs{Y=ZSdIIhYYeoyUSi^k9jo3t(;gO^mxt>?2~KuvI6U6G-K-zceaxtC7wxF))^ViL?4"
    "C!Bd<6FmWICY%WPY}L!e<*{8+gE5`u+vhd=t5F>XfxlAIA7=L6G}plx-i6;B@EUFRt#Ivz)f&!ufKkU%c*8dYv`7SiM7bPbs"
    "8mP8Js658#}3qB$vi&#&I3o-4I>6qjivbX=O4l^SQb(=l2^7t>HYO2>%{@B6LJ|8mpXH@R43=O1b@NUwoCt3B*=Y82Yg!R?i"
    ")>PsGM_pdE~wyLj-TH^*iFTP*p(ut)iE`Ntme0zn9c)D9cJ-8i0P~19l>moV^1{Dt9nFRAS&np#te6OotPSXRxPCKh!_qFES"
    "oE4;PYqG2_UobnBYXY4+m19lZ6V!9KO)6G%sgExdGq~PIx1w1RTfNoW<)=(-w=$2traC&6@N<Wt&S(K>mZ;9D3EWoFH@m`4V"
    "lYWw%?@)JBl-gN_UA?=k@vO1#FqDD`7lf|vsfSO9+tgh%VN4M0@rDB!aAh_oAcP~+5&qU?A9xG#x+3@nyk?g3S}O@RlHd%je"
    "^!MHzLdxeZ@<K;qo}kOv~`v#i2axO~^Zu&jN7y%**%TfLsSSRv&DnwfY)dtzP-&=8`{@>&|&#ZLDlkgHGP%pcqJ9NmRLKZz-"
    "-jx9NrJmEXVKd<Eq->*Kg$T{?Tpg{U6WLPuF=TtOLCZ7;o`4zyN{+I$aMkM0uLjsP|tuG@DiSvnhUE^wtfv8s{jEWV=zUODZ"
    "S?zB@a8zn(?)DpJw`PXluP*Jq{a<phz*J`EAjBeGY08*tl{}`aSx59&G90To;nvZ(>i&w917U%iNL}p^OcBFUIV+puIKAvEy"
    "aqrgyu2$+kM%K7BmDv7pU4KzK^6OTEjK0Qies!;{4&9<|hgoo(AG@LU8btG9R~#;hXrYy&d2;1?qwa1oEz(hx*7f*4)GsICV"
    "RMg<2soMf<7f;tHuPm7Gv)Fa#heM?5bu+-6yEK)a`_GNCt#q}YiJ)4`#53h#IZ30+Ck+hybd?QEcnouC*)0$JhWernyu)(dW"
    ")Ow4r`Xw-E=u&T;L3prBSrnq8Io$c2_TVxg>}oy74DXJb<-MZ(KWW805iK8{KB*(W5KZLJLnx1}-GcQQetv#O7DjGvVR{*WM"
    "3)Q<&HM*Km1;F1Z!mcbofGuIzqr)<YkaEe2Cj9Zcy;DoO8|b|*WT51WC3zQzI)s&-%uQg8INGt@1`2SN5&Yc1TyeR(#e=f*p"
    "A!+Zy$E5a`%4Mi{9EdBsoo(1N^GPu-UjG!?gNL+8NeG@laiq-iDAe)_AHFA>ivcn7WcMi0;7d3a86TaL!l}3ZTFvs>Y-rei6"
    "_w0?BwF5Lq%Nw$^si*oD_3iy$D&PIkop7L=3HH@U@BY9STw&TsezUMJ;hJ4;>`QM1+s+WfCRC^OyzNKDZ8N1cber6ESsxr4N"
    "I~58A3p_w?#u@Y*z(aVFcWoyMDc#Q3sARzi=&@{3CVH#9J9)|7N<HCb$uIf2YMyd_r2HNp6ha9w<Ol_8pa5YrHe%?ZTmjnYF"
    "2pCr(VO;;4uMDFi)T2puNf`J?^WYs!YjVFP!)GT?Ukl<{<Ci_PJeK=MVC6em&Y`WX*v=S{S|iYHgA+A2GVS8s!hs+gcWX?IX"
    "Q;Q>%1`*PA8L2db_$h2V`Z-Fj5*ah(xjgNxrX+P&RGKc~F~JY)|ak!0MtHabWX3@w%I4^GwE`LGgQb(LSedcllLZh$6h3N8m"
    "RiPmjU30foW0jhPr>Y6)YRe{jaV$iAdU~zuXPAec(T<BMugYJ3od8tMFRw7N-XE@4s<|7!C8~Y*j=yhk`#eAUSf>XR7tI*sC"
    "VW}>gl|JZW*<{n*46DSoAcD$F%b%~?sLUifJ?O55DyPhEej_vo(y+pe`U3Y=sn-@l<8a5f(8$#8cd|TgB7;8A(QJfO&@H-qH"
    "e&12eEy}-m^96{8rJ>ZsTM5fkj{?Hw%r&3Ulj=DyVu%BZj%#JxZ(>)TBoTo$Y(1y{s;z*aelO1e&W1^v>`e%b9?VDYe5(+kb"
    "E>w-=kndP^VFW#{3Ecy{xrVDi6Nrs0QX0a$Zygb8bLSzh8H+i+f6+%<e%LPUIe{dyNc*w1RkVQttE+&t@SOk-dG@e@~wFaez"
    "tn=tVH6imq`lggem^if90KBbJ43v>oCd?N`7HsJA_$nSN#si$>$Vm~P9~+*oaD)EU<~lMWB|Le#+B5$T@O^AVqIL|sT0s404"
    "Vd?Bj#{DqB&=uKB$w#VmR@wYXf0)z=4Maw?BrxP)E(`<BEZHctsUMt|~Y8y2GHh7DErLi&AZ|2R33K2n1_#(AqciL>nk%=J>"
    "3p$CkuxUO@GULMWwSm!Lhr8!ap8iskRMubXbI{MJa6DdwZ^QAgi>pMJoS@6?j%N8G&syEmz%E#GlQM~HG938YwYr_Ajlzhe3"
    "rFyopPdu4{y;-`msf2LL`ggxt}FZNUQl85@OvIFr{m^w+i6xb&>f#Pq!Ll}Vc3ZhML`69JY|MU|K&a`6!;me*LIsg;Qr5Gcn"
    "K_~ms?aN+LM|86AbR!H_rB6Ky~GLuPJo!Qgz>(*);Q+-C)+jm44&VEpLnTj7w{ICbC-rEpmm*QRIr9l~#=wN~d>kj;8k2X`P"
    "|aGXX_kGgnpKLYjVgvXV;7$3gQKxS)1>f>g#JHSW{W`ddfvT&L#M;;>U)U*r#XF67oLd-0{XFlwHrRaKVx?1{@Ss~GY{Q`i}"
    "v=C(lkd2WP36V$Z?3SL00ElIkil|@IAIp-2c?<1vN-L}jQoZ6K8O^NwbqwsN<dHLos_;GVNoeAU4;qQN|{;gX5$H6p$gIW2v"
    "G&%lY|5mO3?Qeer{zypkzx~mVi5dUV{}`^R`+qz=cFg1_wDgPl$GjN2t0#8g|NEbdWgnA2f$wI2!Z<jjMsoRcvx1BH>Bsi3r"
    "k@0H_Q$^b@q^@#Z<+`5@MoX>Ni6gC`A;{0LRa_y@#oL#b2W$$wy%HMj2nM~s~NlcU(G+grM_Y5{>eOaKCM3;#^xo7?SHk`!_"
    "c+O%fIH41pn;b4L%RYe?HTX{m-CQ5Zig+|Ko4!=jW-T#dN4L)5StiKi~e~fBydWzaP^ijm^(@^<uiNngogNUMl~(s{U&Ms{d"
    "Tl-;a5>FNy7+{?8`*;hbI)^HBYC`k#iI)AIk%;q2p|OQI*{KTc*+`L}<*3iUr;g?ae5fB(aL+MCt?_P@3H>!Z7FaQmzP{^#*"
    "ov5K9TUiHs0i2wZc?emiUe8xZi!wCGNefo#-SvFJsSLc7-zr6laf8U8e{?o00PV>|1zx|0)-B0W!`!65)9sJ+FkNq``fA#y1"
    "e~wzI{?~mUzmxutU-h3Jd;NcZMf&Ap#>t=0n?lstVz{2t;(!0QD}wyv6u1ZV=mtUjZ~x=7K&hWp{tN1<{=xA-(BJ;=&HtLf|"
    "God%{Okh5to(ES|HUx>{$Jkskkh|zK>z;P(f`j$s0(383!icS``<4{Y=_C^AJ+z?#zC5V=*#pz`|`6SdTdyK|M$OAQ?%93hW"
    "9>u&;9HFZ<GJG>y5^L5UFZ=p>8Dq=gr$^udAQmq}%a7`)|Pi_$T$MrvDeq`Ck`pFa1z(@VyWQu5DzhYwzO^8~9%Ra0vgm74n"
    "b&7~0dF|G4(Qt>)kV$KU=UG4*p*^>xqui(dZI%KU4i{PP(%&<~aG)Bpazzp3x_|M9!N!eYIm#m~x~FCT{TSK~`1|IGKF+|&y"
    "{jOkL@e>PAP`NtkZ|AWQ;t@0l_l%Lpst<i^bbP^_~)m{Pj9d^@}SG&6a!!tmv+D^fIQ{sR>R1;2aZ1i3!wJvNC(eB&3kT2MW"
    ">L0^;Q>=cA-`18p3k7Q(tNP4D<J5l6-VOgbhc=UkOK*BVfChOF<#`yX{o|E>nBS5Kt6nGeFz{JY>f}?p?s|k8S0^6xmY>sGH"
    "k(XJ4dkIw&(edemkAOYgHqu}ROuon7gkRWxl~Q?*uHGmZSL}8QT^x<ytB7?WP;;Q#*sa3p8p0_KeX<IyIx-7_gZam#n!Ias@"
    "!hJ1rPjaFEo72pQmJ9%6s?D#-Ag+d+j_YTVPapw$bbME=@YRHtO_<BX(w(BQ>{U_0ie-`Q4Gfj{G&(LCfA^u@BVh+eGtGDZ!"
    ";W_;aMtdM6Xs_l$|uH$={(M4e1tQk;=(Qdc(6lg`U+nd)rpQvNtx_D^TN%a{D;@k%bw%HMUKR~Gz%=(LZFjCUNRP^@d|YDub"
    "UadX2sR$a=PU#YFH<uTAa2jp2c=@ZxKH#r4VEA^o%4&`+<118{e!Chp!c7EU&oZGq{l66Px0j1nmzbJ83-A0=#nU3*G(G{4P"
    "@qV31I=N+GxzcS77TCHeu5>(k5Coxs(3$SmHUD~Vsr@bIy&1GPpL%iPB9m3WHFMkJ=e>Q4)zbEh%+tp(DhAQmMxx|i>b9T#r"
    "F!ply1RF_sHTz)lslsr^`x)wY-Q2ogMZ*rWzo}M{Z?K8yeUA-^<p&05<cnETq`|Ghl;$1&3nHdFLR^Q{g&pf+j-&QBb0i^TZ"
    "z6Q_E+#&v7!WOZ79=wGOtPeD6{iq&z^=<UVbPWyvS26=^@2@JUI!&93%Cpc3oN!cN-S}s@o@o%hG9+IcKr?tza-v0bIe@^7&"
    "lp!V5Ls3J};H=yatsA(!)?SG;K58zxxQh`8~DKY(a5EZ4lWvPr9l{_y>cX3YChm1r~#XADHA?wVO+WxWm17)RFgoC@Gaqaq>"
    "dQOGo4#$F?{TD#{z?rM?}pcPMTyVPMc;Zi&tpBJa+1cTCzVi!|A@qWZ_;T?28qdP;+IYC8YDMIx_X_MWX=fxC4rWgBlZ-b@d"
    "7U+P}aCO*tE$vn-DucIdaNKn+sj-31!Jtnv^YGb{Mv^nxK&7}l7gM6wqXdTo>ZjeW-`SMWX{T`u@wP@(v#qkdt=nsBa`8&#!"
    "iaeBm;5~~9a(BQ!<<m8U7Ao^S99hR<%8|*P;Na8EC`NIu~WEMyjxuIld-Q5@hEH`$r16bBG<E2cm_hlsR{%-i}e{PX*{Nax8"
    "$qFg6u}SVep_TQ@XhEMm;fsUnv$l)|9|nmkIF{4aS?hN|JnJXW-f9ztX7kgj=#QB#|~@xwF*ogY7-l<!aj25kIoWtbnYIja6"
    "YsYrN~w+La{4c4^bD3)!Pk=nA(!C=CjvVP7JkQVz9p_ZF|j`t?K_YSXcIYz6r(svrbnSN$=6VDg|0ydfB7cLxuka5gTvkypz"
    "|y?#4KEwuqOs<)$msor|sZ>r~JJfsVo*}P2CrRJ!)Ijv7?SDh5#;fP=5hPn)FgdbwBnMIU)Y15WG&M%z13*mlFm*G&VZFObg"
    "P^V)A`?uVHQHHI@5Ls4ueWCn(eO(Rpz9LcXq4Z0OTC~(y6Ev52-7j}#_dN@#=l*?%*>Kc-^a+Y>w5NgiIHV8G3*e+=uZB`9%"
    "nsjuIV|4;5#L>#;_PI;>&=llDIh1<fd~gO#GfA*(zt_w9of~MsCE(@p(q??qb>sRqJCRNm^qY<M_KJ=9j~Qq%cE^I1yAzwoB"
    "7{L-7uIFnPg!+<_67vP&Cr}nyY6kcR1lI{wAw3ba`(;0Hn(gtkkCjZ)``)T`=uZ;kFxWcGSw2F#|E_?ilDq9ab&P@^ZD6>=w"
    "%9)?WTr%q@%x_j&5Fh`;eieBhChXO+E^Sm6wwjLR$5{0d+9mRP#f;xvpGmbx%P?ftI4I@MN>+AsAoKYpzbO2WU;rEz=eMmpN"
    "mlG<yk%-_ByrpU66Im}NWJ#4d49EEeSQ+>Td`k5Z?{Hg=C1grG(wY)+V6f=)-;Z>WJ>S|_Iw=*CKPIL5rwFK@aw<;v~F5Acx"
    "%+oTt+is9XqhjJD0Or1b-0BUq`Z<)1b%@m76lX!f8(uyt0@k0)7RX<#=QH(Nj2c=iSX#gLXY3I5Sz}YZrKN4IWalEt_3>^`g"
    "{b+YTrMpv9m`5^5mO>-v|pcYwQB)sRhf?r2`<_M&|DXs_3+h#W`*rK?sQ+B%JUfDV0w0b<4iwo7PtAU(sa&npKefxkUPuQ_j"
    "8rctLr0NcXr3y>^gz2<T4wgLoMj_cpz>M8UD^J2lwJv2MI@KyxIfzd^SE>G<7)DH0pm6b)`$nGR^W@2$2wpiC_`M0L2v$6=W"
    "6pfubTRi>N4q%e9|*PJH8g0W(6=U7cB#mGv&S<~IxLa2-mSo=Z8fAC1A~F?eUVQ0lmT)v7oZ3a{kvM4b6GcG}rJUs4N8)B&%"
    "LGGHx9ix`p3)mIsL=uo|Pe|?F+O@mmeb(Y~Fbqe=g9$~Y2dt6!A-Ht<GlO>$qxLyxlWB&W#6Sm=Z9}lxiv#-VRLHDWru*&e;"
    "@%CIoA7~X`I!3QetfWX$cIsUe5?N7!YyG>~K&+_N@0&jM9T(f^o<~*H8>fmI(4x>@qGoYA0neOn6eZ%L()Vk9^pwqU5sxeS^"
    "tPotw&fr3$vs?6i8E9!&nLzGZBzO(+cH;8b^M2auoz3DmLso!r;Podz_&*B)t6Dx)}0s8DUH!iGaT6%Iwbj0wzZWKer^lvL3"
    "uF&OP5Qdxa9SY<Giq-MCp@bHxEaE(vVg@K)}GPRCw(kKmI~3Cq!3*xPc><VdG0MkM7!08S2CIcAbW;>0%3(Qt5rF_7S07Z5f"
    "IW8dkqG+c6|=e8$6z{N(fpQXbfaTRbh;@M5xEjkEFWI9K(aM0C|ZsjOZ`uM?|RP-L;g<7@joIpEHPPN2&K@@Aa`%wPs`dmDT"
    "bJ7JQf@X5YyfAK<!MYWF6)cI^$*&Rz#G0N}T2J&8Qy3VyJ)OO4<>4U~OY+nBs4DI9etet4ZXS^0SE$u+<@NQAW^~Qi60QSqc"
    "N;v>+Vs!qjK?ou5!3BSX@4aL^)BvDe-*n_rchnh93}sRPBDiqjP{FBr#`TCw?SS97?VqhLyK(jxAUl6c6R0@vO!%T*eiNP;F"
    "L`cSs(d<f?OF^m=k%GBR_<jb%b#_1H#Kmrwi@u)*x}E5rxtcODDICM&l6yFo~>DGnig&(qZM%aJa<m&pX+&Vw$}a>UO!0lm%"
    "}(o{ILq`ka<7Zdbfpu+*+)rAz9NZdare>i#<<S;n#FfRLbe%WUtEyqkKSi$6c8lm-$fQ0aJ#=;m6kQ)Ydtu^0YOE*vsJ}?v{"
    "TJU7@bFdh(-;Tt)+R+w}J2`3F|^Kvp}q50mH0mD)72Y1mYKJjvJJ$`RJq>{)XfSRAt^HeG8Gi4*!s6L53li8TdJ2=K@+M_eq"
    "az9b1<(}+fwyEA=>+TA}?HXB(v)0**dAFW_cUg%Z|EiGOmd$<bR+sN~r{K2E1-A`^1n~7bJBve$+iXW{yJ@m!-o%OX5-E8jq"
    "<XT@HE{dF}esz8J<>h?nR2y16Ac4@o=pO-TEs3H>3icvXny=dWmkn?Q7$IHvCz-$hCaJU1B``v2D6!(aUZE3&%%b{uJ6_g8e"
    "2BEcU0D%7qySQJdR}!NF1&GPlY!pEDd(eGS{=m*<LnxE;NbNOMxb+H7g$rqbLCrVy|FpdaQ_<eEoZhul#x+DC@?u1DDsx(Ve"
    "_$v%XMY)_&wA9xZ%xGHR;CPO{3nl`&cHQmYv2ka-O5txA(Gsv)ui9J+_{uSr2_jmiq3=ugi1Le_f5Hr$T&sm`s<&g!S=GJv8"
    "@{=J)VI5`RH$B-mlP_Ql<}Tzia9H6S*{dU>_God^}$jr<B17mM=xZ%(w2YZ6z_Pm1hhF@_!{Fg!V(s>x_ml%rp&*<hoTPz8A"
    "8VW(zN=b&=2E(zj@V3zPiCd~SgZmX>&;<3i&d^;DDS9s==yp#C<O-w{heddQr7pRZAo_$oe0Li_58lRo9d2@z#{5<45#~YZt"
    "C?42La&?{4?PzJ=jloXgRw$m-MR4Y(U4t0E#Dl<`p0i*M?@3SON@2OX;<-OKITe5zyP)>KvZ^*3OJTzeBxZ%PcUvW9=lfv#&"
    "o4dc<Xneva5#p3*I8K321r2<q2IrFhMrl;y2tAbqc1y?8Z!MI*-j2+QTkzqC%4<Zeuc0YcyV-hD;4#V*Z<n=mZ;yOSzG=RR>"
    "+a$=Tz!#uiNyY&`;aQ_cA9~mz?ZHS;)9UcX2tD)JFrtE27E{x;IZDF~1=-mos4eU_Bol=jW-qDYy1XXSTj~4kGYg@_*%kr+n"
    "!4kKC?VopC-hxU@1yL66>;2Buu*$XN~7L_&ghZvb8)lXD-F1=7EO;AGXckc(=rywbOdv6kvI<1=Z^oA7s+eFh`bmkM>tjE}|"
    "bi4{ktHaXeBcE}v%UEf%O<mV<}z0+De5)JmnY1zsxaZOVe!d%#-K>u)LgwX!n??1ZVB3Ep^^5=iM9NX#T@G3|3u(cn~m)ou~"
    "Odq`pTN3Gd;zf*9Pi{7*yv*qL!|tEg^Sd=4E3@v7-uT}tH<`3X>a^Y^SI@z!Isy7&b`_*?T@glsQs*TxTd<2TE)#PaDE8=!e"
    "{IgpeF{FUMx=<bZT?h6ZEeIfP~8$!R5h4QPr;N80eua(ZZmv9?Vcg7H}js8X0LezS+aQ_%y$;mspY*^6|B&+GE7_I+i4Sm$3"
    "3|G7{h5@=baVK9KQ^L%D})cav3A~YyT2wht0_3)w6muvMHX=SEhgkz2;F1A+rJRZzC4%YPQ?zb~jbn_CC<1TIl(CQ6N9tY<#"
    "$l-M2Aa44$o<CWqgs@p$e#LFQ3V&0)x<F5TNn<%Zd)zgO?6ev!A`$9{&t9zPB6=W`gygADm$)`OvwZRR3#E}RK>npGQ>8ui)"
    "lu15L%C|&T*ca)N;bt<=CLW;=8o&~vb>;(Y<lsKT;<z_wVvpTka@N0;dMKI{q$z_T{*p)42dD`9`#|*8d=jN-yANcF2v>n1N"
    "KjfPE*X8Efj(w6u6>MJ*s%}-d%I%HQD3|TKu8x{Yb=92P;ch%^(u`&`39DtCAK2aq@IVv1{xKTYi(vA)+;%Of8_s&t8)aU^J"
    "stPQw`NuA2$wGSXLP{S$>x~8Ll$Svyw5q>*qD@oL-@5Q0a&&1Elw8vW&o~?e}pe^q@&X+XQRPwyuOxp0q1DtJ_yvU^(`NO+X"
    "G2g#yhIoAqT-l;Ev@E(FmaVxsKYYahUIfeRnIThL@#YkG0T}E``UCsOo>kdTGELy;6Lx`2$T1g_FHG*W*-H*Z!)}F#QpZGEh"
    "P1aj3a|_r71J<&KzxG$><zf;kQRD>OR;E8&AA?n|Ec{YrJBFop1m91$P@@dD3h=f9s4VHv$?b3*AH);Otuph3MzlO1=XJ%^J"
    "%`~sD(<H2cCDU;XdI_WjS{m+9Nuc-2I+L@}m6MgSjmXnUs757)p8xa=12dN!b^>+_Tk#yNPd>tj=A#dyrdxHSNVlv#1uWoJ2"
    "j^B4BtVx@~D=Li^*tF)rV-2K82}3g#ygu&=n-{~&s9En+P0N~@ulhQ{M~gMAJEaC(yy`U!3(Aj%lj?sG@S)}P_NDe^8+SpkW"
    "*V?JN@g%3G!B+goj)?cKfl&L>*J3og4V*{+@s@7A#X?PraZ&B;+*m*o23vlGP8^7bMX0fIPK2O3CEl_U-gD(9&_M0`lM1XDy"
    "qYW(G2g9c(SG)i_j8ZNWtU&wuxHf7Bt0zDt#QPq`uIrelAI8I=)?oGRWAIkd>%0_ng+h9`jw43e!;0+iSL+jVb4#qt;U)TJD"
    "6$E?cS~9ASg@4(j{Vkp>$LfOiel$9-g{+WnmCUTd?v$5*!_w`L9p_<Z(x)$k`hmRf+pQpH28vVV#lZgCpbZ~pb7Tg<u|4|Jt"
    "}E+lMv)L_B=;UV$*vyx(io&bAzHC-~GIs5SBYWe$ctFT?~gp?g}<B8emYzd$>`WVwfD>2pGaP;j|@37Otm^M1H2?;sH2XlRO"
    "A9nXx=T^d<p{5%#36#6_7B!x7qxxkTHtipJJ2B1RI>9%Uqg`fRT9x*D{~4V%$#I>1l5>NvWm~knH5xZ-Ig876@cwM7n{<h%i"
    "~~&iv*SroP7ONuTU(&|MpITCwHWiJBh;|O(~I3?95yOyM@%R6k?|;=WBe61GK~L{?D#zV0Uo2vk9uG3Kt9asHr+V3={M83nw"
    "h^9Rn+)_9CdxK`prbkf!qB@PUiQ;XU3i6xDEMIX8gxF%ig0s3MY^W9M(}s8$}(s7HX%>F7-`VM%v4VK9$;AzM2bZJf=GZlzw"
    "gxKLV0Bl2Rl1yiY$!egZxLzZdK6!@5NLBAm_s#sf=4-noOX;r5oTnOx&{KbB||slD7i9i*cXR=G+;Vj!k?3vLk~nlD`>4B7&"
    "nlV#U0#U6x@OO-pXXl9xB?r33Q$g7zKu!D@`al4p+`XU5~OGdmXDfpu?z3IIoH0-iFTv8r>;6`w3g=n(4FP)4nOR-twQi6`W"
    "b6vj*F<&ZI3?#GNt?dFI$83Kdq4VOBpdKtc>Ed;tD=>Fnp0?S&&McfKHM&9@<L?Fz7SKv*x9w%qw#sets%O54iGeX}i(vUcw"
    "(&us*SLEd3ha5$*A%W|tybpR_AHXru^-%_CZq!kNHnlXm4G&E^i9Nn=3cf;ZqL)`9oc!Tp*?wp_ZA99L2OfY9?~SAD6HFPDm"
    "|NtCO0eAQ<=RPAf8vUI?xiPfd3QpOzEq^yRVw#Z5EdMa%8!Z^@gXs5&1XZ{qh24*s8KMF@X1cxzh;&W(+^XZcWVz`k`H1Tk0"
    ">BX+;U-&Hj!QMw9P1a;^UY2mU&PrnANNET$#t@#nwUkB9xh$VI9ow(!ZoqMTUwPr|PA5sdjerPKYpKOzHhC3cD^)_bRrG8!k"
    "ce!m37vhr<hsoYLyGi?`u&58paDcc~l8QxRsN%lc~GpKg6W6>Gkac*nM-ulzS7{!Oi&%<ufgxV$R$>=lTJlTdd@m9w|{jpkK"
    "QI3s=L%8h(gK#w&N`Hcg8oQs?t@-2Vm*B2T>nF)?oqY89WM*1)JJh*`KgnjNfIb10y7AzbWr6L}YXA=_Qy$g#<Ycwx(#ke|v"
    "&)d`%9^it<aWhv+4HqcJ>UCg@Mm~=+++=nN-X(N&Bex@&Kp&VLWOO;5z^_WGq{+O)*iIw39xnd5AX^in^;pJ`!JlW;GXo4A<"
    "9r*xV>M@bRRziaqZWcki58Q4FgX(w8Wsl>BH*X<rzvE^C)x-6hTr9pZ;a?WG~w(CvC7giR026%b%U~?+RDWf{AS3WuE@1>b7"
    "+)euRUzb9ZFz)T?_Q+Sjf>aZp~5sMEbZIN`x?F#R$9RBIpFRtMYH%zHYo$7*DkuGjuC<6s%Dm*AOu{*9?YFmI-BUxX7cZgTB"
    "AIQWJlvtrw~?Y5Y)OPxLd%RYEDw)(<*%!Hi|*9XdKj<D-f^B}57{qY)e^y%|_;&{$%xA%!yzs<+_zS7DTk86d4Cnm8xZZ=mC"
    "6iY%{{{)i`R{1-$!_C}%_kQj)3rLFRf$A6Ht%H`uu<-t@Qjeh-G(@=IsIod;0`qEK9Q5vhe2m_U9;v%WAKLZZ<&+U>V|7^(M"
    "&-3~gS*3JiKsY?yZq%E)}nO1=Tp@7!>v;#4#aj^SN^WKoqJ+#pzpLtk7H}-&1y>1{~1T$$lR!vj%h8mSh;3vQUsgr3H@yP`E"
    "0pZ{vrv-T?W%D#2}2NSJk4@<G3)buBw4f`zOL~6w(;C#3kShhe}Ii4`KVc2hA~un)^1mi@Nhv&;IvQ#C7-@JW9{WFIRcr8c4"
    "4=kMBc`2dzHTA-M2~)4wdjl0G+}qtkg$%jL$d)k2=Xx51Xjzo|d9(v%z#t3_#+T-QO_@@Kng2|kC7IvRHaffK0uVQ@oH7!4j"
    "BHhjb#i>~zUzY$hDw7^HdYS`+!(jNlQFZa4>*03MxQ1k3phm})3Xujw6-(;l0|Es&Pu8bX`_k5aQ8$EBm+hVKwI*P0j%(8SF"
    "QIhMLUA~baf#q+!==wPJ#K0~;cA=p+r0C~UWpQr(<;ww41*oca(yqoO7^a_9Z%>r1O5?n1++W)`dS2gR!*}@X5LTe@UKun_W"
    "{&cLxS!KxSUqI7#dvev=;H$sZ{AVM9@WxA|Dtq{P0N2!27A8zwSOzXD=rSHnXaEW&p>B;liG=gR8`=Hc)Q+gBIw!@lU~n=Y("
    "QMjDE8yo>}2=H59f4eFf~J>_rgZX-fbeD<;k}!Rw8}GPaPj>1*^t<e(8YqJ)|_MjjpM7nvRsrxnb*ZJqPD-BoO$%{|g|P$wq"
    "CrRTd2z&l3T#b*xa6Bcg5_x0?;67w%`q#XYxquKeP~MdvSuX^C_Bw84uwGL>urg<b~@ql><;4qKemN*`c0Thixw3eHO<y26R"
    "V-E+jh=Nu|sTT58%s71i2v7ljZQ(Zl#9CtcA9q1HsHrJJEPoNpE`pLuw?|MxNqq+;-dHtigrPJNxGJ4CMmU(r;9jUBATEHGe"
    "gXmd$^rdXFx=jCcZfatd7W?y+?c|V-G~8w5c+8b-P$DGiUNiM8rYbyGKVV*YR^dVik%r5`qH?K>OXKaXGuY?%ILqNXbF?CJq"
    "#F0dJ;KjeFzRiut;3AiF8{2%Ry#cl+{OLc$Lo2U%KN=hr!XWk^V%=$Bc;p7#$Uefp?RrcOb6`+uidllWUrndjKt9cZfn#fgy"
    "cRmVEUcLGhlDbYH(XLr!85jW!&}A`TV5}Q6c;}oGWWc(wfnUKDEyEM9Wb4-P?>5wu7*xL-PvnD%2hPEb`5Hv%8m%2Hgdh$oo"
    "pHcm*W+LDAPMbL>X4Pk@eY9qgu^uU!T4cy51KH2)Hz-o=OTQ&2)5mQCd(QU&Gws&4B_{gIHjTI#0x#SZ+#v+7<e6K__u?o1E"
    "wMpXREW7fOv5tL8u4L?6VmV?c$v_TGC_W}a_uj~@cpZ$}v*f%z5Tk2#s!vfu)B+i0RmZN6i^arS*;gM7G#y~t)9sseBDi9V)"
    "pCiyQW*Fo|_d^ow=)rh*C$h?bnOS<PBM3OuWsmu$ndjxNAL+RHI*wdG#|G;kTN>VRU|7DC{a+SHgQ>b8nNJ%gd&Auxe!!^@4"
    "*(9C!i^-NPUJ&32k}wlUPH+6`$Yd;e^sq@@8`R#@s9I`qo`FXSGH9b+o_f1DtclTw|=S+MvK+$={g%*@$84{3r#YwlhhRZmZ"
    "R9k-Z3sa_*w5hEmgC0u=FAUa=vwUj~5ozlTCm4Jio%{vmo5OCA|~Nf@kayUk_)L>AC>-aOVxZ@-D}Ocb-e#od~uIl5jitcT%"
    "q8X}8H{ylXX9kRv@y23A#5aTb-kJ!|jzOBaFb-()*4G%q!|(<PZX?Bu6+!{1JnxjB1sE3WU+L30|E><?j)<s&d4g3)!}68gW"
    "fMLx};X|OnzR=i5SmyIm%ml6efQk6qe^U&(q>fo>1ItGTm`)A$qet-_&IZr)qTIyE?xv(5$!b^6<F_=<wTE!UIyR-4(hAVQ5"
    "jdwJJPt0!~jCb?xE^l1wm0p>RYHDLPsSZ1jA;{M;kBUeQT`Z3G{$n?=`@2s2TZfLK+1QJ2lK5%*Gu3gyzFldq4khUMsIRIAc"
    "(Sb%I~dlt5X(!gb~iZ&<jScI3C@h;W$D$t*m2|2c|V)UZ}{v_Lg&~Gu(-R+uQhUZ5JcnlH}1o~3S6_XeJo*Znh_g1ur%*@Es"
    "a2e{?WlYG0aPB<P0tnGYXq1)_n;?ec&EFvXP_B)rB7s{zagt@0{h>skAmYty!sjzP;DG2K!4Nr^X`VZ$Wn}nWaW#zkw3f%1w"
    "IXyMDzc0Sufq&SIn)R%@m?hsy`o)3<%D34-3pRnoRx?&u|k9Go-8BoTLxf3Cq1u8ZrQVj6!lqO}!TUd6gSJRO5U8d6^AS~^t"
    "bZI`z?&TPjaAKr37)s^2)Ut8{2_}0>TnY5syxIYK6?`fG$+~050Cbo=v>(935SDeoa@yhqert9f)M>^R9cj!}r%mmGJ!Haca"
    "$>`gXGp86QvChS{{d}Fpt>!I(307I{Z|Zd6w;X>=Ub=O<eD_{|WCR7b`dTQG-2iD&1Ok*76>sf+*2Dcs|LAha0|OD>!nv(Pk"
    "^=d`2f;U_R#zgw%hn<)^cNfQQbJh-lYTyO{MEMjhGuyvR=`~;<3Fh0>6V&hzuFU-mph)8a5^cULDrt&*>qy7=IAmnCR(ixgz"
    "=r#rh|2RcdAS3qci_Z-Nm}_8jmF2IV8DVCSLEkD$x1z&vYCE?o0RvKbPW09>XUgc<n38QN=eG<Mx!_Crf2R-qYz+X5O1hrFW"
    "E%z3ghAO_u+?Xr3qCZIuhV`tItlbT@Qb^5w`-(D?EjP`S={z7}$P{5AK&-^H%Fg3#Npo=Ch(YbZhTs(oaq{TQ)IL0rDxZw>j"
    "U%(wRJ54ofcVMN^Wz2e?1?bf~et=_%{UzTx?2fv~cnjgt<!iXxYwG+j`TJ{r3+mn5ZVrXCAx>68IKMMxp2U^M978tv^+#jSq"
    "(RsuUnFqU+T(cpZStaZ#;4b0LOzG0v4TSB_aYr{h!S5>nN#s^8H;PJ&(310+-DfehIlCDONZXSwN19CZ&am<8z$>U7pc>IB$"
    "@P^(o1K%-H0cGkGQ+3zT!2wqN%_6y3^H#SjHy7nGK>BGGvf7oYw%Z`I{chWGe^d{*bN)+#f0Dgg5$Ua?{2f(l<jlMW5C8YDY"
    "`77(SZKv6|Nu^(|Aq8Ufa_~hjCZ<LjD8rase~zC`HqMqv@=@7A^sNmTD{ap$}DHEAM5HIcbgYA*u#!qdncF!vX9TpxerP5Th"
    "RfUMl-(yHmBw%RK5mIHO2^Jbz@CXP|nONc2w>!f@KSO?msV_!;>OeqKXY&2JKW+!W8i?8srSZ|natrLKAR(<{jh8Vidd!EX8"
    "1?cz1Ug6ZGA_LbF^co7Y6&sfHP|HV)<YrKCZZokKWehF*7`W~h?!o%d25Pcwp_;_$qJt8*@pEvI+Y?;O6e?L{FU&=yx(NT2U"
    "xL#SKI`c`lM2yRMdE%ga^EUppT#xB4g+8uLQFJY#$9%-?26TCaird{MAO_{lw5M=4c;)f>R(!4o`3+nPaoRrM2qX^vxax2L0"
    "!xo7y!v+10s!i;I*$EzbwqI18|j?1=x4GQlBWR*)ozslg?Ibj9*m+j)EQP~qPB1MC<DTEoOo6h#x3gS0eoAKH`?b-HJB_{UW"
    "oh+u(qQixQ~j-xvh{_3m)cCD#Jk1<i^X5)3M^#F(r;G?2c~y^pzOrO>*kB?R;J0tlsDYH9muz0@1x4!kN?iuI=j$n-ncmOka"
    "cBFV%eYTD`xwI%?tKBF((p2#Qu8ygIWS(owhJ^E6$)c9g&k$3A=4XK**ZsH)MBrueWs8~M=h3;ssj>#M>Zysn^jhcnSITkbN"
    "KvH<;>UCehVRWrAFW4cutcz2NY_l3#*Zuwg578R>NA_qB?p<?9XvlUaF{-Q&Nys-XV#8b^R>S$eElaoiSGQIly+l%T3ewLrX"
    "3Y?(52mc|O;rh>KI<B0D@4Ub4%k<;cG_`HDbPA=EdRAWkh)BlQ@rdM9;yICu6s);DDW0eGgs}Qi`4ZzN`wRrN@qB)S+~Z>cH"
    "s`0%9vD#{=MGP<wanj_3i^jV-h&plJL&r|9sb@QSNCQ2>#a`@d6c2D6QZ25S<zd9=x#OvP|LmD0VoK3IcbH-5f;)(Wq8}Bt4"
    "A?i{xHmQ|4Sa^stk^$(b2TJ@(ukSBp^_G+68z>*N(M1^$_FmO>!Np!m4FC<Kw;QRE_d6vm7|=2$S8QJ+HN#(fLvBWrJW-@#)"
    "_AIc#;SUZojD#G#E;!WGnAlz3$(E*B26v@>9>9wGC*`<V^y+gNTT$*5cXy9u!eaHt5|vhaQuS+2$xm{LFrM&w7lmxBuEm1aX"
    "EC%U5t4`loROe*gk^T<o%6?tXu0<UbaK{Phv@e>6gTO`X7YsDULou4PC$N;-I6nrN_*@<$r<aRs(?-6sc>^N}4uAx8E7wLVr"
    "zd>?c+eIRebZ#^9{X*%6L~3Pnt5;<MZ*MI1V9$KyK$yk$bpG^4fQd&3asM{$W^v#P0PC6P|9;8^-21ieMiSXQbhl@<RLs%wP"
    "_k(rzedI#q<6CSoTbO@wOO<eK^Z4-aI-W^?Up}V4My0q+3k+$i-9IrUY8DySFJnpxy$OguB2hFMP#{l`<C1|cTg0kcEx(I?6"
    "q1~!_V@u3u_!W$k6s^{Jvg#VPDpIU0(yUGYa<>G2lvDe0Z+OP}v*^C#JM`VzY#DeT_?;aj-O3Tfom|-x9scTrR$}s`mT3df4"
    "|6`@X#=`8rD{R2CQa$tgiU>*wIWDC*f3p4+ym+Vr5`gjZ_g_YfSDXd$0Qtl{*;7QH_XfRomqC<A<z@3cN&r|<LWAFim+lVjM"
    "mi}H>d#m#u#Mn)Eb52RB=MMM%kP3oQP0L)x%4Hoy(P!R9>wXDjc(RchDwKBW~Tp)eiK(F24vDo3vY_RO%ogiFZ-@B#{+H<n!"
    "TzjSXUg4kdIG_GBr=7)-2Ke7KJAJfnb-ie?=e%$4yFVmd|GJIfeU@~TM_6%`bpG6f54xgm)<&?m8s>r6^)E-sW-ndHn%hrtF"
    "?oM=x6>7WFZV?f0phEDe3s^Tg3l42v)5s4wZu+t!?iJAHmuzcHSXQ!ym!$nP-U^|XXO(PmoK-hje*;J-*eWLYw0pzylDnsB!"
    "7Mac6S@?X!hUaQ;=>oMKG@n-s(#af>-VfRQ+6veyopvn~v&jo?d@ybYeGbr37My+qf>jLE&{cpd!#7g<+TEMnc&|hGK~0*}q"
    "y5x-1Ksy$oDRuB^c<I;<47*wTkf&&a^k0LDK%!O};VkK|YEv`Y82&d*+7Lrk;y3Qb*QT>9+Zrnk^P(S}G$Q_X7D9zQ^hjf-T"
    "NhU(U8gzVgXD60?jwpaT~^)gAS{{8MG-@-a!sKe7LfB>`)-8R)2+Q`Co<jU@ipR_dWW(jr0Y;$+4H=1|%ew(z;C&3X)fZY_e"
    "E%W*fW=LZ&=#(Dg^{w}O4KhP5JD}Y;sB>1p`bNzE8QBRH28R8yKmJMc{*^DQeLOt=9OHgTjKcVQIf<Dw9HLhgoBE$CT4D``u"
    "LON~ldtx18X;KewesE&V;~m`MTVVtK6I=q(QM5)jX5a0#QS15`BB_IN^M3(gb4iHhvy2*w3Lkc2vJB~vTDiw^x4|aZX^ajwt"
    "KHt%ojky+XKVgiu5d32G9k6`nxUv`FhVSf=Q|M*pUZ)epE=kZz==+_k?R$sTiE8Bo@ytG&3X$C}>#FmuClQg^Kg+P8D>#)H)"
    "}u*2;}4Z)*lTTGVEfX;Tq$^3;3<uf9H+dBeE+XDxzw!qe$zuL~rd<kjmqJI3&1Yn2tyn10M*N&8&)nAr8u^d3IW6fLOr%d17"
    "VOUtXZ$xm7He1u@K_xQO4mm7P&8)48N0(0QKfCvZ(3|lV#V#HoIs`hXgZx=7)e%A!;1gvI;7Fqe3epA=E$U;Z({RVJ4RJcnL"
    "RUr+>231h6W#eHyyv1bWzS{nB@9Va0&pS!~hJ=reyx2<(fpkis8MXeZzhM#ZfyzSa>oblt>=U$=D*(H6)R_$xO#Q{DFLK?i{"
    "D=%U?=IANGp_0vu7n3B$~_Yw26OK``wrRbC#Ayz)EW1K4z|^^o$k(&-<t2})i&)l&HXwqqaT7=pC96NL{&cLcYa&~9$MjL7U"
    "9mGJDnpJp8j-CHH3+I2l8L=g(mzK2JKJzLqIb7&#0a%9)P>JDGQgy?p$NLn%y3{)~NuMVU5)HbFL6pVQ(osld(?kW-*y?EHK"
    ">L@ciSg*N<uh;#B4*&@_8z3$6(tux7bOfI7RuG3JtV2d4T*Z_d>R+FP`#Dlu#HP`P+@&Wj#X-%aK6z3ZKmlK(L#<yX0_%s{+"
    "aCl9<+_&_rbYTLxDirt^Vtr<Ku$fREOj$dccB*Or^PN9**^Qj3nC6V~6q+)?i%qZ+md2mhH3TNML+OJ`p%#SX|jGEBn*w!wa"
    "z2{KsY>|dl>oT9r<%IU_&7mw=jE9#JqSZ&$u2k-1#1c|VL;lCS;*fRR=Gz%Y{-wm7qBmiWba>w|UM0O9s|#1^uDu4ZX@a~W_"
    "7I3km-Onx4PhNl{nmvzMwjg4;Evou1TvZr>c3Fz0H2h@sn$#AirjwJ7GHtiMlX$~8%1>|qNQvpv*+Zu-N`G3llLm>>&V>hqs"
    "0DsL|lx-T(<sb-{U0}tr@BN%PAJBM!CJ+%)$hdf?6n1|9#4Bp`;7F@G9ns*KFePmEFBo0rP;9OTS{=hrWjX9v6x+VLVrt22o"
    "e5;{NNZP}WNMl-F;&4mMI=(8)=co4oc41FTF<*zDLvPZTq_aOBy|d5F8V@FueCQI&H$Bp)MAa(0Jl<8oLLTBKXjtLlLF=Gh+"
    "*@5JBxz&hK*i$9!O7aDMTlIpK@m&X*?EZazD;Bmk~cRou!zdb4$ysG^!Tc@)?eW=r;hw1no{1Qybei*e)?vEL&QG@Jgckz+l"
    "Bz*?he6k3)L1Po$qqa;@fAkVvpqj;KH>NVY>++p#?y^B;lVUGvU{W<FEAPP&sAI>`fMEx}fRs74JAv^)KCS1?L_26%h~v`t_"
    "Nm?3zB7FjwVtjG)>Jodu@BRCp#DL)U!RpFGn-t1{p&f>)fSD+1Nm_IDx|Ldi=@1mXwtnnYm&kV<@YY(i?&Sl9l<H%n)lh?X6"
    "y7dMD!L!yq2p@1r#LJP?^UuS+|hQoU})yt7DiWpu9Zq>#frptwODL^if|f)3^T$BKA#sZL7&JY37wfdsvsyb|)<LK0na3iFS"
    "{_i?FB7sWzMN$n57<OJc`GZ1IRq=?)+dUf(n@(}%&zc0!C_dx-|12R{|oDcTjTVnI`_SEg}vS)n**lMdgDaD{iyHR{xtkJD}"
    "ShTLC=-VPvFQ@7ew*=L!tN~v0|OojHRCHW@r9Z&4}uQn{i(e0gTE7etspZS*XBoap!nmp*7#?^b%Wib8_>|u~wOm+nSDSc0q"
    "J_XGO564mbZmRsUsm1uUx8R~$x!-(|+yV((ueQ`6Ul=-_uf&TKEzVKgPVlJPh}+dJ1Iv&72Y7QHevlu$RN9YO_NbxU2&y$N)"
    "lOwm`ZFjZCWl#!BJAM>4Dw{o!LEIyi?n@2uccuz6^6nRtOa;u3N}EU1~!LWXc|GRs7qxz{#YODVI5~?E1@Qf`Y==Tk#v`b!2"
    "%;_`zW&&_`Oj63LRO!k7IVVJ7~)+d_mOXVgS=dUCx(S*mV5Iu$aMn9)AuZ9PbY|r^~F}X;&w+>j?`0wN_#zDzPicJT>**PB`"
    "z)xy<V5I6k}wqxp2ck$2a>VS83Y*0$4$RZ*4Os5IpnupU+jQXvs~xLm@;@s_63koH|{=W0FRx`YR}S8v$KLj%8^O|MgzLvEN"
    "`acTpz9iUU6QlX`jwh^ih5L7y?DF16+AO_kk4y(U}$W6+fYvLDoNu2wv*RT;a{za}XL7t&B?OrzTN8ki4v{ub}=htpa?o46r"
    "vj9!(*%_y<t)1FC*#g`^N}JNN?O(f+e3`j7{m%&Od5zoOme`qDI~ko|hwQM+e6E}9^5_6Psl;9lp|h{30C3N&`Z4Kj(uCR!j"
    "N<^O%vn;6V#t|SnJZGJJo-g~E!XbjyRmBdbgjLgt}s#-YVZ>?d;LGA`4;HadSC&1qjM|R24d7LZMJS{b32Ye#+b8iawsR;`M"
    "4f;fOS=iwhIgAo|v=-wT=80cy{-c_j6M9P=}G#xp)|hJQ0rfg8%*A<gk<2t^N;Co}};x-<0vGjW*ZONa5@GfN#A;$f|gL?=3"
    "UMU_34vv!6Ka`ra>F^A}Hg!S950+(7=Gc%}EJpTi1SlmV~$hS}#~2yJ!)&HHDrUTf6Ae9OU@cn+)!FW&mQ`TRC=Q2lnWp*go"
    "ZW83mlny=)07wU@95&`HZ5-Cy>Jq)KK{c*n<QtdC3a!JZ_zHe+l$6C}{pi=Q|L3+(*(aAC&+<tv$`B*L~7<zcjyCyQmGryCc"
    "4Hu!Nr`;-4u48b^Oa>o_dA7<m|8jq+INu+2hVqD|dq?H{IrT9>x&!H_Rzn}EeQ7ijUladymizo(UI!P!rlt$=GWxjbBsZQ^_"
    "oq}BO8kQI6RdLe+BKie(j7%^^U9eV&BEa#uF}|deDEMCCMX3FUcqEFGH_-cXi)D<CFPFS%z$Xvsv&9kyJ?E6pj2K&=k&<K?F"
    "To1Ve`{iJ&Z!N9&j+AH0#y6e(sFpK}J<K9nW@7;O!1jq4o#oSj-#U$FOyHa<4%-rMv2QMYYEux7{{Ncg0$nwqT%0a0mHyyVU"
    "0O`MexNZG`5b#X=|mjr40oI^9oWsM4x%3%OQqwx+$~QSsVnu=gi}#a|S@tI<y{bS<XTYdX=|O{u0p$0ei-5UY(2qxzhRixG6"
    "K=1s<2HX&${x{}qpq>pGL;6chDexboPGwZDo^B^;;_nc8}7Kmqy`_CeBpR3lNXAb9#%YMam-%?WA4iLCK>cYCRmnYB|I`0O|"
    "7{&S3`06ptFLf)Of~GZFJIp9%oBaXRV-mCc6=q2_zdIQzvv<~8K+m>w9@+UTI17v3GT0-{EFlnMEcta#oL`7huf43}yGJ6y;"
    "BW-a6~7ZjH{r!*#$ls8(NO)C^75l9H4Qb^n}rq+#%w;7y#`9q<-PR6d_ftnMu}(X_L1(3Eu6?6myt;1gOQQ5mo!Adit@NE>X"
    "!Zsi@M!o`C>L1ENe|WXZ|=TvgXef_*!RQqm!Zy;Q0k;)bL%2mtW_pQAxvB;aOKfjk6IZwyll%z{U)lu%KsOe+Z;5KAy$-LYQ"
    "cC#7kiFEv+e=oBkZ6gZ>myF?<hyvHrVDAF)jSbDC`e@Wz>BXc=my<nr7V+O}gcWK6Mjd|A}{W5_;t#Gh>irJLJzs@X9$a?ZO"
    "}1=Jd4^|l>?#<jO~ZhOjpSHf?_VBcz=2knu+F75LYbG^`)Jr-bQgSJN9_fF2QA3D6+kwLtA3+X9Wl;md=H!&013Ebt{Ar}L-"
    "=BcWQHxvDcyn6lB>=iVzxL)qG{__KX(QO@WsBFdTU{bxR(MfArZh>PSYhDk&im$yc>Oc4$DW}}>lHQ6S-cv8`m8ENNLG^A8|"
    "F%e;ZOB(INwWo6dwryJuW$n^gO|p@D2T>iX_33MM7h2mo5(TD6f@g)>fmfUeqT!wgs+_Ewbwuv_xdniDPm#~?Y$BwVC$(PtM"
    "JU@jI^x{@{3#>w43Rz_Z_cgs%(07LY*xVF1_`=0|qQisg*n03ZT2Q{zl6VCz_bu7U!%}UC|;<kM0MjYGEYei_X%NNED$eqiY"
    "rW)>});q<OWWo}JXSr8<YIT;F&fy(ifitjkX$$fU9uDERMg6<^QA`Cggr^;uu3?Fn3!fRx@C%k$zu7dqMQUqvk($voLZjZrJ"
    "?A5i$TN_T$$4=aEQv_>iN=nZ6IzDTEn*qPF-_Ilt$i!=a)Z&%WO{oxonH}&-M&mBiUHGLC74`PJprwTQ!j>i4T$XRzvv-y5I"
    "Zos#lzI`w?SN|r=CKoWD;oWMk%RZkeKf`2kq&F*TS6$MIbJVqiRy}mA&}>wVnrybMT&xk`PkU#pZX4ev(}VpP9`<&BrOD3mR"
    "Q~2NhyCqZvK<2MwG*D8Czq+Gjb=DBRlTX$mhMPeecq+AfGqMe>R+pkT046zlKG$wp0*8(jGagSK|J3Px=ZI{e)f$NV>3Ivt>"
    "i>a_7DG^%8MwF_agHV+zV<**aiXT6SxAaM_#eS3vi|OmD+q1HC%MCA=y4OyuA*c2AnQ7FSty)0{iSWADaen9k*lPlp_>P{MA"
    "(+=ZOnUh0Ja}xt7q)51i}#UQTcPaj*5TZAoi4G}jN#^d2>qT6w_^TVDLr^h&{I#?Z>_PE;%<*$O{XAR*2rcF_vX@$?fhef7A"
    "ne@~jN)PX%iW8=#PRay12I|jgDpbc0{jBll=jJ>-*@mPMnvzF$mrL{o8-nIIvR(aA%reIux<-I1=zxSnf*`MVte+JF>ZJCUK"
    "d%J4B9nfHoc7!X)N?1-#R55;!gn(WAlA!+nIum-+>_HP%63sWzYq1+AxY1oQ(e`m;#~pK8pT#g#!NFH$(M^!_Z^(!=l-T9Ib"
    "Yks6)qwCJxIQC^TTq?*8IFgC@MLi_M!m7LF=5c3`>bexYOU=cZbjI=JtNyTTPCMn`kp~>;ST>B#ybchz9bu2>#$<4pTr5~$z"
    "e02DBm=1Ci#vG8^M{~V5~3eGuaEy$A!*?=U7H2%6<EM@tSvTY$;Vc9#(+v6%Te=aAT|m%+c1WvcbEJ-|z7!Up?~&<oFIu<I_"
    "9sxyj{Zquk<auz5k9_yTJ^c=g6zQd?Ceh-l%fs)oN3ri-Ltxq4(Ymzzu7CVsA3aacFp+Q&XFsQYG1tcm5YWGM7159cM`e9|~"
    "G6dLoq#F_Tjb%$cfl~yVJNQ7IDo^&zYCbbl~NwY3d96J_NUo7xu%vW}v432`1)Z09McN1CZ&U>XtQ<(OIh4I8$)orop!tyu8"
    "Pnk3q-|K>&uOfJswybHXN(vKpIZQHBoyNDlWa;0r2$h)@A9i7+krDS;t?k%A@<+sr%lm8cOZiP`{fem}xyeStU^^D;S|tv&j"
    "fE0Jku$gHAqirxbEQU@^G24Oa6snp7#|NlMD3Q4lY=X7?7#8my-SlCOHJ}Iue4+UKD4&oc0pHIo1b@ED}D1z@E&KRtk$2V9%"
    "^IvR`y+=Yj*mHJ*$?%(eLYwN76(8S7~b)4?ZR?dG4KcFZ<X%Q#b+M@Xt8<_6u{~D#cXXY~0+gRB3~?kG4K%Ghi1g%}nxIB9m"
    "jtw5F5@xwPA?Igy*|)a+ZOdl1dljG{X`OPk(mgJsQ(B)vLq><9CUS?k=CuKOgM-%IT`JnlHVIk%~iTsm;*3Ii|A^Iapr@pL}"
    "L(?ij_NyU?F^2(tz9c;{VIDEFAB>q+8I&&SkL#u&Xzf|~f(0esE%`q~qVyF44+B<dkXZQg>hFi@Ibm|fHs&Ufkg`fAy*{zo#"
    "?&G9+=$xX{UhPQbZxhe75Pl<Tsgq&+^zMOMm0ZuE(FH{^5y5CRrLYdQNh``o6`IT2l%Ar;w>5qVudzQl*A~BgwJ`4Q)(f;B0"
    "y~=oPUG6LP^w?zP%;fS*G7c@v3AT}q_9=ms{M7fy5{pFSWfH;I*Tl}VaW%O&}v-@UaxEUBn~n57K09ddJMKWeQuVEJ0ivN#("
    "mKmcC5_5U3oOZOB*3n#us$|dzpG=xO2V#WsH}8qhU_Qo~$s>kbXH8+~`Z>?%Ri1==~R0X&oB&apoVHrahvznuRmH*}HDq+wI"
    "!WO`F`V;rU@(Yw^n3<Hg=GQvwt=yz;EUw)U|SyCZ0I?5}&oEy{=7k#8N&@;FP9PIX{-fBQMV1xd<#>dokCjcdN5R{ggAYOC<"
    "9v8_}O99Z<+_VwVT*c9KXYH*up{dL1m@tH8Od(qmwaH&4sb(_17TjRr;efdg_CU<vqb|z(Qa3ZREdOmfrB_j;AC(%7I#O!rf"
    "ZN!w<S5^<Vqv&Tnok9Mkw<=5EDo2Fg`Mv}Jwk{>g4Etrv^Te7_>&Z0c0(3?`sxeud8-F$8IdWQXtqYs|5S0Z(^gH4)mn#kBO"
    "2KTq{?nftgFYl)3fY}MN7wVcaid4*a7eHZSwwmlCKyr>saKTlmeYEXU9p$1j@;HXYP-XO?+Yc9nQE<<jNHC@9u3Vi*9R3zoD"
    "JfyJg*c@tFJsby{}g^xWE%4hRmrbQil=zu23}-A(2rd-E2fG)2HqJWCEH>d$*yVJo`kq$H}Y;cU_`=mDH(WW<~pXdC@ig>a^"
    "R@d`@)Ikaqv<@$=eYAWf#Lqc9A5-hH&9%B_>0)JvoVPfp~r?AW`<<0Fd;;5F&oGk%+IJ%HJx=ObSWYE2ZrqGe<Ci+(Eui`AB"
    "L3^v)R66*8jXNS%n@oxIflAK=sd^p<9&P43a@*$YRi1Re3Su?@*T&Y^j1?;Ai(|WM!8{TtpBYh4F(ek)8<&i}C4C?`3ThCpV"
    "tp)i$e5KtNigeel%d0LlKw{EAD}QI>5YJCnW7`cDs_^aftrfAz!8|K^N%NX{x5GiuNeg`y#>#GAk4iq6o^wFx_4^<}<G5H~+"
    "_=<v5Sj!6>JAQDscU+~c9#bS8st4p)*>~5(#@fT^e4;SUfevLzx#~1r!z=ii<RS@{$&)f^mAKG)~zu)WbK|PCK0#4-@oehMk"
    "hMi880o`b+K2i6k)<hr|Du@%_c&&q`Rz<`qg)%8A{>X?{7!sVcY8mR6p!u)BO)OR>x-0xsP0n``PaadA`9e{GK$AHQ^u^CiC"
    "`e`!B-YP`!9>k}QeKaa2~zh*dDv{G~s>iTTr{Umw4ZoB4F%EP5Da7QcYH3WsaD3oU6{L~rP-o6N4^?#`*gL|M)MkEm;HI@j%"
    "@pQS-lj2Y2^DHVlECqm@~awtWGq#Qr{<NwUuey{JDxN}{5ueJADdp}}moE$BC55_T)_gl?w<MO5N&AfE4)R+Cy^eIMDQPHF&"
    "E~oSIPs<H18X3XEQ?4X><wbcMQed-LAmj*SSIbV3-t}YeGb&$h<hhO<>_NMYosbaGYzcUpy<~DiqVr^V%I}jyKQ+M@{a7c<s"
    ";CBZxSVkYCp;Q|8!mynlH%!0EvNNmHYwfj-oeC$G8txIK$&(qtc$Y3zQJYXAtah7(T89}b|MkTW;G@KEhN}g-qHaVxvhUYEm"
    "vp#3=y|^v+&W3O#&x2P&#dJ$-Un@`@!mhkhi;PN`?1kW;GCxH;h!|h?i{%+0T*buhZ#vIYTCJ^neHb0}5Y_9Jq!eF+IKW=N6"
    "3d>eNE+LnnOS$HpOW>WfHp+tb+?{4`4wUb`Nq_^PY=f&uRt31El~WQMou<09mdf8QZI{xiB?I^0kiN?~I>u2Clj98F_YeG^s"
    "iI>RL9u)mEbqw^7FZ-$y?zNoI!V=Zmedbesw7dq0-5F(l}%JRPI{*1S;bGVXgU4@^oaBnBpi}e${6T)H@B*Wp-8CKWz)OUVn"
    "ry8^@bU^I7$%NC!WX|D~h!r+hgIBM&Pb%a6LX;Y2Q73~g8%gR0!q1KEe@NxFxTQ0GxpnMopIfX<0$+dQxkj(rp}hO-Hpg0IK"
    "IenN)BY2zfTxcd!l3&w=1a>ci_f`KfU|D)#9p?-`$~U)U%&H--yie%g!-9zRwun#xi5{vt+JU51M$`Xk?pN=7|nEBzU#EPQQ"
    "J-|E}Cbyh>gMh6B{o0YV>o&*1uG%t{a|M0t&)R63YXsaOr+#O7EU^4@R$8VIyyJB>N>|_a=+Qf#GuYae_U{9j8>1N}qn#q)5"
    "jOr0ndup})Z~2rOw>>rB-&DZGKT+Fw<x^-)($){_v)%2fkJ6k2pgVjAyqn@v=qy&pf@3CByvJ8{0T?V@#TtM<KAdlw7O-f6N"
    "==M$)}0p*%?Aa^YVdIjQCn-}f0aO9Kn`fhADe|uuu>Qv8dMkSLIPfiBR+2kP@Spp(ne^5g<zUr-=IP&-X^G*>xHxnm^6PFf;"
    "I5b-ODzF+dyMuS#!V?eb0nFvW?Y+}s1jhA2J^A^N3}#<^^gE5ZCSBN)2Vb7`rX&5B`noy%jE*e|pHF3OOxDKaF=4g^)sJAoQ"
    "0W<2y5)TBbt(?0r@%G63&Ww$aoOq40^&03cbdWh_(FMot_g+!KfGz)-;-#q6@FHB!@5#;q5M~m2@BJ3RXlN21W`++d4@y#8w"
    "(DxMLW~{F>ujg=@_TrT<Gu*t@7oHQT!QP$%4A`U#NVp+%Lb~)D2wkmW?Kco7kg8bAuOj%N~cOjdQ!_V6;T%N`ENiy699DE3e"
    "La^tiR%zbLn}EYodA1A^sYaw_bTOg*)CY|cmu#qo=x7n~k&tG6qmTjy^%*jgxzS_xo>&4KfPPp@F$d%bSCRJ-U^w(+zc!OAV"
    "p*!hrd8(GAkPqnFXYn{Q=8`?Njt!xJGIy-GbCX$sFowt0D8?rsFjxBg^^Fwyo5B$F}F?0<fO4`@Y@uf>%rvCW!^aseR!&fli"
    "alquvnY~XVZd{^U(1+QmzkOQYqb?WQo#u69w-xB!eIOS;y|mhAXFgUM;m2;4ZT?n!_aCjv<=pIu@P1Wj3EC3J4H2sGOsN=j&"
    "sSJH6=4ngLPyWQ`ai98)M_y@)D64q!FaihVsLgrP6LH0oBdAJta;U)7(RMJcDr+h<tCgltSKI33cO>eb0o&VIb9`hGJ9l&{="
    "iMj36QL9LAiHUmo|_?v&@UDf4+vaq^Xrc3fo&yD(&y9g&sT}w=B;x(o>1NQ|q3%AlUVr<o=q+qMyEU3Je-6Q~>l!y$&sB;_m"
    "z`6uBiTF4z}V7dFe!0O>91)^H~|hn=|mvrImL)5&p`>)T%Tij;YA>9ADN-0Ey;PGNoxK@7a+7A+v4PyFfoPA0<=-qa>urzn+"
    "|eMx(5O0x(smL0Ezzv)+zN{_^;_F~$<qPCZz-TC~HMo|2NsY;nc`}5&_q;#RMvE!n3Fs|9qE$_lOatVADukd-he6f?nd#}j4"
    "%xn&ZaiXWx^X3WH8+@l4@nla2*9MghI`2_gUoilO*Jhj5()Wo;__z7YpErAF`>6;=;<5`ZW0vr>c)BO!GC2mG2TN=gp{)DpP"
    "bOkqK92DF_}Q<v+9NXDCx^RTtcXz%u2;3iufvmek9&uij#ZVj9pUPCB23UB1q%*MC421cbzhJ4{eavr(@LZ0%GOmIO&D@6SJ"
    "ZB_wdzI27`#3mPH2_aW5X;rXRRs0_)%~JRK^(E&*SY6S;P?aARFE&CUoRWpSg1PQDV2>)K|UVXE`Q>6#VQEE?wK6N5`OlZ-C"
    "c_8&w0#X<bXj3Bf$o74dFg=^q+oy$UWCq1HilXEpxL#HLnqr}y?y=+oKh8wEG_;cK+q`<)_kv)>gvwPqd6-OajcYZLf{uR!y"
    "v))3F`pgas5Yktgo@h^(D+I!%nQp32tw};6L8tz*$GoDW{<NmBg<WN4X9?q~Snz%nG7Ue4YiKg*-Zz>J<Q%~ES+=%7m(pb+B"
    "vkBbxR|0-Hv*pz!Pd}72AHXxM^F9GhaZ(-k+gqul%fs4sjqVzBD@6VEy9`OUt-2JGTcgq|A2FOauKB}#-sp2#n{33jh%d)Y<"
    "c!`8VkX>5{P>0_d|})$NpHTpKz}vj+jf+t;mcK@jMmt1UPuX<zHa@CsEPio{=Fm4FY=5udsb!wn-etpAe<WA(tF8u%1o)Wx~"
    "w6hCDqOpemGJiuGg3}mOHCi?BzaPTb^c|1%Z6qQ?@}|H|0*JLY^kFC$$>Qj9HG2^jzxPIuG<u&c8vj>*vno?5f}dhS)1A2{A"
    "E!se3ZJC_bOOy~)^NDs%31un4|eEWlxFe%e&`)A=2a%IIgZ<J-jvy_?`tB1xb9PjX7k*jtjz_>nIee*F}iY`N5D=luy+OEzt"
    "~4;C0#$#G@ekeWFijcw^7vxDV1e=jk19hg?vnPc|zUPis=xI4^Cr6$`~z9I3@be|Q-5hCV~LNStH_sE{wrP=wlW>7utMl*8?"
    "i0CYCPu;F*Po^Md71m2tm=)Sq+ugLolkB|}f?cWg`tDvp?+2w_WAXiE88@m0?n{7vK=5+35A3nETVQ>`FRM0>J|EL)S-1a<O"
    "d4%hM9#d~_Lw}c+oaG`V&zqcManVFpg1u{^Ls*B72~<y$=rLlXoF|1)A4KPJVvwN9b5+4^qEITwR@S<((HVQ`Q`lkze{WEj)"
    "s-G`se%E_0t5Gl{e_%T-c#Vs@J?#vQL4hi#$4uo{H+%1A)aqFhQ@c;R=+=VaM2Sw)lo>lhOOMr6G!ER6v5>a03gucGuRP&Po"
    ">v3<oCRSW^C`vP$b-jUDwP<e6u$->Pr$z~Sks6I_nN|Br??Hke@zwHImZk~LHphr5c<0@)<mVe3g7a*4xc`>CnqnUb2KzW&4"
    "%BU`|Z`64a)%X-G;JE@Jlkf{7>K`vNFYV5&Zt%`g1aq4hS={%Io!JFC_-5`Q+lq-*CqiL|}l#alAuL$}SPqswd;ftnyy$nuS"
    "uq<Tu6C^KczDL&5gHjnZ!cH*~Oi~_O41Xr1`J$@L(_5oS71}|eTcUMpY$b1e&fstUm|frDxy|2)6t(!w_xARBTY)vT(JD!=i"
    "ABYr<jD0SOx?1T)#5ugI3IfxjBU;h>i52=tTxzul<l!wo=U0zhL;0>t~n6=QKtnscU40&6l|()^G%WrylY*({*$UkgFrZHyk"
    "B7UqXR@cjYB4Z<1c2E)j>D^2s^^-f02M1$aMnmY=n3II#+(Njfhh2dJAV4klWLm%Fcka1WY<s*vQ{q<HkMvrNI!Git{Y!8-t"
    "@NuLoGGBjY7N^5*QTvMRr@*>A(|2x>sqX{20xXH9Vi3q3A5+hfUkHMk`Ltq+w3N4&@Pcd^GC+xFv89=>wIqM2CcX+y5UMrA^"
    "IjrH*(UZno(c}!0q_l<2Z@fa)><|l@3)~DIp9H`0O#oc_#Ao28bHTxU!S#8(gJBz6GR*i}(In0C0u(03zl}@(t_}1msA{1H2"
    "6Yw$@Vr)vW`zEX`@KdE%6T5_@+Q~pk9>2zm6E}~;IihbDP#rQ(gkf~++mF79?o6N09uZN#k%jGW2>I$eke*p;*aACQVvTNvl"
    "Gu0cl+ER4*~qhjQzOpZ=1i_Mi#|3;DiqFe-Fs@5tv{Z1T=kkA@?Kz?KDKX+=i$0TV%Pyz>d(%0^c478aXun*<&>KyBU!rsEN"
    "L&{`*ac`A}m?F>c6?Wbhm8-6Kq~lFsIwSokw<qE?&`RAs-B6xYApDq-WRqM)B$eMMyY<b3K0~<|VY7^sH8-oh6rL50@M@jH1"
    "=K01>*<%<TiFyHD$Vldty!`Ue+o<+dY2+S6+@trjnrdNDvprPj3qKO1nG`sI#LUb}7E;NgN`)h|2Ru2(KR5u6|We5pv2%2>v"
    "XWu4*|SL$a_YWQ%`h%wKGJ3Z=5?RJxLvsi=?h9YL`pnQ=@r?O*))FbD~&^NNV)MpWg+;BBsTuu6hGP97%#@y?HdW7uF6of01"
    "xAEQ_rSD4MR+NV0zcRvO-$t6xd(c_>l?xcfj9(q9Ah9F|$9r|Wyqkk1M!!)QGl>yAPxkML?6SXklgqe^X$36b!t?&Hh^=sff"
    "zjGywH^Sc+b-|y89Y522Wh#8%gbUjw#E2<o(;K;jn`YUwFw=VT0^T)=yHw8?WELn)<kBEERR1D(JM0@Ec}f)i?j1e%rYMKWi"
    "C4*W3e+2#^#Vc@VmM$2?9}B04R30@lq3V>H=L@t9p2@Q?EqQe1P0At?!L{IKrEYxf#1#Sh>_j3v&9fE;G879Gvj)EPlaWJ-^"
    "&<K?HY+GON(QflwZZa?dMs5Erx}u_qqR^vn5wyH<TL$hC2iy)qPCev`B9?pW|UVM9$3UZyS*wLf=Thr3B6@|Zq1({8kp)aTX"
    "LbUK?i0=|(8T66!oDN_k<oz~OE#VwNuXNz@S+T!50Lp%A+bFIbY6&a#ew5@z&`*MeWw%53&Lc!)n!PHz5_<+20C_&hR0>4zh"
    "AG#-$#ZfoPl^QX}{h^gCY5bLMBM`hNPwp0KsH=~*+9Pg4P;!6#aUuo6M>U=$NH!E|QG_6HQC{jBWO2W<$4wS5FV79gAQiO=$"
    "0`eDov3b=sQYi>FkLjF(%tRb^lGxUo3*=b)YZheIC8hws=c^gvRQCCe$c?6)q{m$-y$t{9fj$AY1)*3*(m+l>+lU=ty0nHpD"
    "*M6p*gLawKyR-<DluX)A|!4!UR6mEde}9X@7^0=2_{mB7XK8@j|N4MP-6g!-|dHa^Z+dhK6db0IYloI#O(C0<G?#XFTtcB_2"
    "8ejfyH8=SDag6ncoQ_CIAes<j7h=zZh+$S$`gt!Dp4h4Zr1t941|{--oQ(f%(reS&9gT^l9|99P&YW_4$A01WD-z@$c-w>CE"
    "TeR7%Zob7VodKigDgQMyBQhKn?Gp*pfl1wyj<xJhFs=e`2ZzDJ8sf!j|GtwRqGNiI}ul<}>BD4wEQ)$;LT>%W3h!mWTwhZ&d"
    "0-2T9&Agz_QZ(Iirt%RE1ed;_e6faI<)&4h&V1mNLfVebPqWggQ9tk=VV~?K7zEmCtz4$W<o@IQ;r)yv?!!q$^~z6#YHDPK>"
    "2%-j(f2hO06gbLCA(H1@aVC*e9*$M7(VV;l#l1xg_>`93E2bP<CPz`nsG!}$aPH&uqXNfW<BGsu3qm7`!{QF0<>lgIhc*N#f"
    "`KIAJ(q0`gC_0h;zqk$tXG=do}0U6FciCmTA{%sLrTcgLtuP%1r9my{R8dZ0V5#^H5o4GDfU_<EY<*#vMxKcZ&d<Nk|QAr%7"
    "~a-kLJnkr6?eq-6(AqSk$*4$h*}c((B--o=c|+Tvhkhjk56a*=3>BjM|&Z!97Qg;a-&ORal-nsJ2(f5MSc@rfY(E6bim%gfd"
    "M?mCW>Wcs=`YU<T`7Y41iJHOskGn)?q@p&F2s(u-r-`Zq5lV@zq+2{Dke@u6zfN35=QfYBpsO#RXlk?zv$A_c8iqR?A<n$Pr"
    "Za#BLtie;=C19Q4-i-^dD?1vJi{*ptd{fk5*<STppN|v0T*5I6d_?<jXXp0>7}q>)YsOxOZyr||=N_GO5y|NMcICHB^ZaLXa"
    "76vGg@M|5jz-gmNl$BG;l}2>06CaVw&7WimErdws=eqxxWc-3*fPUueHPX7DUalX2WU+oSSYZ)+E7h&)>;2Rhkd*}#lBci7u"
    "#$xDcWPFc)ZR%@^I>x?@urnlC7)HL!0e70ZE>?{i=WN#<0ir`o1@NfR$`;9P}`IC4^|PG5X>+Wf=Dd*yGw9m^JZNx;8f}RU5"
    "!1p%>g9ewbF7gF9M&bX3xYN}%Drl*Q%B=p1gl+tOlnq&{C>&izu~`62fsDz~4W-`mS~2ozU~OXGxHUaw5m<_covX798((~@2"
    "2R)+5)`P{Z>WzDe8baD2(juGFourhzMo083^Bq6;krS2U+H7I=@lG)A<^uKaW*H*v<J;lf3Hre(HC3=`TC&58-2o*NvbJ=T&"
    "AGcAc0{;}Iq!u9z*nfy_KTF${@tf1~Wyb^5L1v@uLO?2FYO<-jQ2p%w*|tB-AOnEL!I;~;-5niH?}Irz`TpDEbmim=w|Gu?;"
    "oeCnu~Woxch3LiWjfzK`w2DUwfU3N{b|o_@3>yiSIhRtJZqK0*mP<OhN_X}%iB6zT+`6khhBqd8<`LHd#CtT=10*E+?LMBlg"
    "BExE~O7k9x+Xt@0K(2tiO4PI<wiNr^UFj>x_f4P9K9<h<CR=zJFlUmmZR7$E!NSaJcp+qVDr4wM-a!&W@kNHs-Z<X2J(nIg5"
    "&;wl>VAcD~*r<$9L}M<D3C-*ZM3{`MMr-*D`mW&-2bf)sS@MtGW(kwfV&ce*;%K(ZB{I8T8+vJdlnj0<9?DDFqWY_M%W<~Li"
    "4gg#Q=%YGWs6LUX%;=Q}=7%;{P)2s2GN06=mAuzmF%`J3}ZiA#e8ypkfd96ngsEsP+r_yRzI&eC=n@cs-#}gIB*UBQi48hTk"
    "><BUK)zbTHR_@Eq%!9o8C^<LA#k9p58Nsx>coAcy;h^|8HfZlE!*mO9&>~63vO6Aeino|-j$dlBT3vdSF6(Y^ONT>$xw^E;e"
    "_f#!*m=G%(mnN>kF(uqalp89?4f&Dy>GXKxCj}f9DW0fwOAYm!&?lZqEx=0LHS{n@%7`%#z1}}fX@5qDK1ft&9qLD9%Fn%b5"
    "b6U@muou;yZ<TnX2G0%feUDf2|rN{d8i+qr+y{x}`2+-)0HZ`B2EBF)FU}q7vM$+*_}ghKj@9fLo=FfZpsn?2v;(r6N4h=D-"
    "Fm-;!ZoHz%_%A&BX})@UV6O!5ThTRuE(0_DIYeer@6Z-@OC)ocpmiN9^;2b;Rp2HE9UXx?fYtx7dMz=*wX6m>PeBDK!c{^q$"
    "`2s>As`f2*6%U)->Xg8f=WD6rSZ{CdaN9t0%6pkD0XUoo`!hkT@FP-SjX#Lq73fg7G0_S2ae7G;i*nfi4K)%SkABJhMvmF$j"
    "XyDocrq?z1Fb};9caso5vX>cme#tGjJ74f)?PiGH<X9(m%TqI}*qw<4@R{TxMYqPh>c{u#GF9hW6+JCh=S#CVu|(ENQ8J$Vm"
    "T5cC{575LuR*idJJ*GyCt%yBSTcCU4$R`yxnaM$vSvxocjGx-_nhUt&I<7W=`${*YK15~`#mKXH7M5`Pu&Ju!~GhY7IWc4s^"
    "5FUwbcGrIUX8=UrXOc%G>h-F3tROTiRVNyHmZTN+#ws6*P8+){v}iaaAsE4-eSztAMh(%0Jr!aLbR-iQovVG<wm_C;3unOJ<"
    "FIW_b09kON?bUenr8`i73@^qbD`t9MxHRb}+#yPB^8(iv{mPx!gZ)QC~b*cbhmvR2OW;zgl(+OP=gPapI4j9EBiYD)PnTH@-"
    "B@N<5po{r#F-+ITTt3BZ}S4;;&`E>;<i|J)?LDB|!cV}ZyWe+IV$`+%3d-~YlfB6LWDvCI1KU;VmcjXe9doN(pdIi(sd#Rda"
    "{X@%r=k=;qYGopTCwm<0!G+dR8UJqSU<L9!nLA6LFPo3T^)tWTE{hYO6Yq52OYbqkEv0V|vyR2!IWVey3@Xj6pf$`c=Ak?EI"
    "(XeVG5X_Z5Y@)0ph}>3NBS=G8Ts08do*8MGlsXU|7M}z3=Q^X<GDK7KGb3&?ONyVTE1}fDMyx8R=M;_yHe%k@Dwh+X$0uXCf"
    "eU$nDsHee1z-3J&4;el2=}UEIMF|n}P3;SEoE!@xP!sD{p<_@51&3(4CBwIt}mEm8ABDHtYS2c4*;5;N%^sy{UHK2@mu%UnA"
    "u@1cb<axxp@F+J7p1>z;qHxisHQB^+e%TE0GSx=D1}1#zK*k&~?ZWt+=CnH{)<%C>OQ?NMV%372j=D@WeR1QNo~TX=?UF@(0"
    "G(YaNa)2rFhrU)S`1RE%SsKAm(E&#T9&0D8?_&DwnM`MO=!PeI+*qyE>AE227dh^fN#K^Uf01gO)g0@E#$l{b+V90WB5y9%&"
    "CT7_=eh=N<t(dhcxY<2Dp5=9f=E9<{N7aZ!$+c+NJ+pRw*JA7&s`&n8`BO^b9mp(A<`=D{%q>(+Z%TjM8uyGD&qc*iR@@*F0"
    "$vg5Q3fQqUK7vRf*jtK=k5a>N#`z=9KB(OPHMrXYZ{vo-!aHmSu;X=!}k!`ZGnwHto|_2tkY=itA5cO2dg?tgIDJW52pb#!M"
    "8JxzJ4|>$8`?h#IZtz>-gPH@E1S0tXA-T-`t%ZMiiRoTSXXOkMePQ{~>j)=<LejokKT7o;z9hF{q>z!xixgMDO2LwDD1>&$1"
    "{3r}@QQ%PU>}MrluJtZax73-$bVKMSzZS&Gvp_q~C)xu%)_mP$WY`T05B_MF*b82GjMT<IzWOIlc%d`GjYRbJcnUi59Whu_4"
    "astWzE)H)Ikp|S^p;`N&BcKj7r;Yxwmex8giVj8871u&J?UTllnbo>a^x{l7IZZ3GGll={$#IhyLY56(djTWL7Iq9A%FrC7E"
    "dS4{(X-9UA+Lx<;h}0Ebp<Tf^q+UcX;`qE?RX*A4&!l=`vb4BAc7j2BiSqrXHVw}vO>=mOK$79z@hi|YJ>f@m{Y3Z5ZIV5Qx"
    "d=`AFJA<@yXmGBF`o7W*Lin8Z;Nl4*{K2CsN>H)*izc*&x77rKa?%D^&0u{zy~j_5>4M<l4|dv<RU!xo9@IPl1uDMndE6gE8"
    "cB*mr9#ryAPGIFN9gac=uSh(Ea}7cERm^bSG0!a_{Zqllm+XB2iv{BJ8aGEL|v()#i*6XLCp_#vogF3q6_CT<QY$-hJvZ*eC"
    "~l6P~k?d64P@;BZrf->;fBIw2OYlWfzZM?)W6gxt0FS%%(YmDU(*kg(E9q%XG_I6@|xJnY3c5g!Zj*i_V;9UqXkUd4f9_cs5"
    "+`B@REdfK?J7syGOrhGnzQs>v+TW<SON~TTXvL8jQi4q!_p6(Bf4Ofo}c=X9Yr3MO60=Lt7v?+1VlEZeUSAN86ql0POp7HTl"
    ")|a1XX?4~L1wU$7Omy!+6LN7bKywc2Mo>7j=hflp0Y}dBcyLgTqx0o1j2^oS1~1w~i!Qyu?D-BW_c62|>R(*6_U-B0y!sZwF"
    "6w4%72?0~>#gq7br==gw|*Y(GR2c&aNRCfa&hbx#!$if@L6pT`_|*tq|qBRD0%^OmIpCtBZt~OZ;n|6V}{0VJ&o+T@ThGvvV"
    "V3un!0b+<ypvfslst*K$T->)*QLxi};tT&AtH+0LvY{vvhFTzG{~3**rhxlf0LQn$6sD66sI!z&X@#xhl2_Vm@{o)B!c$P<4"
    "XPqSpAlo;i$Vvb^8yfadWtXSAmx{F>msy252<jw)d5yv}5%po-JH7(N@fCUzNJDpXSI<nXL+>^a=AVA_y}gA%{;ug97r+~$$"
    "%TwLoWL<>yps9^J)$?Nqz4Tw^K@ldrcoy_*%On9~@bD!TV6m&1#o546()cPvmf^})Ys2i5-&_bXCV`^~pF66jCeeZZs4$$KD"
    "VZi%9w0gtR2C6apYvU-;Uo!QGeWWoDrrm}zS`TJ<t+#+F=%jX0WBey^y$svweOz6fQDI9JPWnZapV*ds=sk$I*6rE(w|1svr"
    "kutLt;eXMKr2hxiXAZKKA-1}s5u#qg8+-vYb#<tU4}2Z?&CXG2#TN3bl3HmB&N`I?HqQH)Qd6?@i@ugLi<!Qgz3dqpm<$7*j"
    "@eS=3+Lt;td5BcAU1Hy)r;KNs)KT<*+F9j~2_Ebu!1q-h6G%>*8~)Y<EWUj$QT$RA}8GWUvpUg-%}g;$28P*v($3rQ&qlT!c"
    "W0hOtN<{f*fWOjHS7LuxY8q%CmxgI_&_z=Ntm_}O_V5^h+td*(0zMa8bS`f1>ku{7*7Doe+w5TD&}HZ$$s{4SD+Y!|AC+&2k"
    "lHrV{=kTP)&8PUJ&xP7%ZlnRb9wC>0CwZ~jjcg=&s@H$)_4``+9mph1f1O^Fn9agP1@LQ0FTkRR3q`})EJgS6Oi@R$=t4+B$"
    "3pe#@9_jt#m@qeSOY6O`k+FZzg2m)H1jeo60_@bA(+z3&;k&=Sf3u>2Rt{(n;hw$R5;DB<D>IGCL9<_U%91kMSkG9QA_Is(-"
    "Yv@Ngh$o|Unj(YKgzZIT45LbSIrx3TD9W>e+=Ht>yEu@lij%FgHZ)5j~NEET5z34uV<;**V?n~Q_U^p$&{F%s-$oCA!`N-@R"
    "v+CgT>A*B3mSQe>Ocu4M8$llg9{GfJ>?wr7jb1(8@<e2QgL{eYsd$p(k6crB4=_=UBCHQ2<`Xt>ALrv7o+)&lGZa<1fwaun>"
    "<sz0uNWl4t8kwW3(QU<)Z>^7tdw_Q#{OY2oCyld9hbx&2#l8v8R6>yGLu(O8c<eEg)tp;2VMSzTRK(g*hAR_7-iuhE9RAz>r"
    "k(Z<xq9V1qWA{A>)(4TvP8FAD~+YiZW>E3c+_IkeVz-J#hK9Z4OX*V^Xn0voHD>ZdyJuf7k43eq#bMrfJrDnPxXyNuQF#@>P"
    "=;6?YXGc154sHRZXW<swCNrb{kr!l?y&CGJHV#(;nL?_#ChXF^7Z=&@QSUhAj+?!MZ;E(dwIJ02vWwIno05F|li{Y0<nh~1;"
    "7@K=W4%1p4O;#bo@`+x^(P`eDrW^4cNUy*Xq}P$MU%#zNyXUd{pbp=p;q+SSMbXwA8OK}p(UoWLW|nHeBy^oHri+puJBy_%`"
    "0m~(@P13_v7qp7tA^yBQ0)lW3K_GDB0%L)%_O5WlAZ`P5?KtqgjkJDs=Zn^+v-YHZc(G{G{w6{z3~+R7>G!DZ1DLu)D0LRn?"
    "`(ioE)pNxj}MgZ5%Di)-`BDkh>VHT2stC&NbDl&)?5K)XZmzFr>ka(l{s<Hk*~N=gSNC*U#A=7VL*2_iQd6OG4zx>oIU%=t_"
    "4*kLHW;Q<jnr@OzY5Gw_FO!UyNB`siFnW6UfBs3wSlO*q6zT33}`Zkr{QChK5F|>fzG7G_Ax!M;aa{{M6VxE`tN4WSV#E~j-"
    "eWVF=wwYfsUy}~J)V#)U@F!IeIxYxwimV~O)T+%=r-ypwG$t>|aRyYZrIw!hI7Fzu%8KEKz68hSw2rw9XT=7$1I?E|vYt+l<"
    "hK3b&Rwo)SBHNU&X^aA#I|Q2vbDbcA=CMZUkU2Adugr*jP9;M*`zBzH_QSUtRpRlYxTRU4oo;}r*=pxFkPNu<4&uVmx<m~y="
    "g=foRK^2w3GW5xm;hkV0D^((F|T62oG`Ndd=tG1es<9G9SfCVOAikl`mq}F>(|c*NO4OZWcD$087aC<Hm~$sXFyDTE3_v59C"
    "xrUvzCa#<2I)o61w_T$&@PYh^Cn6_(}c_Qs^!{+2*}bvas$%<at33%iA<@dPq72F+Fn8g!lN^xh9x!4;~310&QqPppTC#(y$"
    "cr(#M&A-Aa?0)uh$9dER4Nz|zFr6iFtV>O=oKxcOVOq~w`<dC;`04+wat+9nl8Er}X)k2*}1OBr+<?R)>gWhE8)HrsxT2GQH"
    "ky|n(e(;bi=0kI&bo^^sC6?beUX3%jUA5;Gbm;}@>A+X^Vx;}lrW!J0`o(vK#mZ(ozP}cTseRIK>Np#%o${XBuhtsxZKPUH*"
    "C=`Nz{fytXosWGMr)nRxa=;mX~@DrF{!WE^8lb<dCyP9bOTR1gW$$R<^EqmpB;RTORYb=$Lqto5Y~tGxw%x}MX^1&=@dLsqE"
    "1dXN67)#j_PF}lkwxg&7J_-JcE6H?H)(fo}slG=dHtf0HEI4<Qa^P_VB5MFK9VEJmSB;B2^iWpvASP%-4hRrWtjZK83EX<q3"
    "1MuY@|j>7Pqw>2lHqf-C;SM;IU<bcqma0&;)KC2Mt~>o015Xgm0|jJ(jwzT0o1qr8~k`>_Viw%i|&A6UzEbM1R%r4)*~L^;I"
    "3*><5WT47tGc{Ns}Vrw7HHA?MKnr~_T<Bhah!#CE6V-H~l&4Q&{gU7Dnll&&n7fZ31F2~R3R<IUK8-#1y<gXA;-uUehX1b8+"
    "m2QiN%K7$g7J8kV=qXPr!r3ZPL*zK7OS40@hsMS08yp44cHt^SmF9FtoyCb`=SeFXw#xN`T7C3UXqr>CZ@L})yl;BD2tA#c8"
    "8+$<JZCMNQ{q$OJ(wEkc9EAm1?=pf{a(IjV!X_8%e1m6T~qk16zND8f#Z?!#ssaMdCv>fi5OP)3H*q7L~M1hP4m1-_J7qD?`"
    "$j?bseKD*UHP*G48SE?FiAkbmuDI9@WLemYCB;9-br}Ka9GqCFlq!c<EY8felK;`SOzFbMF!dpZfD-wpnE}6r{B9^RuIz&gO"
    "Z2MO%dJXlCd2v14>}55>iy;O@&ZZe;CsdHN07&?3U;5vV-n(%E@C`C|e?**ey1G+XIyGox$VD|KekNySs;*GQW21i0tR*K#V"
    "_)wvc<pCmZmF^z>2mIv2g(eYWKg6Bs=gzcaWkqY3wCUE^YLz@9I+*8}!oGIqWo-GfuFErDL7`aypgPUWm+}%iPn{wSwuqq2r"
    "p5|+W-1@NFxo}LvS*JNjdIg?fJ^oKgD3Sgh8x+&Ks$aqpB7sjiS$nf<e}Z!LY`UakvUXpuI(OwhsleAY`?gA1wCexjLGMj<%"
    "WV=hTo=%R9kvJ;M7#C&%meUX^oxC-W1YmA$<~`|?}8IE1{b4rNetUFawx;bcE-NR3n~p3vQVea_v@fSVRd3-P)T__9gCDW1p"
    "heALVBB{)!ABcu<VQpY@8>}`|LN8?u+afkxGS%z?$_Pva#xdNL~uC)v9xk=SGI_$(UwNFjFW`3WKCj_jc>egS%^KS9K<933-"
    "y0C(q`y;qygt&`<iJPCSlJ*kd&D%JuB~eUx>p0~krAQy4T+q<wZN{qvk$2WEOf%G8n5`#{^V5IJI)<}OE*$*qDKE8oQNq<}9"
    "g%7?LFKpsAJPAWUw(FR@oiT8ADTd!>g!Hk<L&5BJ+yJoTYcIW<bIgZwk{WI!6d2ZV(oYtEIv0guT3dwESDsQfv4MomM=Day-"
    "-!+KsN~E&ayXr`8Eg?t-iKYTUM{9ij6VVyHH}v|+nekJiAKXC&d8d!rCMC(mf@`^P5HuK=BrOa%05e*dU#>?}UG}k|GQpai0"
    "j4HS{jmbK7~5)Wzvt&&=yr{3jc*t0pUNVysi7<G3hb0~4f}64<Q_uD6kXn89i_J@7cBR3F0~hBTzC*~X%>^&lLyoLF?Vx2Ow"
    "VbNe3U=+W4l!I+s}Q+-}wIOd7<{bw^A;&8k9+uk(%0zT|gvK@U*GaQ=$2%_PCbs4RdT_c#UZyjW26CZ5F*w$cnv+ey+>ALw|"
    "Feuc&y^jdl;}+Ufh_>$aPRBOu;uw%f%jMtc*_ZsiF&>l{>;30|tzYdo3r!eRw25@&x}AaH)xdE=0Rxh0OBKI_fsZUkJm!cHR"
    "&U4bRhlzmn9xp3@@iNy=*xWBttP|9%Lqzvy!S6T=Ub7qZf>SG_SOKkIvotqDdwbp_-t<RR7<nN;(>g1PA)+rMzPWn~=z#Z&("
    "+aDUKfG-9~UOm(-W!~&H#FH&NrsGGc+$p7~&^9U?`M-3%7_=02QW{PgKYbe8TdIrOq6q5|pp|>i`w4_7!f*lB!A={Z_1nC<c"
    "Ad#gx!jqM+FK}>>0tRdAFBuZ9$IcA2kP}viEMmgj&pbgebEwD<nR0a%pROB>)t3p99!&ME3~<8?D9_SqKi<gz<iLc0w1Prq<"
    "#ti3?QASGC$^xWrhL6Bt>sMV7h#zAj=yr?dpET_8ut>b-Z}=h%xa)HKy41T4=9+U4R2nWBk6kS|nF}*+QT2cGo{1;|I#`8XJ"
    ">s?p{ack)V(AymNBJZBKhK;l5s?<!#&C>_`7*IrVX$s3(PY+Zc+8H_1?36$G<wo~T1}&rQ$xH`&Om=Kj9Hy<W}h28-6>Y2X4"
    "hI^9+PfOvkl)T-y?kQui{Q>#?=+N%tFBG0?c0e+WV|At{)J2XCRwO_jTrfJ>WDsfO_2eZS_Lb6h&4*=Kx$*p_!!2MFv)?@=N"
    "&Fgy5xTd3yhfH9rzxGY~-Jdooa`S7v*8+G#Ev>6J2+yLVO1sc#4XOKCiHOBbI63ppLBL(9VO%bkPrh!Za4RdfTJu1Yp6dF#L"
    "wZ4*iqg^YO97kD#91phnEK?9heYQ1a2=kV>z`J>DWOO6o`TK&r5)M*kx<Q;WI8YGM*I+snew3<U8;wccihI4EmgWJkLK(p=0"
    "=K2@KKVwvc96SZa)XRI=#JQzn`);Tok5QxlDMQH6LCc0l|)Xd@8cUee30&sK7nyHSHO}pR;FM^D<Gb*33?m$ykAOB4!=v0pb"
    "T<+gKhg<Y$w1&X?VejM!EGNr(g;pgo~L>;MrOQwjbM2YyvA^~=*IHB}3p)mihX8sq8h57MG<7kYND%)y1M_{x1(_1D8IdJmA"
    "hRQZ1K8ImzOW($D%Ur}g{Jo<wvaO%Mgnw$|_W1cN7ml=LJRQr(M@^j77TFiDvp9L0c_^%eghjPrdpNunJ`r#4R_%fT+4o!S@"
    "r*8f8YqX2F)c`rJp6oKlVB6^YZD508d{Uq5V;Q6(Yf~xUB=q9Td0=HrpbWY0Yk2J^0^t@SI(ggO?#z0U?&(UOT`S+(0Q8=X+"
    "80+QnsfTZcDD2+ucWhOE$M)Pwt<XGsqX#tS5BL9{z^;!N}&N7_LbIn^FDuB6hHjJjrzyNOM*ROrrr{y&Dna(%&r(94L&ua|1"
    "g@N&x~+y2K7s%cWIW~e4}5H*`jqh#C?7-d5!xGQg#1i0s)&k;pynM#}u`MqYtI*eV7?%pc#8Y`8_=?Ja;jv_1!1Zz}Kyt?3H"
    "B=8@U*8m$BDbe{F!Lxr*Y(_>hxGZOhM-bRpQi@~Q_zgl!hCv%jj2QbK96S2;AW$eiuxf$t(ae>a{opubZ;<JR$Z8jI&*_Nr_"
    "z&k;0lq99kXV>Rs_sr`kiIIoBb)lEr6{T!yryR_(^O2jNIy&&rLz3Xe~=W!#twqG951t>g><`MK<G*%ek<^&TT7^ug4tpB~?"
    "j|;!P>o@cSPUYQMx7zNt-sP9?z2sBp>)^Co?5VhucH9+GSa15s@}(aq2I%4QRrkUDxw82I-l*eqQ@hO@Ft@J5N2NRGGo5BpG"
    "7brHvDL<Cs1EjvQoR7g7nfp24=t3Sep|m`MM)gcp_nmpjVSHYme#ygNUT&gHu4g-wx?)MeTsi9Qs4FP0_b@2{r<yLQ+}X!Dw"
    "@7CoQ25?Kr`FS4rEu7vu;J$9)%-S387#@%KL-Tg3IxBzeI!Tlp6q#DJG%5Vxn+~X$Ys(Yc$pX*!fI;RqZ$&tQ*JczCbd%c|F"
    "4V!-J1yW<?O=(A5Xj6YFm#4;?nSfk}Gn?QG%Yo{?-$3{Yw4W@b0Eb3S>i(}D~i3XOg%8TyDieI-Y^arL~z<L9av3*EV0>1O-"
    "SeZ;%&?rf;l>{dG$x-37U@BmtUQ$QbBp9Py)q`XiAyW)-j8G6}+b?9R&`fFXPHE6_wqcb5F>%n4|u6u<E3GS1Y=bzZwAL~Wo"
    "Yk&=o6u#y<TzC2lg5VchH&6Aqr={9`cbMHgnKGcQOfg}F6jZWUJ~jw@S)OWot=C~&SH#{X+S<RYuQQJ2krg>)MR8lKc$wSvg"
    "O$}W{DsuN#5NED$HUmU2Y(b5o8Oa&zd7Bw%il{P4a|~VJL}9&<CNZp{S<~ldsf4!Xs-e&w#`Agx>#<P%ynF-*9jkaG){$hiv"
    "7wyV(qXR?^Ts;;@kNL?n-@^)4y?Th>D}J)c`v7O9pJ`<bD5kv6GQNMyJJn%aV5y59;k;j&X~$h%t?Q2jjSVb^KXvRJsnr#Z6"
    "i&w5_u7V_1^sLlM2k6pbww#*jII$Ce~e5)$>BNz+{j2haJ;lunui+GyB%(dZWBoVMf~3Vfg{I|JUa{nt0YwuuKEt=5n;zmdZ"
    "kFxk8P`KT*&Dd=x@H!O-1(#mfrq59kQu9R5`TeX4|Gz+k_t84Ib6(2xyp4-@s5E5vqca-ummzDunZp50NO&6>4C+O0Rmq8cW"
    "9$5Qazr>j7zlims!EfW_Cuvb`CHRH~X<Kh9)oh8A@e*y)<<one+hYR6xgtZ3uNhc8%ks2W&7EGY-QWh2Oe<@(nGFT8F~D?wz"
    "CUAVc!j}eiK7fysd$g_^HkqT1K{<m{|VPu8$Yj?_B0)J(cQScb<nmR2j_-dv35khe#I_SGAGTR!Ei;&9A3{0b6Gv>#oZ8S(1"
    ")8v$^M+e->brCpv@ut*UkHMPkU_w&7-q1Q!7X<`8*0n&I|F#h!p-N%%|}5`?HpZnJug9=+TRn`}^KMb@k?jKvJ_&=Vu2yzmX"
    "HX<01zHfSmVjPASuc{`{Ra2G|~t^9H)TSVd;iZ7`C0{7{m;4ic$gI)$Y180>ktXpWHb;Xd}U3SXSoC-tDiMtgZKt?L3?CrdT"
    "B4T8~(fRQ0oo0Y6OT1KA<OR7%i$o#boPp4VD)kx_+0v_YCxhu|fWcuS&!*Mp5*77Y7AR$AT0@XENjh-VpX+dGoTTX$^u9{0$"
    "lx_M%hVf=zcy}%Uv${iP0^%RfQBDlt^BCTS#7AE|ONGW$%-d*o62*%>uRc=ukIV_DQlZu;YnydJ<O<NM5|KOZNl%ou!eI79="
    "gx3EDIQ;SXx+L%n5$=Y-6S3Z5uRF`WVuy7O@R(Rp2QwEJS#sc&Q926lKCr)+zV$CtBhqo(#OI;U0(*!EILoavFskZ8gs5`Zh"
    "BC<q!h0Yud_est}VNY8Pz#JPm^aE?F!Kwlo9xTb71Obh?}+*uH$(Lcxu7B7rlka&!b<!QM<d|?&j`{xA7hHN*rO>>-@LYqPp"
    "xHwvOXm#^Alb>a&9VG`5eM2eoE>#R~?jBp?6ABor^naT0c?m)pY6OLIA!Z<p7~VUr{ZZjz0?ZjS9moNIc^E_Kcmw4xN=RB+{"
    "HVASJOuhbhv3+igMV(a6q2@FEO9)RHRy#9quclK$hvHzRmU$-9yt}9&$y{W?c;uwk1iKqU_dk9haoH}5}zB*kUoqSK_0zoI6"
    "3d4kp_aREuQS7hk3_IWo5{!y5u^fSKfB0d5tc>U?viJ=2CCB~>l^QsT-9lwcn8~xcAAj*lL3TevaBIH-`Pqxf8@%1Tp~}w(v"
    "K^NPLDVQYCB%ICcKI&&m%?_i8+kgiqN7pdcjk`5ektb?RfWs^GulW0EvMZy<bG>c#=@JD1qIx4Hdf)ToyTgk$Om)-@0?Z&b5"
    "~Tro&0IN&zJWSa@;!IpPyEr{vIFt_nRJ9wWNSO)T1QdF1Cr#-R$V|#pX#b+c-Ti*%Pe7+8{t#<<|kkP60XCcq!DJj}i{gS#t"
    ";we7$nt`Gs9LjeE#~$aay2Y1*?#Os>^YJ3QmWf>+Y&eP?|1otpMmdXzOOJTYp+oelQG(IhOFWHv@bZk7I#=`(~@9}Y9|VH+e"
    "-vw5=0syn<a_C_?+M`+Rqx<c^qrnTyH+PF;ZsB*51s&%lvVc6`~*e^UEkEo;VYU6f(K(a%#E%gcWG;@w;mBe1=>_9P|=pp`6"
    "mX21db&{2m=*X4X;k6jLs)YQWg!L!<paXBbc20~lI7!}Zg3}-U{}FZV+vfVw_E{QXbQyaLMPfvXE>ud9;*X?4gmO#e^6Y1S&"
    "pF%g3A%k+YtFUinhSew+gF7h4Swv$`bi#6YF$hxdA@;z)_2J=)H}>KT;sh5=Lxps*u{8zX}8w?ax$LYBDbo01kuik^MeI7v>"
    "`&z(n&-bL$qel`hjY<@yzH=Np@~G@7^}-_g+$k5!h{H!aeWJEOmX^{R8C3C2-&de(B*&sh36~2wx-F%D78en*}=9u014H4E$"
    "_N_Vg5e(8mkfhvl9<*3$sqGr<yHnE<w`x4WmH8i$`~@OwzITv;B6MS%>vlEWr{<^;e?={K1&NX<B4%Eo)L&($1faJl5yCYt("
    "{Y*rVWJL6HURnL-L-acP(F+6%}vcS{>^yb;AWNq9>B*XhNvL>F+YU<6NwLU9p`+Q)Z#n15%B{x#v_<H&wtz9}_vC?P#1dd^?"
    "ZQZy9P@T$~a&>P|bh|ekKKTbCG;x_27I|>k6;un-t0e-*7m2^<&A1!C)3Km&mCn(=Sp_4rTr$SfYKnq0dPc9wKhx?YDu%C)V"
    "^s>h1)M)Drjbtyc*BZoEjCYR=dYUSqm3UXPIM5bOe>%|v+#6lBY5F&T0Qta>{7%;(RNrM<=yccsZNgcv*ava;Wo#0?0(Yz8c"
    "cR_P{8xuF<QTfncAq6JD~Z3;tDGw=B^kFx!c=k*8HXL$Rv4`o*{@<X8VFT#|!b~ZFLhbzph_U3M}3Q>zMX|s6?I!RNho^^Qx"
    "Pax>xTz)MdDGJRT~SP$hP-w%))t&=#3XQUmf5HmW}7)4g5~S^707-z=Fjx!sw6jfCpqx`o>vGOA14`JcJhJ9!b{vJ3)IQ`jt"
    "!@7VpE@WrSBA~`KT1L#_;LZI6dw`l&nB9B_pqu~|0mF<$C*z#b}qqWj?7S>g6jW$ckeA*Z3XOh`&yYgN|AIuslW8fepr^6AZ"
    "lmY$JMjVd)U8>nZJD*YN1V&f6m<?|U>jU>ELHg~fDV5^MzWRcP4GtNz6<|<pPt{L-s7{|bzR2hU%$yJN+0C{4wYYo+t1Y2@G"
    "0XJq+h%M+zob9wC0aUqHn+$690*45%xe#az&3|3!fAxN#bjEWAx5vW(YOJ0mJ|&P$)p$ze<m&Z<5eEf!A$e7UCFa|m&P+^L1"
    "|$DoUTc(_j!ZSVm`MqTbxIeV(xcO1cFvj$m>NK@tD1d!B3scq~!VX7`k5gUp??*RNW!l0LTUa-0V)SJ*BMuW;ptGE253RW=6"
    "^SL?|@T{pD8(JvakAeK|hPT9%XOT|AbRyi6-ue~Dn-lTik{O?i-?#nS1h84qwhF7+$5zl|ExaQyTYj=2%yg{qH7VWLwfchYx"
    "xn*s6uBI$XLpI}-oZt>o}&7<p7zvcuKI%2C86y^ORA!K33*$@UFOk{Lf>^2wJvO>)uh^yg4H_ZH7ixh*n?c}v#sl={ods^2<"
    "uvwMVSy^q@8osjVcZRJ;xqToeyykCbL|$gylHSI>V31GkK?hTI?kZTn*RJ9?v3{5!<I@Zcg*%r3Ac`*d>F`f6Fc%c$03V=v*"
    "yw6l#XVH3y`+;y0mYMhuDjvS{&9Y4B&B%8QW=h&5@m3pKozEtXS=&XnvjQVNyH9pX8lS1fa|V&vCuCp&qdpa=!h5VFOs!zUc"
    "aPK$K9i+(V~_9TllyO91t)!TYaE?PR!B*4sgE#ZfOD^+as-CkGmw}*F`{nAKDE6+)ngP@6w8wz>Qja5;)}rFy9)dRRL%qLXd"
    "w}bma&Vb~WL=a=|M%*Sf#o!wI#{u1OldXs!Oa*#&ajnQzj+fZ}RV;mT_UiBGPcmQ^!&mhcE)Zi>#MysR~b6@JzkJEMXW<l6G"
    "<Grb)rRL3FA{bye>ju5Iabc5~a<6#}<)5&jc=E?&r8$QbZQtIzmJ?<@*Q?olfRAa@up5EpmeEuA^(H#mpy-yNP`OWKfSoW%#"
    "o4&u?G19?G7MH6!5@3tTaRhnXG@A1RVxiC|``lCqQgSXBfC+Zyjr<0nT#a8!i?Ay8<hYriri<aJHBgK6Hs#d_#7k>T(<#rGM"
    "@0UMtAYEnWws2-HS?*n5oUD2`oFwxP%m}v)Lp{b7fT7#UXvTE?b9(HhCn{=|BZ59<<meRjf`O9`=Pa{w$^n?Qa-odk$E1tI{"
    "kY*@?cdqA2pAKUiSQ#n?2^2n>q~{5$n#u!i?jg)a83O=xVf;>ENq5t_zi`xQf_h6uBNn65X{@zkKeO4M-WJ=I<--5xN~5y+d"
    "cvDaLfNX~KhPW`z8n^ewZjiMgo5^(os6UzOkRR2zm}kPe+np@Pq>lwI!MllfGpaV<gVvPF{R&Md;-`4(9TeA!-Q#(oHsjZ_^"
    "IM<Cqc%t37yU%Bs{h~Gl%Hh6Au=?2lg6<TkBoE-1_L+e&{bL1mliMvjdrK=fC=j(F?OU`Ka*0Sa!cHa%LUOrkA_<N}IwJpoP"
    "Aur_ElBSlbceYwpUW466T+hAjP^S3>r`2}|+1HN7Wq@9*j7E|znK<|62fbvJo1Qgqg5YD6jp62lDcu}pm+X1tKuI=a%&!slH"
    "w)WoUYjh>hQeE6^w^;gr}~U4%ot-_|Lcn=hy!v%gq>+6S;q+b&&3FQY^Jkq#o5;1s_}#e)~7~SrbvuG9ha{T%LUvcfcx0I&Y"
    "{O~(~6AJ*sY0spZ}3}b+3)>PX1xc+>o8Sw3}(U@>ilWM8`v~4UuMhJF7PtdPExZ+}GrER$s&2+MLfGpU1M-AmM7e-6JW5*K="
    "x-l;CZn(yup4ht(NY_9;c3MglBOkM1;rg=|tQA<6j4p^I<+8MLBK(yz`p7xXvbvRM66>rZkOy!(UmtIbUYb6_Z7oZYyNhiS*"
    "d#lfsp!)HUbGj}PHeg)cW;%4dtn?LO4b)O+ys|PeW#K*_7m*}-u()jZ|IE(DG(b&LCOIkF&d*cpyFSbhHUW17^QS=v;<)nGH"
    "6tw+8oOD;a4(QUoo|Jxp=W;MJmv3k|sLZK0P(>_3n!#n$oFq#JlP{vbG2%;l=pcVzs#jR@-RJBw<M&Bdc|Wh8d)IgOzc|Fa_"
    "nhCuuij8On~r`lGON6Jbmh*B_(=>qa(<9I$f8&Z%EGFj-GGafsAKX({L5Bv?cK7tB<Jw>t=sbCAhfWd*cqrS(VlKJum#%H&;"
    "iOg)?Y5Rc5b3oM0Lj{(QUAzlZ|UM1uw$%UD}qajSjG@p&v5kJ@hxrz|p*{vCrX>&?P#8ma<cP+eB&O$4Xsddg?d3OvqO!vF0"
    "nFGe_Eb(yWljXHCTgtIn|XU#Lob<(8awAs_Wt)a*W^&#Gm>1t-@?sJ>lDs%h#a#kD`%=Ap8~?zV%Q$6iCKvK}>+)z<0KrXE="
    "|K8(?7N$>YlSyZ>abtY!F>rmHZwH|(_*Cckww50ozn!Mog+B{X{1?Th{d~}O@iUWvR1dKSSySojG=~j1+g_Oif2&MOKViM2h"
    "U|jlKek!#T<;a14WPff}pB4fHtzYsG`n~GpB-mf^MZ~Q*+TWmi!w!fOQJv>`C!U^Kmvrjw8*BDR2Rrwb#sRc=32lji->X^Ul"
    "cU9OR=zR=ezuRh(oCXC>}E)Ti#>ZmO3%sQwA<d(=Pn(6D@4q^{hNujwlH>r-7o)I{>)d!%XIG8vp!;>MdvrsEMHISGQd}&`1"
    ">V_x;#%oV)yukBf`D;OE~C}OnB;bMh;|S2Fd55)>%{NS7U8^i>oqP-22yfVzqEwP15Z38FuTa9&^OHcA)os+bcwp21Ng3&9U"
    "vIi(MK^y;qYxl#cglnkT+u0sVW64Wr6-fk)`<IJs}Y+m5WQtXuF`gJQByacO}4oKttrO?9_$)~EGrdLVy8=@t1jE4P>SbNNQ"
    "Qiq>z+h`q%gCBn#S%Mpqo!CmT9#1$siK0@{??Q^?JLWkOva>35|WS2JG@_oBIj23NSxzBvB_3cY9yAcp^G|dvMGNM?B_iEES"
    "vjIj1LV&d&IWmS@x)_vx>@)Cv{gZ<<5l$xxUUS36hB9cMlf=sgc}ebj*If6T?EbWausZb$vcqTsrvd6_+({fZsS;g6o*{9&C"
    "qv9|$7Dt`nVD{1=QuG3<9_<&Go}6-6Mq8qO%xbb=r$Ae7QzTWR9ePwY66>K+6L!hFC5?;vA95%+&xMwyVZo%{o;|Y10`AX)O"
    "PzecW)y<v7jyUKGktL;jgCDxebwVt38odb9}^Tf9meU)1-T{=u}GNQCOBjda6~pO^KcH+Ur`bkhgjh#+7l9FXr_~f~ZlkQ?7"
    "c@#w&Scu2kP^f>NZ)rO_#0uQi;^_dZs=#Byc*GNQZIaTUV<o0)j*lrRzI@Rl2mChV&pjWM?dw|-jpk;}-V$9E9YE#pKxG?Jj"
    "&p5dRSsn@q|ur8iO-(+j6QE@(&q&a<D4zO7wf>Xht1BdI>U!6;{X1Cus@|bg}?t6zMn)OJ+ZPCtVZJmpbKN{Zn>$zH;+27@z"
    "yS4TicTerM$(z~f?k|4|p(M1|yWMTFyK;^GQ7TsIFD9uh60+~4MMb;cNUd!J?0b^YWt4@lLC@D4?@~9U;WgbTPh0Xd4Nenz*"
    "S%o3Lj4PwRV`_kr`d87-FpqJ(KKx!vA3=FLf%v}$68e+mr5qa%BoI+b*~C<0Q~Gup304{E`!ysX)KTCwgH}2`-L(Q9;-!`fL"
    "bV~aqZrxAymugzgBDIH@YU7m3oXqwuE4|njhp*jY|W1vW(|<^?5r>jrzN-V|ah_nz!z@MmCPhb+q3T^SLZmT7<kjl*iai{e@"
    "@onogwNW*1JKpPu;jiYlIZW$dAoed>Hn)oKM>#wJ3|?RTKj9PSncf9TpqKwjE|hcr;vfmXCSkP>a6%|x#dM@6+LtgNjeR(|_"
    "8j$HNIW8lZ>fMCXtg!PpF8OMKiLdzsQ)92D%e!hF#cy*eE`#~&jkAryJH^-f<kCuo*S+U<fv9@DjNl=L#kL>G}gg@1#F~<w<"
    "dJida6fEwS-fQ`?XDP8m7fh8}#X^Vo#J@}Fl?oLaseyympl5CE_Ek6%i|fQ?aoXuUUkLVW?^heyHv+H!)~lOUyNf-?79(6oD"
    "p+q^A$)S>@$RrRQ1BHqXYy$NI5L2^CtR!Jn)Vg@)45B(Xzm@EcL&(eGemL2Nu8<9yPVSmpN3f!>W*%0#Y}kU(q3~H5A}7bvZ"
    "KLteD)iC^9HAv9~HWOO|=vxv7_;LVlVqE&5P^93`Onb7NYmB#o^?m)$wEwV?X7(Y)MM9oW2(ospe^i=T|$dO&*n3$}=CSER+"
    "?6&0hQCqVf!)^|*M%?LHPjMys+ttR`?C@Qu}V1U;*Zquq*Gl0wA_1!TlL@3`;xpxul16smRnUly0w{E<2tw-B_m{WAgDtR~B"
    "2(H&QDr`(SJ>1a0h?quD(2ux+x%<NUfRoRUV1qyj+d0MiKmvTMyPHHWqsqIGmKD9)VXZg_s<f}zR(MoBi)ylDGU#PAzJMSex"
    "xB#Way#~<G+680XdS=&3QO}z}dsi9l?CC*y!f$wii}w=U1$KJ-)j$9)Zh^H*R=iq^m*@Nm3ZvcUbw)K5A6$$Yt3pdf@>hXTv"
    "xWT{o)2I4Yd6@qjV6fnH)y@{X!+Z&BJF);d!j02Q^goN-7_csll0ihc+1+Q@%ke3kJ|6=6WO=(GyC&8{Q238-DvfGdCke-b`"
    "v12D9=p8P(Jg2&}9_Nc3q%h7(JEWR9BM$+nF7YVkgMX^cL7UWEZ=$^VxbtAJt$kk=MiQ$Lk@w2~Moa@r@ru{p*ew)baD;><0"
    "^q$d=YB1G_%V4ky`aIi%!U!;uK@{&<AfMoE5d$6=%PHh&}RYh+ApV&06+amB4FMoEO^&7?2PvZi$R=CwWg8mM7)n!g>Q^d}x"
    "I)3o`%c80T>i1vF^i(A6a`cOGME}P=IeVmSnQNba<RNpT1%jOrqZIE&8H)RSpjPCljy%YRL_dTu#0P<5B4q?Tb-dJ`VzP~1p"
    "@F`uW=d0-=86?!pJ+r7h+TRs6Q?OKNHG}#n^L87fS<9aHru&I(wXr>wrf6d_JV}sgXGB<<+`w#kpH7=n9oY|alaG|K<1}RqV"
    "fw^;Hf?dY+o|G0s_>mdI^kSVzry*jKQEQmC}#lo1xq}G@T}fb?^bD!`nBJ%Q7SHzpbdAaYu<WcWZ4;!<F^9shq!~zyYKT#`A"
    "xvB64BgUA+iXh5q54@pN{n?%+jBYNbth)gPn{}h|%3_MbVv!*7=R!0a0$Bij$ALOaiamhx?#&QkwwLT<|;`5AnxT(wCZx`N!"
    "NMn;rWgmD|%#A#-=5eFrsWv0GgF3%|{uzjE!=(g(LL{cX2F+UC$Isn7IyS=dmvY}oJJfxp4hp;Yf^BbD7r_#DXWqS9KAuce&"
    "Zy?+7q$Pd@uUZww-L6y;wzh#iQ1z*}_SF260OI#GMzZrK7-m$sZKK!YmCSxop4Sy;4FTXbqgABfsxk=rBs2OSxPg#*$?I8@`"
    "H_M-&#gF5@Z!;S}W|aj|#c`hAk6P)oVR&cT`QE%8W{IyR@1uwh_VWk+i+P-Lb1AnEWKFw6r}PMUTa9tck30zaVXt?`*)`t<("
    "?~#Ibuv^y#5l@3kMU&(hWNaS4%~=<8n3AKn<gZCpcsFHMd09bY?XVQ((jzMCi7Zn2YxYlcP`iyJM%GT_N&gd+PzEn!Jwx56D"
    "yv|vZ_06@zQu89E;4LbZ*8HUY?Rrz{|9{I@>pP>H1umMCXlIIj_GyKI_3UWWHalRFPneI@CtODA5-=I%F1GvBtU2b{VOamyH"
    "=cu4jGNPMD~2)1wL6jZ7&nI|+9<*37+0jh0+r`8Yt+AAWYV+)Hjt#x7CjMqdTJ$DN=536jnlB*@jnbMEAt29|k=rsQ#&aPBk"
    "_0rd4avLj+d%#h1nNaU>dN;>j1AGDH;C^tVd?q~U^?Qm;h{i8jiX)WMqnDXH5LG{sy?(U_RzWS@BTfb18%L=T!w0pz#Xxo%s"
    "<r1~zPEB1kpiS0gBbpsuN9Lw`a*yZP*m{5aT&p|0=r=j6HX41f@yZCr=`TMhQFU}U;L^KRa~4X`D8)%=X=hUR=4dj02zHm*u"
    "=f4yQ%uZ+>H3wHICTA7UEAI+*?x%VaXA<g2PQ|;xE@SzE_j_POC!+{T$K>u>yz7MV%ieLvA=yDz^U7K9wB*m<M5r#f4sb(Hq"
    "hP<9|o5EMt13u)d3@YorbsKRIQmFQWW+%><vMmGL>}QjMHsT_j^xElZKmUWcpSb-VO9`<Il{phstrQt?>0zUDZ46=_dy>O)>"
    "N62pRP;?9upc4O=F{c{KVSU&{6Pj4t^EP@0r;I;s$ecOK-Y<h;hYUNH}^SyAnJ=a-io<?qBHQpwXrQx+%G#6pbcg6$%e`RA-"
    "ec*o+fqcNQsk7U@I>k17_qHCs_fPu779u=eaq}b}`k}xQayOOz4kE2dL2}c!UWDQTfU^$@L4{1MG|4Hm3d(5A0>9EnH9mu|l"
    "9$wp8_4VGq+^9$7;N@9y?bc!Iq(2?a>AaLj`|@gzp2{*>4Te)IUhzDuEBR08OWF~B2aVqd=QBvhZc~eR4{x5Azkd1UxnR(8m"
    "pNTRk2+g^xl0%4NUAs1Ra_CK<ZyL<1T1T9hTWW7Ib|xlmD)$4A0&_Ap>uiE4%2euInQJqU*>Lopx5Fy$1OKqIXN#{JkL;y_h"
    "&~i+Y!zou9&B`N(<?M8dgJiMHBWp4x`r;y}<UfFeSzwS+`bS2NbZes~8`n0y`SAeX?Ld2sIlY`aRI&a{tpEPz|yz8lN)Hbu8"
    "Q_?*1S1IKDmkjK6<>(q22+WZMOFNFgh_Rf$6&9H<ZKYJiY6UTPDbo>;-FmDl)*+?+Er0d=o5py+WXdAB`L+ziP4v{$XuAe-F"
    "WWLm?J)O0^*|8Ku$pUHQ~ZtVcfsQr^I%arZJn}EIbpCOg2{TIc8$wv3+7qtkN&uVqem~dQ<f`<GHQMw@33l;S~9-heTHxqDP"
    "eRyH?t#Iepyq|t2H;CqK9OK^Gfr+n+0g}$S&M#kCBAsj+M2T1?OIh>947jo{*fsS%P=;mdB4#Uau2tpwbbi=}DP5mnNv4l4t"
    "IzZNjzU$O`z=N8v;f0jUXK0_=fKAi2`-d{@BY|%Vr%_7UKq1V@XPuwzll)ZsXi)`{Le96F9N1pZ(alFOOy`OqeyF`Dze|cKy"
    "6lmwu1)vsl8@bErqagI!vbmGb$(@<&H~@tK9EwkcWOWb^7-&c<Fb$<paOop3Tqhtj;mDW$MSU^=wXo{&Bc@diP`(<U>juPaE"
    "5#9An!8JzJ>@R>Nd<9NTwyBNgp=7nR2&Jry|x5E=m1<zpDV)@~%Ymc2hnsOe_VD+RppxIOF|)C8{#(ydz*+!WEuid9-*ld=0"
    "7dqC1u)slgOv;gux@R)UO`}$9g&X(r5`G(Q{xtQ|b-tW86OT$lZw|;8fddhNGtK22`rADiZiV36x-Gv!WX2$yGal+h}uFpe-"
    "+z%VT2Oq@%g`N@CEv=;?yPR~R>1F+q!2bC1oTsQal7Tu2kNx?dAz8~!$591B11eqK=9WC}-1v07TYYXTQ4JIATt<uS=P5#2K"
    "80tU3fcOsgXA{Ka7Y&FA8Ld5cw%DjyVYgeS>@$r>1ggiQ3;Zuwb?W;F4_AlxSC$d)=s0|kdR2os1_ww$!hO&4xsjYe7Yv8)9"
    "kNKt=B*AX#fYE+iYh)t?AhsF=XxH`Qgr6!<&Oq<xZ!;ILK`kWKQqTyGYx{Tm0vYjQ?FqzCGui=jug{uc5_Y^&hE!8EZDb^L6"
    "(L4=;#P0)y8zi3hHcf16aL%Uq}|YE6S%)H$cSh%byGo;)-4yZ&^GtlVjwpz~p`+KU+{wBNPEbG>Ogw#BjIr(6}}a$s~{J?m5"
    "sl%@Qg0O|X;HCdgm<2tYIm(@owIs(&f??N=Y6*lPI2<4}I*ybm$#`EiA<Jp9>61A7HaJ3#7r*$EO_yQ_7*Oi^;%b%g>yIxCO"
    "p6hk;YyE?RFGstaF4=ZsX+Ig>?BI)AWk9o12%i$x3d0)r_9K5lhNre+RRFHt5$E9|M8^5?jQw<xYcX2hHa*{fsWCb{6GvBcf"
    "tG`PTj<BLjVar6>V0Z5x^D@(x{?!GhV)lE{%MnJy*8U0(_5M?%KA;3T*-LUDeUn2Dtnj6)L?AW<jA4jzX4zATW8VG7DOb4rT"
    "Y1@o#9@qzunbF>(XkuPdsbbw#ux|Fr2lU9h6s$G^dL-9zD3)=v)~z$91r{6wR!D=tm%{yr%1%9o#NTQ|=N4MhBzQcRS?E(pr"
    "eb^B!IdhURY!t&p?CzMlufBt(QNfF3TAZkc9M%Z0r&D%UGGiU;8Kk)LtFyd=mkd%RDpK_l-!hbd}z&cI?ke0`teWzKqlL9~%"
    "p<ujA2(DhRugz)agbRGG&1Rx`69m`~9Ouh_=5rG9a?ACiuD1$Bxcd9Sl%kpm4OPne}7-bIywzXdv$DdL~Rh9keW#pgSwP*bN"
    "-m%&KTX95WZ#H|&(&4`$ORkX&HvVRN6jd}>^-qY@-+{!+so2?5+n0CQK}4pT*%4;lIX$>^ObHXtTEBZi^Bw?ZEc~ih=BsmlZ"
    "&0GN&o-@ZQCU89wCX<e&AAGoeQg$G4?mA_S_OS&7#ovd(r82KF-g1KWJ@L4{g)3uDSg^HiHb8^B&`uH9Lay96Px<-{l$7R&4"
    "t5x|3b=*S>G0<W-j#u*=e-HEmj%aRrvOq=HlX)+VoK!>?at{V;ytmKBE7!*-Bdp$Ir7;91x*bIbPo>^GU2r{qdQV`?rl|Rk}"
    "uVHLLlp^I6zfrxdKp=DV{Do*p4HLn3Zs^1c%>$x^AM1yb*$OS&!GntMH4i0WP4B7X(o^Ac#Y<@-9Y(qDpn-!Bl}-=UwyNq@b"
    "LU{zc>Gb;g+JJW>23P9z!t=%z9Sx$4w?=?TDNPFxEl8CsAaLE-6zeV<x;Mmx#KkFH!o_AxWE!0G;*m@9OU>7D8em(~pTir5{"
    "c-KCY|92NITfJWBNytX%(4^vyO}nVmh1xXDgOvECYjq7}aqp`Uu7RB!O5u%%hHBQM%pMhE6rD++S?i<s4F;a9PrWkU21v<6G"
    "N#jsuBST9H9Wpn-XBlbbE#kX0JG733m^Kaf+CN}a&`(!Jsd)&#Zx@#%Jpv$an4=W$!e{U%MrP4yLZL!g~A&ifPLeJkKW<gJd"
    "m(1{Jy=TV}pZiR2{dUZB_IAYw!)n$NLzyPUGWj?nUzOaA;wo4s`m)aS3ayT8$N*pt?K!jOeu0QD35~lxu5wMUY6}KQacuysx"
    "v)hhq(XOpTFj0jx)QEm!1o)+I>q+eQuUtGNbT;YDw~Ct`0he`+Hmn`rueU0<!mnbK1rXXQffYW<eQV)euR#O7m=ZQ7Z17cm;"
    "_2~#3x)q<>U3!q^W&Wg*=zXY}s$kK4();g0v{~r39aDv)g7bopuE%{x$x-a{k^vgyUy;?f4Pe(3#SNAZ9Z@uNVFJ<ZjUVuXk"
    "+9nDFtvJu)l-qhWCWErX)6XYTuD&kFU}?pVx8irM-BofafoIp8?W5}3;}@X3S$-82Hshc)5G$*J@BeK4A1{;WmmrE~)#*0|C"
    "Z!#FyS|rpob%oG3+Kmwx4y(IGSVF*S{?4D+oGZy^@AUfR^5hAd%xY9Ku^wbKi8twn5_9ztAxI8H9TwHdJpVaeo2hAX#xX0Qf"
    "@o@vGPoU=+g{Bli;%MmF7#+0Tes_o7AY1;pSSa^mxEy-F}XJ9SNDRI*V`7>Ae>#slBHG|9&^E=iqh*X0L|U9^)n?_S+l&Mj="
    "7Eu5Px`a~A2Fc@VG)bZfp0as%F!iKH1kT{L>c*^p`u><yq&M^(4wcNSWI-AYbrOUoH6zRP5B88vmyDJu|Gjvq^+@pjOOS+tl"
    "EcttLigNro8%HrYn3(Gnr_vO3YZ`;d$?ZEZIo}s;3te;(bT3$a_Sex*x75$X1`pZ8zfn861wqwDq{wynr>*Aqip-M7H?#1p&"
    "sE<Llv$-l0j_QmOVJ~c@_Nj4tuGUa@Nh0c%JD}>uV9EIcX*7F|TKU<^r(M3Hm3a02@JWSqIF^XlvpBjVC98J+TyFt96EvzcH"
    "#q)Oa7ZQmmj5_$Fu|!npQ6~kXq`&%kinX(#_%_&Eyd+N6>Ievq*Na7t~$~7A_sk?6%6Yeopg~UYj&;y_;@X(>h*co!WIvN_l"
    "`clZ7U^WeoAhY>H_%dDvfKlSxaOw9FOp}gLbOg%z8nWtXbB&KMy9qN5-OO)wz3R7q~NjIj`T=LGh~3frcbVQ~<n@A=5eHFjv"
    "d?>EI`h*e(ccKi_A0L|sO@Ew`IRh}PkzAEk>9{F2UK{w3%r2vl$V)_%|1J7+@fCMByvyjxF5ks51us^ZY22R^6tAO+NRn2*i"
    "<3QeY?n<1xxRrWLXY`5(F93W`xDuc(C>c7)n^~~OGn4@ihzQLG(jXi}A##6~7@co+VGXK@)GiK9)$gv1p(iA}WU0!=1n9T|p"
    "L4-H&LL%x~lY4w>mlIQhmREXI!RgMkqagX*8LgDfmhM?@bD9ksX=T<_;YQQ>=I))joWPC7#u;DbAaAb(Kc&LeXS4k=p_=bHu"
    "d2y4gXlz&Mj^C0|L63H%io2z-bHcAJAjYz$pu=bc+HwdkRE6~U^OhEuEN)xhe&|;I%*HI<d?0Db$v=V$}vYgf$h0W(H)<Ex+"
    "gKvCp9N7xPjU&x?o#vu@*{pDn>#5Z8JYF5*Nf{4R#o&=q7Kgi@1BnI<~mZ<IxGcboDZ#rw8n6NJi0jeJYJ6N=hq)Op?`<`()"
    "FZa!Qj^+_s52O|09_dJPf(jMeIO|K=38hxEq~Qi02tpT(&5T3q_tlsiBA(%bESXsf{66B|Xd#plw%!kaE<^_&;4Tmwbo>RD`"
    "$u*dlkDl)Oo&f09@CEDO9;yvjN9e6h^4XC(QHEi%|{MF+cytn^UCJC9lBL*8@Q3lmklaKd!u|Ldzo{OEhIptWKu5b83dwCtk"
    "Eui$)CnhF$ibW4<h=;|-uXpPo!QdrU+lgHQ`M^_#SUeLdZgkSx+g8t;&LNEA%H+7CddalDQ4Fay-W+xzH=S4iG@!JCUI#$$F"
    "vm=fi7R_|+d#bC*sXU{pcsToxNw1Z_I7@m=BQ>rk&>y_aq4-oJ-FXqzK%GM0gGKLWIAr?K~>i~d8kJ&ju(fyIA>@v;17REZy"
    "=OAzp%A4&rg-<3?2dVP9)qPfy{44jaK387Wz}FTd$rx_DUB!NiWbvz)QM)Aa5``Jx9%0ydQ%dl1-h^ePtx~Y7j-lKe|O3rzP"
    "hmY4~5zI`dcaKr>+0n>4zyx2Qxe4M)f7g?&Hkn?o|1T3f%r-&Q<&LOs}pjR@t7D{Qtt&1v<HHTQqAv$Mo{?!#P`2C3g_=Hx?"
    "g&Oj+NA?F~|e+8+tGoB~%R)We;Db*Y`Cb=gmjf(7Yc)t3LE^Q?>+7wXwan)xaP`L<Or4yHq8sB<?GSd~<qsKN5)tZfTJwdrF"
    "no+w%IA6WweutL~#z{-cqb1-Ie`Bp{;Jy(s+$P&y_TNQY$Hu*dvS(c1*MvuQ*zmvif$ZD?&~9Gpub{GQ;<ZLd@3;fK+dA@r$"
    "F@d$wIXX+bN-A6S(q&Yx-?%G`EXJv_nY48vanGk_sY$JsG|pU-2OW=uf|qZsdkF|f-7?8E73k|X(cEp1y4yLZQhiv!Eo9j$F"
    "s$WSzX3&eA~LU(a6c1G5nd1{jBcCg7$(e?U4VzaOgRkP3G_O>`<baeM#h!_utzs4hOS1j;_Gnh3}Nu{x{urwr+y5NGW)pH(E"
    "&MLi!54fSi2^H9_ARCojr$id+ahiWIA`vssL7)wh$gkz>n8e@ac~LH2XJSNHQ_M_7=9zfq>o=_T5s=zfr)Y0x{$<O4O?ZC~1"
    "n96e^a!viI!DQJuO4Xj4t=2EJb0XaR5Gi>$7=xTImyV+G|yfS`kjIYh$0nDg#e=0N#^?o~MQHuXnTI%ino5ZTlVk&AQ;AbB>"
    "i_veyaOy3%&hY)oh<`-1@JUNMnCUE@RLfwDhXkio(M=VMG0V;*Zq$5lbed&Jv_GwF6R|Q+y7j#Ay!e?7x7MG~%E2dHP+Ae3-"
    "@hJ4HQg1MUb8_z(dI`8C#PXo?kl}kUpq&BoP~Jfd+x4tW~v8Cv+<incb&FwE=SE~-{<Mo&KpuAIfK>t5#)VJ&zTujChp#Od&"
    "YL9fb<-7f&+;vmnmz_*TxUSX`_uU_h{1%ndld@fq#LkSHk@35No8k7(MjP@YX%ePwP`0Jr+N6*2q`y9gda=%%&2NeLU~4)%e"
    "uf@fO>$0#sAar?xkuACq2tF|+i8ZKNHfaXceno$Z=@bGw4y1VzzDSQ-EANuMXfqDorq^dd0%+rjKkX<8gS4*NaMCyD_^yyB{"
    "RU+ykWHDZCl<YC+$+K_9n=8rr2h^vh#K9fKE*eJW-PD!%TW9Hbe`UWdo+RkenO9zf>MeV_|T0_+>{kWU%^uC+q=2R_5%eEmB"
    ">t5{GlRn)AnssKpDc!=%Ft64Lt5o)_XM5>iC(+ZvTIYS0tl;-Q<AIu4sMjRx-5I%}apT<;Z@KwlN9aU1&WpD)f7w;tdHx2x*"
    "I{|_)YNF@@qn*|tWeel*?67d)}~@Y>ttDg;eLW}h%Q}@qcc&-^A)y??&_bUgJ_p(I=HbWDSey$<ac&xOra5W8KGjM+sDX-*r"
    "Op(vv;dLrVi>AKc?Tgaa!LljXFdeTY=E8k0t5C(das>Y)<gVfFJu6FTjtEXQuF0wEragQi7+~VZGcP2c_E8Pj?fY(MQ8)pXA"
    "5WuDj}wKor5|dV2yE=ZXPx8khpy;vSMmh87OcMQ8VzzPQ~WaZ$4IulQIgmYGw8Qj9F;V?%SpJAQiC@6#tq54Nbcr)qlK+C!!"
    "C1aLM5+_1<-cRGE38DU;ZdJ8{hmPV8p(5<@82?}hSW#**m+$PT__{N@|NZhy`?V2%~Uc)x2ExLF9TYnCB(5pJ{X+z?{v~vh{"
    "3|R44)$8W|1h)1QvXq=AGboBzvmoz+m3(fDG`4&<`mdt;f&_<s`Ir|?SL<&LaSbnd&s!<K?|pwrcx8OOJ~C3K{j3;``OzcnF"
    "Ojb7!QH{+(w!~JwVlySzW{IqF$(?SZOEv{5ugLe!QAU66HRC5y}%VvFdXP<j<c7|JT5D|_DwHj-5w8x^1NIUyc|@Y+G8n<G4"
    "TL+8@iocSNai!iHBAgjXl*pozxjSM+RrKFB^;82HvfJ!WFCGGvcS-&-gS>B|!hWN-g`_{o>W{SI?n(DTHvXyzkG7F}?JXchV"
    "?MbSh=%8u`&~h216zO}_`d5AyJtYT^a-Kyi0~8>;i?HeHNn=f_pCi$c~@>K9ZxONlPIGW-5J*7fT3LK1do^l#m|_Dik09cXE"
    "-H)0OXbR``Rr?;ySb4;<21_?ARkzzbnR?WDJbxrY~KlL2|YC$wBYdwp>oLPc6qf7GxJEad_njl84@#i#C7RV+#IbK^Y(Ysds"
    "iOB@!BmoS##Pd`h{jB81;UOd3WQpF`xiNWI?Z@~sj>-klb4<#fUJfy_p1FhhOAO5W&eVQ%?Qn~E`>0C&mYPEGpDE1mK?d6J{"
    "7I`yBOi)|&@D&Xu_Qa5$iFyDZBJj7%LtrqsGR+%eGAAQ#z-kC<vVuW`18sTIqJbHfl3^eydnQ|lKL-4b%vb5p?izG>iO{xTZ"
    "!r7GDF#6HkFiqC0bAG`lzguYZrm_XIFc%(lz0A9w=U`Qjtk|rZrx3AZqPruPHWNZg!(>TWsH-o43u8(|XT=J}<>{%4s`$r*e"
    "MeS^lq6?ipaEs+R~<T8`%_t?t?kKDwVPzd4ESKF4x+5|X)|KaCk$np8$G4;~5H>)jsj3rPY%%eEXxt$9)G>O;5L`q-PePfk0"
    "Wp{#6ef-)Q2`Ro6-g{bqs6|a*s+w^ZF3J%Opo+jr3g*Rq#Y&AN$p@L5W`qH!r0yc6d)SbvXAGBrrqjU|)In<kBJVl{Hug^bv"
    "p*YUM4qe^vTG}DGHkTwS^VeFoH5ll_o*;L%TzHrueNJ<7Izz(QYBIUGK*1inHGd)eecdq^B9SH|o_<<!<w0~jH-k<Ks5DWHD"
    "q`$jpKaG@>NKW~@P32*^~tW00-0gA;obg9P=vI7NLqt3RZDqu!>zXREtbHrJFj)718b(QIwo~cSi;r2HDsHe&2qVD_Uw9PL1"
    "h?BnYv$UpKh6iF~Yr@$l;>M8^FlES0zr_%jx8D{Uf#s|4K-Hr?m=o(Qa>Bqh7K}IuTH&24`=2?(EUpdwD-aoiN%7Y>DG^hPr"
    "jO@&Ke#T=748L8!FF!JSU_?&T%Fmnk<mZ+MS8ATa$ce)hu7pTME%>rd1q#h+hra<Dp^RWzQZ&H*^-jd!0CEAIK#g6vjmM6C;"
    "Rqc(2&F7$%+enzY#H6OFhnaY_nKyEK13?>|B+`Jc#^OZ)mhUt7_`V9b+PE>eWP8voX{~7*UaIL8dwlSaAR*W#G+1`VFDenKw"
    "y?V8|X;6d8ZX`6+CED+N&44=(Qk43=(M0OCJD=?&KWYZv-uwLxnEOWq1FcqZpeE;&K~y!lcMh)~^>!Ovk$tp=P^0o8v5Lkf_"
    "iRLcZ?H5LjF~FWw$nimPK}XUkAw8ni(Zv~E_=0(vrr2OUfJV?F5b>O8%C|#HQJL_cD(Q8(h~zWf{WATS?|${&d~dM0eLAJYq"
    "Q|d5pQM<{|x@*vHin(b_}XuMcgv8@H21J=7Y-U)$BGw<VAUDNM-59mcEmWQsac5o!(7+3Ee?6iwlowWS>|{M^U{h{~qX;()B"
    "bUKmjfqZlNvw`*Uv}*bJ&B=X>o(aeHkIQg6+meT_l4hT)t{RnJei5N?M1wMI;jE>Bhzpn-^M<W2G?%uDzk_c`45kHaHx5=$;"
    "=SCDCjs%<8LYscZ}jh2E7Q3(*DeX%dCU9Ed4^iy5hRHNl@#6}-Lu`6pupY;u^R2bf$Go~rOS<Qtb2M(lR37m9J`AW-XCDLzq"
    "D~xN|8&SVJ%T5S2^g4r&<F54FL10AEqZrDW6F2Ipv%NdoXh<iSY)K=Yy*NF8cZOC|S01POVEI>*h|eEC<bMg{W>v9|u1}WFH"
    "*5+Gy@!}w_3Cl_%QX6Z)QhY!SZVCaRmOX)?QPo!*nI=NqnL|RE0Omv15i-*fr=iZ0S}%67y<`mou_5(`S(;bu-qzrVcA3c6B"
    "hNATHY`w2(f2htKT?uaQiAE?02wL?YDeF#&&f51b?<hc)c+E{?xqzuf;E{x~!GwWm`QIt*MyiU`K!!Ijq!p4X8Dm(bqN$>l~"
    "Zp$$;49_>Pfy|0hmr{(Og7Q|{Hw3J>hm{U2blz)<vhgVyzzfg0vCGh`(<;l?98s`|?Kyh_4mH0Y~KtEAQw;=7ZkF+!Xb<L>1"
    "pft#Z-cid1-sC|~8M#}`YjqEVF?1^`q-mba{wO?&A-&**+S|;`fc=f(IvlxCIit7(FtfY&Kk$c1Ss_&3is~H_Y*J+?z{Vyat"
    "opBYqJ^A?o(C-Jz^@dlTjfcZk55A<w3ixPx`@v#^Uz*En<(CF!W&E`3*p2l=V{3F1K8n4~tjI^+=QVz=9zA(#6<nA>q(FmqD"
    "Us{O>W>3A;wmnRXKeYD<_A?xT}F6Ixn0l3;;z00s1bUr`@H~m)D!;mcp{%XxYVDhQ7Sd3@$nIrpi5V6t=Ao-nbeL0>Ou^YJ5"
    "KH&Kf5z5FewDD{}vowg1d}B>rAksS^va}#}c<T-b(-s1AcL>8jdjImwoJg$huy`aadn*E+RHdpAwO%2(B?j{V=Qf^+v#uWKQ"
    "k4ZaMwT$VideuO7T`)GPX@t|5*HuWWebJWNh3UOJ>TZzHQ1++JC-dT~xdURA|vr4R^x|4I|AhF-ajNOmi=U46aE@5lKr`4%Q"
    "^=LK{RNxA&KENjXvuD#OBxf*HrCXu48EB8u?Z|`fOt*~iw+KXZ5vjHh)%(U|7+9^G&D7z#~P43d4E~=lYYYsLB;{GWQc9iGN"
    "wAG5#CSZ75ocP)0X)ZZwIx2gq8X4Oq<rH(SN_VFQMEifcHZY_^_l!oj3QyMc#Pq3B*!|_4+t#?jUV6u4hrw<$^=KjQEP(ugo"
    "_{xEV*!A@_gyv%v82~+#nDAL0>-kU#HDDm=o#qlxh1UMn_q3P6S?QqWyid<7TGc?otGbO+G%rpNyXmlD}L(xrMan1pZ=_zNN"
    "w(2ZhJ&Iwf+W+v5r}*{D{b;**yfoF_`=`sU_9A^vcy0I@+8c(?H9>>Jz?6p3%bEmsZoE)(Cv3fA%*j)0Trz=$7S?fEZQ0=Vh"
    "Dg9j^w!zBj1GPoWOx--b=xFWqA&e&&xd^y!A`V;cJDYJmgwi3+F~Zgr4Kq&@CcSj>9$FW<pOE`Ap@=W?vhXxS9Q#@-lAIGDp"
    "OdA^4*aDu)e(JO7kcH67Ua0jT?!&TH@f;VIsj5+2A?;uOD#pvV6wFrKc@Ry7oe+kl}=M4d1Cfxo;4(IsQsl@QR2py>BRUCWq"
    "Z5?de>}}$QL<M`Ck7BwBr`tTU@3g#Vt)%RJRK5D?m`K6`=BDd485@$`r1qy8{ZvHW(pK@jWXZ$V14P$(KC;G5Y|n(PKFa4!d"
    "op->dIq_FjjjFjtLE5cFZ&+09%W)McdW~xcvHq@%FTX(GZxQ_Nvs~R|Me;M8do=$(&0)_reQgz1(Hvr_DXa!&~eJo>|^1cSM"
    "Tfi1kI$Gyi1ooi=OtGK=`xAz)C=JuRQ&A<25Ms<GIuE=cNSdfGK@<)Lwmr)+_(#Q{1y_Kef%qQQ0c*&DA+vfwrl}c8Nya$;&"
    "-~ZLex;`diIjSnb2~%roxQ_Z@Hk8%}1B>{ZQ601o@~ZQFd55eqq)VnV#M(Q7-%n8PflxA#2{NE>X0ItOWgzy3h@ktBlqJeho"
    "KSo;vr<hj_n(fzASp&(<u+Gxzt6@y+@IDK|9GU<0QB3z9g%wu9!zm7QfO$g^nXc6vhT9Pw{sZPp`Ep~V(&lNkj5zT8=Ud^vf"
    "beA(XyZAg(pN>=MSCsZxWkhcJ%{iye(=diYa*(XIP-wl%LY$e)m-RGUX)MgLD{~J5h4dD`$o7iJQd@d$blXQXrhXr>kEi?q2"
    "X(ViU0cFlm}A!XfZ^k4zqalLlQ;IEL=wj@pgOnKpIS62bt5PJ;TZtQYxJniChkKW+E(WP&Q$vB_U&6~+c^~b?y9N$vnV!ts="
    "ni5W1_?Vt*ZLi>Yi)0p|s8%X{|iGuJC1SBAfqd?dqG-R=W8Ae2O)83Rt0lh*;(J0Sw}miUnHV#^x@`2BIOGZZ=TrwV(YtyGb"
    "BGw0EB8cBVtv^E+qH`#Eviv8+`)?ZKk8Hs%j)cJlqa`lkJEZ|iNV{-^!PcvrVt#e*l6cAwvjp3J%W=hfrs^zghf>O4Gs@gBb"
    "J;Lfi%wTU<0X=ML%%jQP|)^f%d<8Ci|b5!4I^KaF8_4u!?6H+Q@QNQ%wwpz{Y%1^!fa54F-V-6abgY745H?JH0&wn=$&-cay"
    "{hc~EIx)(PpU#o~9p_3$|Krj2w$^=9>}HQ1{Oj}K-r?udFVl{>_(fvboxhvg8S!7a>&FjUZcDrV^@jI8cRpVJk@IiA6&l~J-"
    "c||T{`D}UTc`W`XNS{|qk3aPPula_-1YqYbMe>H&2>7JO=YurgXq*C8O7&juBv3Ssg%j+1WikOd5>dGG~x&VL)OHljH40Z2z"
    "yAImdhGEhv;%SKccpwnKq>h=qVNdvJ?)Q)G@7*Wcyk!2j5;}9Qqa<EzdPZeGV!__ey#yC4_0s?nUeD6xCv)Gdnv^={aVO#>W"
    ";jG_Zs5^aoTbR(4UGKpZ@@i2<Ym^T{gBC0rXa-_gg~lyI7iVZ^k45D$o%OuEtga?xyS{mbqk4Y*5npf&3)HJ#l>ndLX2wCcT"
    "2YOk+-s`t*-0j!a5CVnjun`=6DEspI?&DQ9VNySPbfe0aW!0a(~rE2NRv)kLc)^B$&)4K_mO)STn_!CX1GlAcG8R3lyafK&e"
    "hD?Y}9U-QepxFffkf0GE9s!JifUcZT+lH;}0~C#m8Ia%{+t$b-hQ<hGh9rir(BT3id`~7W*T$4tKdBQc70^7+2z~_4@4m&P7"
    "fa;@(XhaIj!qzkjDU}9NzWIS0K$pRE$S@1T&^yNkp)A{^>JinK*GQ@C2OGxNTDX4ru&Q$hieYO!lD+oy`@Pa&o(0;qj_9<zg"
    "Q})o|_i)xQw@OVWWPo_HNRvko5?b*T|t+VG0zJnP+2yg=y7f@*?yH$--Bn9t3fWy~yEoGsKd*4Dj9>Wjz+9WWn)B7R%*VFw%"
    "d;{loQpLi@-Mop_$+I~Mm=@@-X}HQ4%vrq6vww3+1?be15-;$#w(ZGX_KUp3TT5X?_%r+XnE&N}Df^QznJh~|<eKV0ewnPE-"
    "D5=%x5G-HoMd_0nP)<H`J)1WMwZ;QA<DcZK>fiKd}Fb7iGDEIIE0cl^$6~u^Z<9W_tP$~|>26kb{z>WX%W#J$(Fvq%$J&$<d"
    "@Th39vk0~}g(BqgNY0vyc3JPK7Xz)|{nBfy&`;~7{R&JTrDT$WVzGKyDOdI@rOMuZftc9Ts|Sab0<IV(v#>whFBXV?fXzy22"
    "rrZehqzkM4Rg;tH1z`|5dRg1!ddr1NURITfQxc{DC*p)kC~AJNUj&SuL;d@$Y44fiU`x0gZP*r-6q&U6N@o0!ls_LE03w`LU"
    "uVEL8tYg`N5+-N7&((Zn;7R^7Jt=AO{SP<zX9~H3YyAhAuKe2?Po2czJ|QWUYWu%#rI4ZA(W%Y9W(S?lKE5kz-vjfGlW%h&n"
    "S+9<CFAy_uNe9JMP_+K`KU!1IaF?hzW3hiuCv+?o&sb~@1h(UjW$gn%$Him@{y2C!bW)%rlnY}R|dE<7@V;C7G$xrA+D-_V0"
    "^AA|?yl1yE~a6krWmgD-|lOjMTf*i`hbQ5f)XhNq#8u-fxk<5g~?qfSl-TeC6v6fSL3DZeNE^9IDt~^NSY_-ZJX{~eB#4vdk"
    "MBQ2`36s>jYQ+RxLi4JX5(KWauO!uF|EdM%i$BV;jVZfGk!?&qKpPXEqK%2qmf9Pfb<S?po+5d(yRe|H)}7i>n*;Sk>vg{j)"
    "X4YCD$RSu28W29LF|hG?@hgTp$*z+y%VidA2i#*_EcE(z+(9HoB-~_jRW;EWXKgII#+y#&S(%bxP0}-3>`wv>2e9pevnouiq"
    "iL8*CsN@r%<<sEV&!lB2lzIqM60V2uC&L35I#jvkZbpfX;&uDNM|NPzMq{6Ybx@m5QK%6AM`crC-PfXlVU_E+WfJViCzqSd0"
    "nP1eky~P(ca+zTn0hDhP=DavaEd$y;~g*89--umU!N5THOg>HJhOZbc$s#r3Hm-NqD(JJ|VO$c~jzMAn3SThL%3g4nmYvQ(5"
    "2(X~QM#PN#u8=T7W%|HqC<aFi)X&QIDI0d&qJxjkoJqN)O%Q|517E)n}%|O2RB)d2=0WXh4cUs0%2n4ylV^jT+go{7iaCa|n"
    "fDb9*J~Ap3y8|M+6=YQ4tNi+V)ADu@CA#F0Qab@0?yW0un>ZP%Ocv5+FHVuCcZGYATZ;5=f6(oz$|k1x4=|;B8Pj!4_BX*@`"
    "r!qe*4_b^wu!N^VnXzKrzAi^3u3Q_zfn&fo0v^RLcl_{QM!`08Q{gBuLaz9MU+E9^?<b^81gua{-~i+Q0QUy$w}uWbW|;9Nt"
    "zgKTnk{C=$>rIJM&PKLV}(biYm?S)kZ@S?;Z)zep^*9<6w#2Ms*=O2OI2kzO)T?oT=!T5V5dP0%`(pjackUwm=dGR)hg;K72"
    "+)QLqhJ&L|w=Ua-Loi39C7+E@S%8x@2ajWMhh@B5%=^*t=_utLMzq~IZ;Fqn4Wx<FPS3Igrh5N{3#VP}bnsk|@Cbby6wH0FU"
    "D@{0~DopM}50Y-XL#%Bk1s~;JdIuh}ZJF0ktdbKkp<Xh_XK^o=7+IS6r0CU;xoHcKvHYa~HlaU13hx@zn9Vi>lPCSB8##^)>"
    "tY)<f2y$pDXY-8$iCvh$9b}o<U|fz=`2nX(A^-P4{2%KwVq4VZq9&zA$MMpPS7prbr^SI>Dg+o*{ebS2%X+`B-0!Z^VNo_bL"
    "5}EhbZDz%LkQW8uyLi=r3E{_5Er8?)}Ti`8reki{Ncp%#LFX$0*IYUt>1<?H14Hr$fo`aMPG0#"
)
MAIN_SOURCE = zlib.decompress(base64.b85decode(MAIN_BLOB)).decode("utf-8")
MAIN_PATH = Path.cwd() / "main.py"
MAIN_PATH.write_text(MAIN_SOURCE, encoding="utf-8")
print("main.py bytes:", len(MAIN_SOURCE.encode("utf-8")))
print("main.py sha256:", hashlib.sha256(MAIN_SOURCE.encode("utf-8")).hexdigest())


In [ ]:
# Structural and configuration audit; no competition package is required.
import importlib.util
import sys

spec = importlib.util.spec_from_file_location("v46_submission", MAIN_PATH)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)

assert set(module._V44_ROUTES) == {
    "default", "yarn_first", "yarn_second", "bakery_capital", "yarn_third"
}
assert all(len(route) == 719 for route in module._V44_ROUTES.values())
cfg = module._V44_CONFIG
assert cfg.clone_phase_horizon == 6
assert cfg.clone_phase_future_window == 6
assert cfg.clone_phase_detection_start == 160
assert cfg.clone_phase_detection_stop == 260
assert "FERTILIZER" in cfg.clone_phase_items
assert "WHEAT" not in cfg.clone_phase_items
assert "CARROT" not in cfg.clone_phase_items
print("routes and V46 guards: OK")


In [ ]:
# Demonstrate the third-Yarn public supply gate with synthetic states.
def farm_with(cows, sheep):
    tiles = []
    for animal in ["COW"] * cows + ["SHEEP"] * sheep:
        tiles.append([{"animal": animal}])
    return {"tiles": tiles}

def choose(cows, sheep, prefix=("PIZZA_SHOP", "PET_CAFE")):
    module._V46_YARN_THIRD_LATCH[0] = False
    module._V46_YARN_THIRD_LAST_STEP[0] = -1
    obs = {
        "step": 216,
        "player": 0,
        "town": {"unlocked_shops": [prefix[0], prefix[1], "YARN_STORE"]},
        "farms": [{"tiles": []}, farm_with(cows, sheep)],
    }
    return module._v46_selected_route(obs, module._V44_CONFIG)

assert choose(8, 4) == "yarn_third"
assert choose(6, 4) == "default"
assert choose(8, 4, ("PET_CAFE", "BRUNCH_SPOT")) == "default"
print("third-Yarn gate: OK")


## 5. Complexity, scope, and limits

Online route selection scans a constant number of shops and public farm tiles; the market detector uses a fixed product set and a six-turn window. With board size fixed by the game,

$$
T_{\mathrm{turn}} = O(1),
\qquad
M_{\mathrm{turn}} = O(1),
$$

apart from the stored route table.

The replay-based counterfactual keeps recorded prices and exogenous events fixed, so it is diagnostic rather than a full causal simulator: changed sales can change later prices and opponent behavior. For that reason V46 chooses the smallest principled horizon, preserves all route actions, gates escalation on clean public evidence, and explicitly protects operational inventory.


In [ ]:
# Build an upload archive containing exactly the embedded main.py.
import gzip
import io
import tarfile

ARCHIVE_PATH = Path.cwd() / "submission.tar.gz"
payload = MAIN_SOURCE.encode("utf-8")
info = tarfile.TarInfo("main.py")
info.size = len(payload)
info.mode = 0o644
info.mtime = 0
with ARCHIVE_PATH.open("wb") as raw:
    with gzip.GzipFile(filename="", fileobj=raw, mode="wb", mtime=0) as gz:
        with tarfile.open(fileobj=gz, mode="w") as archive:
            archive.addfile(info, io.BytesIO(payload))
print("created:", ARCHIVE_PATH)


## Reproducibility

Decoded parent source SHA-256: `2fe711896465626350efafc87960a6d2f05b510ec7af62bc4679a80cdf3f9fc7`. The notebook embeds the complete improved source; the delivered archive is generated from the same bytes.
